# 1. Purpose and scope

This notebook supports **SCRUM-9: Define data-cleaning rules** by investigating data-quality issues identified in Notebook 04. The first issue is negative `unit_sales`.

Negative sales may represent returns, corrections, reversals, or anomalies. The goal is to understand their observed prevalence, magnitude, concentration, recurrence, and nearby sales context before selecting a cleaning rule.

This notebook section is analysis-only. It does not clean, replace, clip, drop, overwrite, or otherwise transform any source value, and it does not create a cleaned dataset or forecasting feature.

# 2. Imports and configuration

The merged Parquet is opened read-only. Row-group scans use selected columns so the complete 125.5-million-row dataset is never materialized in pandas.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the EDIP repository.")

repo_root = find_repo_root(Path.cwd())
MERGED_PATH = repo_root / "data" / "processed" / "favorita_merged" / "favorita_merged_base.parquet"

if not MERGED_PATH.is_file():
    raise FileNotFoundError(f"Merged Favorita Parquet not found: {MERGED_PATH}")

parquet_file = pq.ParquetFile(MERGED_PATH)
print({
    "input": MERGED_PATH.relative_to(repo_root).as_posix(),
    "file_size_bytes": MERGED_PATH.stat().st_size,
    "metadata_rows": parquet_file.metadata.num_rows,
    "metadata_columns": parquet_file.metadata.num_columns,
    "row_groups": parquet_file.metadata.num_row_groups,
})

{'input': 'data/processed/favorita_merged/favorita_merged_base.parquet', 'file_size_bytes': 723900039, 'metadata_rows': 125497040, 'metadata_columns': 21, 'row_groups': 502}


# 3. Extract negative unit_sales records

Each row group is read with only the investigation columns. Arrow filters `unit_sales < 0` before the small negative subset is converted to pandas. Store and item denominators are accumulated from the same bounded reads.

In [2]:
PREFERRED_NEGATIVE_COLUMNS = [
    "id",
    "date",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "onpromotion",
    "family",
    "class",
    "perishable",
    "transactions",
    "is_holiday",
]
available_columns = set(parquet_file.schema_arrow.names)
required_negative_columns = {"id", "date", "store_nbr", "item_nbr", "unit_sales", "family"}
missing_required = sorted(required_negative_columns - available_columns)
if missing_required:
    raise ValueError(f"Missing required investigation columns: {missing_required}")

negative_columns = [column for column in PREFERRED_NEGATIVE_COLUMNS if column in available_columns]
negative_tables = []
all_store_nbrs = set()
all_item_nbrs = set()

for row_group_index in range(parquet_file.metadata.num_row_groups):
    row_group = parquet_file.read_row_group(row_group_index, columns=negative_columns)
    all_store_nbrs.update(int(value) for value in pc.unique(row_group["store_nbr"]).to_pylist())
    all_item_nbrs.update(int(value) for value in pc.unique(row_group["item_nbr"]).to_pylist())
    negative_mask = pc.less(row_group["unit_sales"], pa.scalar(0.0))
    negative_rows = row_group.filter(negative_mask)
    if negative_rows.num_rows:
        negative_tables.append(negative_rows)

negative_table = pa.concat_tables(negative_tables) if negative_tables else pa.table({column: [] for column in negative_columns})
negative_df = negative_table.to_pandas().sort_values(["date", "store_nbr", "item_nbr", "id"]).reset_index(drop=True)
negative_df["date"] = pd.to_datetime(negative_df["date"])

total_merged_rows = parquet_file.metadata.num_rows
total_negative_rows = len(negative_df)
negative_percentage = 100.0 * total_negative_rows / total_merged_rows

negative_prevalence = pd.DataFrame([{
    "total_merged_row_count": total_merged_rows,
    "total_negative_row_count": total_negative_rows,
    "negative_row_percentage": negative_percentage,
    "observed_store_count": len(all_store_nbrs),
    "observed_item_count": len(all_item_nbrs),
}])
display(negative_prevalence)
display(negative_df.head(20))

,total_merged_row_count,total_negative_row_count,negative_row_percentage,observed_store_count,observed_item_count
0,125497040,7795,0.006211,54,4036


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,transactions,is_holiday
0,10655,2013-01-02,10,456875,-3.000,<NA>,CLEANING,3015,0,1293.0,False
1,46867,2013-01-03,5,559044,-1.000,<NA>,BREAD/BAKERY,2716,1,1740.0,False
2,50970,2013-01-03,9,365138,-3.000,<NA>,GROCERY I,1072,0,2396.0,False
3,71807,2013-01-03,41,812716,-19.000,<NA>,"LIQUOR,WINE,BEER",1318,0,809.0,False
4,71992,2013-01-03,41,1004551,-27.000,<NA>,"LIQUOR,WINE,BEER",1318,0,809.0,False
5,75255,2013-01-03,46,208530,-8.080,<NA>,FROZEN FOODS,2226,0,3438.0,False
6,91163,2013-01-04,9,457574,-8.516,<NA>,FROZEN FOODS,2226,0,1975.0,False
7,106658,2013-01-04,34,586824,-2.000,<NA>,CLEANING,3090,0,2369.0,False
8,111811,2013-01-04,41,956012,-1.000,<NA>,GROCERY I,1088,0,835.0,False
9,111813,2013-01-04,41,956014,-1.000,<NA>,GROCERY I,1088,0,835.0,False


# 4. Negative unit_sales distribution

Distribution statistics are calculated only from the extracted negative values. Magnitude buckets are non-overlapping and collectively exhaustive for values below zero.

In [3]:
negative_sales = negative_df["unit_sales"].astype("float64")
distribution_summary = pd.DataFrame([{
    "count": int(negative_sales.count()),
    "minimum": float(negative_sales.min()),
    "maximum_negative_value": float(negative_sales.max()),
    "mean": float(negative_sales.mean()),
    "median": float(negative_sales.median()),
    "standard_deviation": float(negative_sales.std()),
}])
display(distribution_summary)

percentile_levels = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
percentile_table = negative_sales.quantile(percentile_levels).rename("unit_sales").reset_index()
percentile_table["percentile"] = (100 * percentile_table["index"]).map(lambda value: f"{value:g}%")
percentile_table = percentile_table[["percentile", "unit_sales"]]
display(percentile_table)

negative_df["absolute_unit_sales_magnitude"] = negative_sales.abs()
bucket_specs = [
    (">= -1 and < 0", (negative_sales >= -1) & (negative_sales < 0)),
    (">= -5 and < -1", (negative_sales >= -5) & (negative_sales < -1)),
    (">= -10 and < -5", (negative_sales >= -10) & (negative_sales < -5)),
    (">= -50 and < -10", (negative_sales >= -50) & (negative_sales < -10)),
    (">= -100 and < -50", (negative_sales >= -100) & (negative_sales < -50)),
    (">= -500 and < -100", (negative_sales >= -500) & (negative_sales < -100)),
    (">= -1000 and < -500", (negative_sales >= -1000) & (negative_sales < -500)),
    ("< -1000", negative_sales < -1000),
]
magnitude_buckets = pd.DataFrame([
    {"magnitude_bucket": label, "negative_row_count": int(mask.sum())}
    for label, mask in bucket_specs
])
magnitude_buckets["percentage_of_negative_rows"] = 100.0 * magnitude_buckets["negative_row_count"] / total_negative_rows
if int(magnitude_buckets["negative_row_count"].sum()) != total_negative_rows:
    raise ValueError("Magnitude buckets do not cover every negative row exactly once.")
display(magnitude_buckets)

,count,minimum,maximum_negative_value,mean,median,standard_deviation
0,7795,-15372.0,-0.002,-18.419606,-1.0,242.254294


,percentile,unit_sales
0,1%,-210.36000
1,5%,-35.00000
2,10%,-14.30600
3,25%,-4.00000
4,50%,-1.00000
5,75%,-1.00000
6,90%,-1.00000
7,95%,-1.00000
8,99%,-0.26928


,magnitude_bucket,negative_row_count,percentage_of_negative_rows
0,>= -1 and < 0,4082,52.366902
1,>= -5 and < -1,2141,27.466325
2,>= -10 and < -5,569,7.299551
3,>= -50 and < -10,720,9.236690
4,>= -100 and < -50,129,1.654907
5,>= -500 and < -100,118,1.513791
6,>= -1000 and < -500,16,0.205260
7,< -1000,20,0.256575


# 5. Most extreme negative records

The 30 lowest `unit_sales` values are shown with available product, promotion, transaction, and holiday context. This is descriptive evidence, not an anomaly label.

In [4]:
extreme_columns = [
    column for column in [
        "date", "store_nbr", "item_nbr", "family", "unit_sales",
        "onpromotion", "transactions", "is_holiday",
    ]
    if column in negative_df.columns
]
most_extreme_negative_records = negative_df.sort_values(
    ["unit_sales", "date", "store_nbr", "item_nbr", "id"]
).head(30)
display(most_extreme_negative_records[extreme_columns])

,date,store_nbr,item_nbr,family,unit_sales,onpromotion,transactions,is_holiday
2910,2015-06-22,18,1166474,BEVERAGES,-15372.000,False,1353.0,False
6805,2017-04-06,32,1158720,GROCERY I,-10002.000,False,755.0,False
4787,2016-05-14,53,119026,CLEANING,-4673.000,False,1967.0,True
4788,2016-05-14,53,323921,PERSONAL CARE,-3606.000,False,1967.0,True
4794,2016-05-14,53,1229028,PERSONAL CARE,-3600.000,False,1967.0,True
6074,2016-12-20,38,2010755,FROZEN FOODS,-3451.363,False,1777.0,False
4622,2016-04-24,46,1463852,BEVERAGES,-2487.000,False,4476.0,True
3037,2015-07-20,49,1082042,GROCERY I,-2400.000,False,2600.0,False
4805,2016-05-16,29,812769,"LIQUOR,WINE,BEER",-2400.000,False,1069.0,True
6810,2017-04-07,7,1430040,BEVERAGES,-1943.000,False,2113.0,False


# 6. Negative records by store

In [5]:
negative_by_store = negative_df.groupby("store_nbr", as_index=False).agg(
    negative_record_count=("unit_sales", "size"),
    total_negative_units=("unit_sales", "sum"),
    mean_negative_unit_sales=("unit_sales", "mean"),
    minimum_unit_sales=("unit_sales", "min"),
).sort_values(["negative_record_count", "minimum_unit_sales", "store_nbr"], ascending=[False, True, True])

affected_store_count = int(negative_by_store["store_nbr"].nunique())
affected_store_percentage = 100.0 * affected_store_count / len(all_store_nbrs)
display(pd.DataFrame([{
    "stores_with_negative_rows": affected_store_count,
    "observed_training_stores": len(all_store_nbrs),
    "percentage_of_stores_affected": affected_store_percentage,
}]))
display(negative_by_store.head(20))

,stores_with_negative_rows,observed_training_stores,percentage_of_stores_affected
0,54,54,100.0


,store_nbr,negative_record_count,total_negative_units,mean_negative_unit_sales,minimum_unit_sales
17,18,483,-18463.3690,-38.226437,-15372.000
43,44,414,-4335.6480,-10.472580,-1806.000
48,49,362,-5116.0760,-14.132807,-2400.000
6,7,320,-3864.3010,-12.075941,-1943.000
2,3,319,-3632.5605,-11.387337,-448.000
50,51,290,-1077.5070,-3.715541,-84.000
7,8,272,-5337.0540,-19.621522,-992.000
0,1,267,-1587.1150,-5.944251,-276.000
1,2,264,-12936.6780,-49.002568,-1797.000
3,4,248,-1355.8860,-5.467282,-134.000


# 7. Negative records by item

In [6]:
negative_by_item = negative_df.groupby("item_nbr", as_index=False).agg(
    family=("family", "first"),
    negative_record_count=("unit_sales", "size"),
    total_negative_units=("unit_sales", "sum"),
    mean_negative_unit_sales=("unit_sales", "mean"),
    minimum_unit_sales=("unit_sales", "min"),
).sort_values(["negative_record_count", "minimum_unit_sales", "item_nbr"], ascending=[False, True, True])

affected_item_count = int(negative_by_item["item_nbr"].nunique())
affected_item_percentage = 100.0 * affected_item_count / len(all_item_nbrs)
display(pd.DataFrame([{
    "items_with_negative_rows": affected_item_count,
    "observed_training_items": len(all_item_nbrs),
    "percentage_of_items_affected": affected_item_percentage,
}]))
display(negative_by_item.head(30))

,items_with_negative_rows,observed_training_items,percentage_of_items_affected
0,2568,4036,63.627354


,item_nbr,family,negative_record_count,total_negative_units,mean_negative_unit_sales,minimum_unit_sales
93,172995,AUTOMOTIVE,69,-170.0000,-2.463768,-6.000
2427,2006144,HOME AND KITCHEN II,66,-70.0000,-1.060606,-2.000
1392,1229440,PERSONAL CARE,64,-104.0000,-1.625000,-25.000
1710,1410850,HOME AND KITCHEN II,58,-60.0000,-1.034483,-2.000
1957,1464240,BEVERAGES,41,-247.0000,-6.024390,-24.000
161,250782,HOME APPLIANCES,38,-42.0000,-1.105263,-5.000
1002,957096,DAIRY,37,-116.0000,-3.135135,-40.000
1123,1049595,AUTOMOTIVE,35,-97.0000,-2.771429,-6.000
1767,1456952,HOME CARE,28,-33.0000,-1.178571,-2.000
1267,1137146,GROCERY I,26,-63.0000,-2.423077,-17.000


# 8. Negative records over time

In [7]:
negative_df["year"] = negative_df["date"].dt.year
negative_df["year_month"] = negative_df["date"].dt.to_period("M").astype(str)

negative_by_year = negative_df.groupby("year", as_index=False).agg(
    negative_record_count=("unit_sales", "size")
)
negative_by_year["percentage_of_negative_rows"] = 100.0 * negative_by_year["negative_record_count"] / total_negative_rows

negative_by_year_month = negative_df.groupby("year_month", as_index=False).agg(
    negative_record_count=("unit_sales", "size")
)
negative_by_year_month["percentage_of_negative_rows"] = 100.0 * negative_by_year_month["negative_record_count"] / total_negative_rows

negative_by_date = negative_df.groupby("date", as_index=False).agg(
    negative_record_count=("unit_sales", "size"),
    total_negative_units=("unit_sales", "sum"),
).sort_values(["negative_record_count", "total_negative_units", "date"], ascending=[False, True, True])

display(negative_by_year)
display(negative_by_year_month)
display(negative_by_date.head(20))

,year,negative_record_count,percentage_of_negative_rows
0,2013,947,12.148813
1,2014,1350,17.318794
2,2015,1619,20.769724
3,2016,2246,28.813342
4,2017,1633,20.949326


,year_month,negative_record_count,percentage_of_negative_rows
0,2013-01,80,1.026299
1,2013-02,76,0.974984
2,2013-03,69,0.885183
3,2013-04,51,0.654266
4,2013-05,88,1.128929
5,2013-06,91,1.167415
6,2013-07,59,0.756895
7,2013-08,67,0.859525
8,2013-09,58,0.744067
9,2013-10,80,1.026299


,date,negative_record_count,total_negative_units
1368,2017-01-12,29,-71.000
637,2014-12-23,24,-964.000
1452,2017-04-07,22,-2364.000
1313,2016-11-16,21,-194.861
1518,2017-06-12,21,-62.000
1547,2017-07-11,20,-90.000
1424,2017-03-09,20,-31.000
1514,2017-06-08,20,-28.000
1195,2016-07-21,19,-263.440
893,2015-09-16,19,-111.000


# 9. Repeated store-item behaviour

In [8]:
negative_by_store_item = negative_df.groupby(["store_nbr", "item_nbr"], as_index=False).agg(
    family=("family", "first"),
    negative_record_count=("unit_sales", "size"),
    minimum_negative_value=("unit_sales", "min"),
    total_negative_unit_sales=("unit_sales", "sum"),
).sort_values(
    ["negative_record_count", "minimum_negative_value", "store_nbr", "item_nbr"],
    ascending=[False, True, True, True],
)

pair_count_distribution = negative_by_store_item["negative_record_count"].value_counts().sort_index().rename_axis(
    "negative_records_per_store_item_pair"
).reset_index(name="store_item_pair_count")
pair_count_distribution["percentage_of_affected_pairs"] = (
    100.0 * pair_count_distribution["store_item_pair_count"] / len(negative_by_store_item)
)

repeated_pairs = negative_by_store_item[negative_by_store_item["negative_record_count"] > 1]
negative_rows_in_repeated_pairs = int(repeated_pairs["negative_record_count"].sum())
display(pd.DataFrame([{
    "unique_store_item_pairs_with_negative_rows": len(negative_by_store_item),
    "pairs_with_repeated_negative_rows": len(repeated_pairs),
    "negative_rows_from_repeated_pairs": negative_rows_in_repeated_pairs,
    "percentage_of_negative_rows_from_repeated_pairs": 100.0 * negative_rows_in_repeated_pairs / total_negative_rows,
}]))
display(pair_count_distribution)
display(negative_by_store_item.head(30))

,unique_store_item_pairs_with_negative_rows,pairs_with_repeated_negative_rows,negative_rows_from_repeated_pairs,percentage_of_negative_rows_from_repeated_pairs
0,6841,703,1657,21.257216


,negative_records_per_store_item_pair,store_item_pair_count,percentage_of_affected_pairs
0,1,6138,89.723725
1,2,545,7.966672
2,3,109,1.593334
3,4,28,0.409297
4,5,13,0.190031
5,6,3,0.043853
6,7,1,0.014618
7,9,3,0.043853
8,11,1,0.014618


,store_nbr,item_nbr,family,negative_record_count,minimum_negative_value,total_negative_unit_sales
5333,45,172995,AUTOMOTIVE,11,-4.000,-33.000
2577,18,1047395,CLEANING,9,-9.000,-28.000
4987,44,172995,AUTOMOTIVE,9,-5.000,-24.000
2403,18,208498,CLEANING,9,-3.000,-11.000
6400,50,1410850,HOME AND KITCHEN II,7,-1.000,-7.000
2404,18,209085,CLEANING,6,-3.000,-10.000
4757,40,2006144,HOME AND KITCHEN II,6,-2.000,-7.000
2316,16,2006144,HOME AND KITCHEN II,6,-1.000,-6.000
668,3,1464008,BEVERAGES,5,-12.000,-33.000
5057,44,759657,LINGERIE,5,-8.000,-16.000


# 10. Before-and-after context for negative records

The deterministic review set contains the ten most extreme negative observations plus up to ten non-duplicate observations drawn from the most frequently recurring negative store-item pairs. A second bounded scan reads only the selected pair keys, date, `unit_sales`, `id`, and `onpromotion`. Missing store-item-date rows remain absent and are never interpreted as zero sales.

In [9]:
extreme_cases = negative_df.sort_values(
    ["unit_sales", "date", "store_nbr", "item_nbr", "id"]
).head(10).copy()
extreme_cases["selection_reason"] = "10 most extreme negative records"

frequent_pair_candidates = negative_by_store_item.head(30)[
    ["store_nbr", "item_nbr", "negative_record_count"]
]
recurring_candidates = negative_df.merge(
    frequent_pair_candidates,
    on=["store_nbr", "item_nbr"],
    how="inner",
    validate="many_to_one",
).sort_values(
    ["negative_record_count", "unit_sales", "date", "store_nbr", "item_nbr", "id"],
    ascending=[False, True, True, True, True, True],
).groupby(["store_nbr", "item_nbr"], as_index=False).head(1)
recurring_cases = recurring_candidates[~recurring_candidates["id"].isin(extreme_cases["id"])].head(10).copy()
recurring_cases["selection_reason"] = "frequently recurring negative store-item case"

selected_cases = pd.concat([extreme_cases, recurring_cases], ignore_index=True).drop_duplicates("id").reset_index(drop=True)
selected_cases["case_id"] = np.arange(1, len(selected_cases) + 1)
selected_cases["window_start"] = selected_cases["date"] - pd.Timedelta(days=7)
selected_cases["window_end"] = selected_cases["date"] + pd.Timedelta(days=7)
display(selected_cases[[
    "case_id", "selection_reason", "id", "date", "store_nbr", "item_nbr", "family", "unit_sales",
    "window_start", "window_end",
]])

CONTEXT_COLUMNS = [
    column for column in ["id", "date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"]
    if column in available_columns
]
PAIR_KEY_MULTIPLIER = 10_000_000
selected_pair_codes = np.array(
    sorted({int(row.store_nbr) * PAIR_KEY_MULTIPLIER + int(row.item_nbr) for row in selected_cases.itertuples()}),
    dtype="int64",
)
overall_window_start = selected_cases["window_start"].min()
overall_window_end = selected_cases["window_end"].max()
context_parts = []

for row_group_index in range(parquet_file.metadata.num_row_groups):
    context_frame = parquet_file.read_row_group(row_group_index, columns=CONTEXT_COLUMNS).to_pandas()
    date_mask = context_frame["date"].between(overall_window_start, overall_window_end)
    pair_codes = (
        context_frame["store_nbr"].astype("int64") * PAIR_KEY_MULTIPLIER
        + context_frame["item_nbr"].astype("int64")
    )
    candidate_rows = context_frame.loc[date_mask & np.isin(pair_codes, selected_pair_codes)]
    if candidate_rows.empty:
        continue
    for case in selected_cases.itertuples(index=False):
        matched = candidate_rows[
            candidate_rows["store_nbr"].eq(case.store_nbr)
            & candidate_rows["item_nbr"].eq(case.item_nbr)
            & candidate_rows["date"].between(case.window_start, case.window_end)
        ].copy()
        if matched.empty:
            continue
        matched["case_id"] = case.case_id
        matched["negative_observation_date"] = case.date
        matched["relative_calendar_day"] = (matched["date"] - case.date).dt.days
        context_parts.append(matched)

context_observations = pd.concat(context_parts, ignore_index=True).sort_values(
    ["case_id", "date", "id"]
)
context_case_summary = context_observations.groupby("case_id", as_index=False).agg(
    observed_context_rows=("unit_sales", "size"),
    observed_date_count=("date", "nunique"),
    nearby_positive_rows=("unit_sales", lambda values: int((values > 0).sum())),
    nearby_negative_rows=("unit_sales", lambda values: int((values < 0).sum())),
    nearby_zero_rows=("unit_sales", lambda values: int((values == 0).sum())),
    minimum_context_unit_sales=("unit_sales", "min"),
    maximum_context_unit_sales=("unit_sales", "max"),
).merge(
    selected_cases[["case_id", "selection_reason", "date", "store_nbr", "item_nbr", "unit_sales"]].rename(
        columns={"date": "selected_negative_date", "unit_sales": "selected_negative_unit_sales"}
    ),
    on="case_id",
    how="left",
    validate="one_to_one",
)
display(context_case_summary)

context_display_columns = [
    column for column in [
        "date", "relative_calendar_day", "store_nbr", "item_nbr", "unit_sales", "onpromotion"
    ]
    if column in context_observations.columns
]
for case in selected_cases.itertuples(index=False):
    display(Markdown(
        f"### Case {case.case_id}: store `{case.store_nbr}`, item `{case.item_nbr}`, "
        f"negative date `{case.date.date().isoformat()}`, selected value `{case.unit_sales:g}`"
    ))
    display(context_observations.loc[context_observations["case_id"].eq(case.case_id), context_display_columns])

,case_id,selection_reason,id,date,store_nbr,item_nbr,family,unit_sales,window_start,window_end
0,1,10 most extreme negative records,49592112,2015-06-22,18,1166474,BEVERAGES,-15372.000,2015-06-15,2015-06-29
1,2,10 most extreme negative records,111606073,2017-04-06,32,1158720,GROCERY I,-10002.000,2017-03-30,2017-04-13
2,3,10 most extreme negative records,79204331,2016-05-14,53,119026,CLEANING,-4673.000,2016-05-07,2016-05-21
3,4,10 most extreme negative records,79204535,2016-05-14,53,323921,PERSONAL CARE,-3606.000,2016-05-07,2016-05-21
4,5,10 most extreme negative records,79205524,2016-05-14,53,1229028,PERSONAL CARE,-3600.000,2016-05-07,2016-05-21
5,6,10 most extreme negative records,100571255,2016-12-20,38,2010755,FROZEN FOODS,-3451.363,2016-12-13,2016-12-27
6,7,10 most extreme negative records,77277007,2016-04-24,46,1463852,BEVERAGES,-2487.000,2016-04-17,2016-05-01
7,8,10 most extreme negative records,51895953,2015-07-20,49,1082042,GROCERY I,-2400.000,2015-07-13,2015-07-27
8,9,10 most extreme negative records,79361005,2016-05-16,29,812769,"LIQUOR,WINE,BEER",-2400.000,2016-05-09,2016-05-23
9,10,10 most extreme negative records,111664986,2017-04-07,7,1430040,BEVERAGES,-1943.000,2017-03-31,2017-04-14


,case_id,observed_context_rows,observed_date_count,nearby_positive_rows,nearby_negative_rows,nearby_zero_rows,minimum_context_unit_sales,maximum_context_unit_sales,selection_reason,selected_negative_date,store_nbr,item_nbr,selected_negative_unit_sales
0,1,14,14,13,1,0,-15372.000,9025.00,10 most extreme negative records,2015-06-22,18,1166474,-15372.000
1,2,7,7,6,1,0,-10002.000,122.00,10 most extreme negative records,2017-04-06,32,1158720,-10002.000
2,3,15,15,14,1,0,-4673.000,4720.00,10 most extreme negative records,2016-05-14,53,119026,-4673.000
3,4,15,15,14,1,0,-3606.000,3628.00,10 most extreme negative records,2016-05-14,53,323921,-3606.000
4,5,15,15,14,1,0,-3600.000,3620.00,10 most extreme negative records,2016-05-14,53,1229028,-3600.000
5,6,12,12,11,1,0,-3451.363,3316.26,10 most extreme negative records,2016-12-20,38,2010755,-3451.363
6,7,14,14,13,1,0,-2487.000,2527.00,10 most extreme negative records,2016-04-24,46,1463852,-2487.000
7,8,12,12,11,1,0,-2400.000,2402.00,10 most extreme negative records,2015-07-20,49,1082042,-2400.000
8,9,5,5,4,1,0,-2400.000,2401.00,10 most extreme negative records,2016-05-16,29,812769,-2400.000
9,10,15,15,14,1,0,-1943.000,2071.00,10 most extreme negative records,2017-04-07,7,1430040,-1943.000


### Case 1: store `18`, item `1166474`, negative date `2015-06-22`, selected value `-15372`

,date,relative_calendar_day,store_nbr,item_nbr,unit_sales,onpromotion
29,2015-06-15,-7,18,1166474,4.0,False
30,2015-06-16,-6,18,1166474,7.0,False
31,2015-06-17,-5,18,1166474,9.0,False
32,2015-06-18,-4,18,1166474,5.0,False
33,2015-06-19,-3,18,1166474,9025.0,False
34,2015-06-20,-2,18,1166474,18.0,False
35,2015-06-21,-1,18,1166474,22.0,False
36,2015-06-22,0,18,1166474,-15372.0,False
37,2015-06-23,1,18,1166474,19.0,False
38,2015-06-24,2,18,1166474,11.0,False


### Case 2: store `32`, item `1158720`, negative date `2017-04-06`, selected value `-10002`

...

,date,relative_calendar_day,store_nbr,item_nbr,unit_sales,onpromotion
155,2016-11-09,-5,16,2006144,6.0,True
156,2016-11-10,-4,16,2006144,4.0,True
160,2016-11-11,-3,16,2006144,1.0,True
163,2016-11-13,-1,16,2006144,3.0,True
164,2016-11-14,0,16,2006144,-1.0,False
170,2016-11-19,5,16,2006144,6.0,True
171,2016-11-20,6,16,2006144,4.0,True
172,2016-11-21,7,16,2006144,1.0,True


### Case 19: store `3`, item `1464008`, negative date `2017-01-10`, selected value `-12`

,date,relative_calendar_day,store_nbr,item_nbr,unit_sales,onpromotion
185,2017-01-03,-7,3,1464008,25.0,False
186,2017-01-04,-6,3,1464008,9.0,False
187,2017-01-05,-5,3,1464008,2.0,False
188,2017-01-06,-4,3,1464008,30.0,False
189,2017-01-07,-3,3,1464008,15.0,False
190,2017-01-08,-2,3,1464008,1.0,False
191,2017-01-09,-1,3,1464008,6.0,False
192,2017-01-10,0,3,1464008,-12.0,False
193,2017-01-11,1,3,1464008,10.0,False
194,2017-01-12,2,3,1464008,1.0,False


### Case 20: store `44`, item `759657`, negative date `2013-12-19`, selected value `-8`

,date,relative_calendar_day,store_nbr,item_nbr,unit_sales,onpromotion
16,2013-12-14,-5,44,759657,1.0,<NA>
17,2013-12-15,-4,44,759657,1.0,<NA>
18,2013-12-16,-3,44,759657,1.0,<NA>
19,2013-12-17,-2,44,759657,52.0,<NA>
20,2013-12-18,-1,44,759657,2.0,<NA>
21,2013-12-19,0,44,759657,-8.0,<NA>
22,2013-12-22,3,44,759657,3.0,<NA>
23,2013-12-26,7,44,759657,2.0,<NA>


# 11. Evidence summary for negative unit_sales

The table separates observed evidence from interpretation and cleaning implications. Nearby behavior refers only to the deterministic context sample and is not generalized to every negative row.

In [10]:
median_negative = float(negative_sales.median())
q25_negative = float(negative_sales.quantile(0.25))
q75_negative = float(negative_sales.quantile(0.75))
extreme_below_1000_count = int((negative_sales < -1000).sum())
top_store = negative_by_store.iloc[0]
top_item = negative_by_item.iloc[0]
largest_year = negative_by_year.sort_values("negative_record_count", ascending=False).iloc[0]
cases_with_nearby_positive = int((context_case_summary["nearby_positive_rows"] > 0).sum())
cases_with_repeated_nearby_negative = int((context_case_summary["nearby_negative_rows"] > 1).sum())

evidence_summary = pd.DataFrame([
    {
        "Check": "Prevalence",
        "Evidence": f"{total_negative_rows:,} of {total_merged_rows:,} rows ({negative_percentage:.6f}%).",
        "Interpretation": "Negative observations are rare in the merged training population.",
        "Cleaning implication": "Rarity alone does not establish that the rows are invalid.",
    },
    {
        "Check": "Typical negative magnitude",
        "Evidence": f"Median {median_negative:g}; middle 50% from {q25_negative:g} to {q75_negative:g}.",
        "Interpretation": "The typical negative value is materially smaller than the most extreme cases.",
        "Cleaning implication": "A single rule should not conflate routine small negatives with extreme magnitudes.",
    },
    {
        "Check": "Extreme negative magnitude",
        "Evidence": f"Minimum {negative_sales.min():g}; {extreme_below_1000_count:,} rows are below -1,000.",
        "Interpretation": "A tail of unusually large negative magnitudes exists.",
        "Cleaning implication": "Extreme cases warrant explicit review or flagging before target derivation.",
    },
    {
        "Check": "Store concentration",
        "Evidence": f"{affected_store_count}/{len(all_store_nbrs)} stores affected; top store {int(top_store.store_nbr)} has {int(top_store.negative_record_count):,} rows.",
        "Interpretation": "Store coverage and top-store concentration are directly observable, but operational cause is unknown.",
        "Cleaning implication": "Do not assume one store-specific data fault without source-system evidence.",
    },
    {
        "Check": "Item concentration",
        "Evidence": f"{affected_item_count}/{len(all_item_nbrs)} items affected; top item {int(top_item.item_nbr)} has {int(top_item.negative_record_count):,} rows.",
        "Interpretation": "Negative records affect a subset of products with uneven recurrence.",
        "Cleaning implication": "Retain item identity when assessing later target treatment.",
    },
    {
        "Check": "Temporal concentration",
        "Evidence": f"All observed years: {negative_by_year['year'].min()}-{negative_by_year['year'].max()}; highest year {int(largest_year.year)} has {int(largest_year.negative_record_count):,} rows.",
        "Interpretation": "Year and date tables show whether the pattern is persistent or period-specific.",
        "Cleaning implication": "Avoid a time-specific rule unless the observed concentration and source context support it.",
    },
    {
        "Check": "Repeated store-item behaviour",
        "Evidence": f"{len(repeated_pairs):,} pairs repeat; they account for {negative_rows_in_repeated_pairs:,} rows ({100.0 * negative_rows_in_repeated_pairs / total_negative_rows:.2f}%).",
        "Interpretation": "Repeated negatives are not exclusively isolated one-off records.",
        "Cleaning implication": "Dropping rows could erase recurring return/correction signals.",
    },
    {
        "Check": "Nearby positive-sales behaviour",
        "Evidence": f"{cases_with_nearby_positive}/{len(context_case_summary)} sampled cases have observed positive rows within ±7 days; {cases_with_repeated_nearby_negative} have additional nearby negatives.",
        "Interpretation": "The deterministic sample contains mixed local behavior; missing dates were not treated as zero.",
        "Cleaning implication": "Preserve raw values and carry explicit context into later target-design review.",
    },
])
display(evidence_summary)

,Check,Evidence,Interpretation,Cleaning implication
0,Prevalence,"7,795 of 125,497,040 rows (0.006211%).",Negative observations are rare in the merged t...,Rarity alone does not establish that the rows ...
1,Typical negative magnitude,Median -1; middle 50% from -4 to -1.,The typical negative value is materially small...,A single rule should not conflate routine smal...
2,Extreme negative magnitude,"Minimum -15372; 20 rows are below -1,000.",A tail of unusually large negative magnitudes ...,Extreme cases warrant explicit review or flagg...
3,Store concentration,54/54 stores affected; top store 18 has 483 rows.,Store coverage and top-store concentration are...,Do not assume one store-specific data fault wi...
4,Item concentration,2568/4036 items affected; top item 172995 has ...,Negative records affect a subset of products w...,Retain item identity when assessing later targ...
5,Temporal concentration,All observed years: 2013-2017; highest year 20...,Year and date tables show whether the pattern ...,Avoid a time-specific rule unless the observed...
6,Repeated store-item behaviour,"703 pairs repeat; they account for 1,657 rows ...",Repeated negatives are not exclusively isolate...,Dropping rows could erase recurring return/cor...
7,Nearby positive-sales behaviour,20/20 sampled cases have observed positive row...,The deterministic sample contains mixed local ...,Preserve raw values and carry explicit context...


# 12. Provisional cleaning-rule decision

## Provisional cleaning rule — negative unit_sales

### 1. What the data shows

- Exactly **7,795 of 125,497,040 rows (0.006211%)** have negative `unit_sales`.
- The distribution is highly skewed: the median is **-1**, the middle 50% is **-4 to -1**, and **79.83%** of negatives fall between -5 and 0. However, **20 records are below -1,000**, with a minimum of **-15,372**.
- Negative rows occur in **54 of 54 stores**, **2,568 of 4,036 items (63.63%)**, and every observed year from 2013 through 2017.
- **703 store-item pairs** have repeated negatives; their **1,657 rows represent 21.26%** of all negative records.
- In the deterministic ±7-day sample, **20 of 20 cases** have observed positive sales nearby. None has an additional negative row inside its selected 15-day window. Missing dates were not interpreted as zero.

### 2. What is inferred

The combination of small recurring negatives, broad store/item/time coverage, and nearby positive sales is consistent with legitimate return, correction, or reversal behavior. Some very large negative values also have large nearby positive observations, which is consistent with reversal-like activity. These interpretations are plausible but are not proven by the available columns.

### 3. What remains uncertain

- The source does not explicitly label returns, corrections, reversals, or data-entry errors.
- The ±7-day context review is a deterministic 20-case sample, not a causal or exhaustive classification of all 7,795 rows.
- The correct forecasting treatment depends on whether the future target represents gross demand, net sales, or another business-defined quantity.
- The extreme tail may mix valid bulk reversals with genuine anomalies; magnitude alone cannot distinguish them.

### 4. Recommended provisional rule

**Option D is best supported: preserve raw negative values unchanged, but derive a separate non-negative forecasting target later after the target semantics are approved.**

Use **Option E as a secondary review control**: clearly extreme negative magnitudes should be flagged for separate investigation, without overwriting or deleting the raw values. Converting every negative value to zero or removing every negative row is not supported by the current evidence because it would erase potentially meaningful return/correction behavior.

No rule is applied in this notebook.

### 5. Carry into the final SCRUM-9 cleaning-rules table

- Preserve the source `unit_sales` column unchanged for provenance and auditability.
- Record a candidate later-stage derived non-negative forecasting target as a separate field, pending an explicit gross-demand versus net-sales decision.
- Record a separate candidate exception-review rule for extreme negatives; define its threshold only after business/source validation.
- Document that negative rows must not be silently dropped, clipped, or converted to zero.
- Carry forward the unresolved requirement to confirm negative-value semantics with authoritative dataset or business documentation.


# 13. `onpromotion` missing-value investigation

`onpromotion` indicates whether a store-item-date observation was recorded as being under promotion. Its observed states are `True`, `False`, and missing. The merged-data review found approximately 21.66 million missing values (about 17.26%), but this investigation calculates exact evidence rather than hardcoding it.

Missing does **not** automatically mean `False`. The purpose is to determine whether missingness is structural, historical, store/item specific, or broadly random before defining a cleaning rule. No cleaning, replacement, or imputation is applied.

# 14. Overall `onpromotion` status counts

One memory-safe row-group pass reads only `id`, `date`, `store_nbr`, `item_nbr`, `family`, `onpromotion`, and `unit_sales`. It accumulates exact status counts and bounded group summaries used by Sections 14–21.

In [11]:
ONPROMOTION_SCAN_COLUMNS = [
    "id", "date", "store_nbr", "item_nbr", "family", "onpromotion", "unit_sales"
]
missing_scan_columns = [column for column in ONPROMOTION_SCAN_COLUMNS if column not in parquet_file.schema_arrow.names]
if missing_scan_columns:
    raise ValueError(f"Missing required onpromotion investigation columns: {missing_scan_columns}")

STATUS_ORDER = ["True", "False", "missing"]
SALES_QUANTILE_SAMPLE_MODULUS = 100
ITEM_MIN_OBSERVATIONS_FOR_RATE_RANKING = 10_000

status_parts = []
year_parts = []
month_parts = []
date_parts = []
store_parts = []
item_parts = []
family_parts = []
store_date_parts = []
sales_parts = []
quantile_samples = {status: [] for status in STATUS_ORDER}
status_minimum_dates = {}

for row_group_index in range(parquet_file.metadata.num_row_groups):
    scan_frame = parquet_file.read_row_group(
        row_group_index,
        columns=ONPROMOTION_SCAN_COLUMNS,
    ).to_pandas()
    scan_frame["promotion_status"] = scan_frame["onpromotion"].map(
        {True: "True", False: "False"}
    ).fillna("missing")
    scan_frame["year"] = scan_frame["date"].dt.year
    scan_frame["year_month"] = scan_frame["date"].dt.to_period("M").astype(str)

    def count_part(keys):
        return scan_frame.groupby(keys, observed=True).size().rename("row_count").reset_index()

    status_parts.append(count_part(["promotion_status"]))
    year_parts.append(count_part(["year", "promotion_status"]))
    month_parts.append(count_part(["year_month", "promotion_status"]))
    date_parts.append(count_part(["date", "promotion_status"]))
    store_parts.append(count_part(["store_nbr", "promotion_status"]))
    item_parts.append(count_part(["item_nbr", "promotion_status"]))
    family_parts.append(count_part(["family", "promotion_status"]))
    store_date_parts.append(count_part(["store_nbr", "date", "promotion_status"]))

    for status, minimum_date in scan_frame.groupby("promotion_status", observed=True)["date"].min().items():
        status_minimum_dates[status] = (
            minimum_date if status not in status_minimum_dates else min(status_minimum_dates[status], minimum_date)
        )

    scan_frame["_unit_sales_squared"] = np.square(scan_frame["unit_sales"].astype("float64"))
    scan_frame["_negative_unit_sales"] = scan_frame["unit_sales"].lt(0).astype("int64")
    sales_part = scan_frame.groupby("promotion_status", observed=True).agg(
        row_count=("unit_sales", "size"),
        unit_sales_sum=("unit_sales", "sum"),
        unit_sales_sum_squares=("_unit_sales_squared", "sum"),
        minimum_unit_sales=("unit_sales", "min"),
        maximum_unit_sales=("unit_sales", "max"),
        negative_unit_sales_count=("_negative_unit_sales", "sum"),
    ).reset_index()
    sales_parts.append(sales_part)

    sampled = scan_frame.loc[scan_frame["id"].mod(SALES_QUANTILE_SAMPLE_MODULUS).eq(0)]
    for status, values in sampled.groupby("promotion_status", observed=True)["unit_sales"]:
        quantile_samples[status].append(values.to_numpy(dtype="float64", copy=True))

def combine_count_parts(parts, keys):
    return pd.concat(parts, ignore_index=True).groupby(keys, as_index=False, observed=True)["row_count"].sum()

def status_pivot(parts, keys):
    combined = combine_count_parts(parts, [*keys, "promotion_status"])
    pivoted = combined.pivot(index=keys, columns="promotion_status", values="row_count").fillna(0).astype("int64")
    for status in STATUS_ORDER:
        if status not in pivoted.columns:
            pivoted[status] = 0
    pivoted = pivoted[STATUS_ORDER].reset_index().rename(columns={
        "True": "true_count",
        "False": "false_count",
        "missing": "missing_count",
    })
    pivoted["total_row_count"] = pivoted[["true_count", "false_count", "missing_count"]].sum(axis=1)
    pivoted["missing_percentage"] = 100.0 * pivoted["missing_count"] / pivoted["total_row_count"]
    return pivoted

overall_status_counts = combine_count_parts(status_parts, ["promotion_status"]).set_index("promotion_status")["row_count"]
overall_status_counts = overall_status_counts.reindex(STATUS_ORDER, fill_value=0).astype("int64")
overall_status_table = overall_status_counts.rename("row_count").reset_index()
overall_status_table["percentage_of_total"] = 100.0 * overall_status_table["row_count"] / total_merged_rows

if int(overall_status_counts.sum()) != total_merged_rows:
    raise ValueError("True + False + missing does not equal the merged Parquet row count.")

yearly_onpromotion = status_pivot(year_parts, ["year"]).sort_values("year").reset_index(drop=True)
monthly_onpromotion = status_pivot(month_parts, ["year_month"]).sort_values("year_month").reset_index(drop=True)
daily_onpromotion = status_pivot(date_parts, ["date"]).sort_values("date").reset_index(drop=True)
store_onpromotion = status_pivot(store_parts, ["store_nbr"]).sort_values("store_nbr").reset_index(drop=True)
item_onpromotion = status_pivot(item_parts, ["item_nbr"]).sort_values("item_nbr").reset_index(drop=True)
family_onpromotion = status_pivot(family_parts, ["family"]).sort_values("family").reset_index(drop=True)
store_daily_onpromotion = status_pivot(store_date_parts, ["store_nbr", "date"]).sort_values(
    ["store_nbr", "date"]
).reset_index(drop=True)

missing_dates = daily_onpromotion.loc[daily_onpromotion["missing_count"] > 0, "date"]
status_date_summary = pd.DataFrame([
    {"promotion_status": status, "minimum_date": status_minimum_dates[status].date().isoformat()}
    for status in STATUS_ORDER
])
status_date_summary.loc[status_date_summary["promotion_status"].eq("missing"), "maximum_date"] = (
    missing_dates.max().date().isoformat()
)

display(overall_status_table)
display(status_date_summary)
print(f"Count reconciliation: {int(overall_status_counts.sum()):,} = {total_merged_rows:,} metadata rows")

,promotion_status,row_count,percentage_of_total
0,True,7810622,6.223750
1,False,96028767,76.518751
2,missing,21657651,17.257499


,promotion_status,minimum_date,maximum_date
0,True,2014-04-01,NaN
1,False,2014-04-01,NaN
2,missing,2013-01-01,2014-03-31


Count reconciliation: 125,497,040 = 125,497,040 metadata rows


# 15. Missing `onpromotion` over time

In [12]:
highest_missing_months = monthly_onpromotion.sort_values(
    ["missing_percentage", "missing_count", "year_month"], ascending=[False, False, True]
).head(20)
zero_missing_months = monthly_onpromotion.loc[monthly_onpromotion["missing_count"].eq(0)]

display(yearly_onpromotion)
display(monthly_onpromotion)
display(Markdown("### First 24 observed months"))
display(monthly_onpromotion.head(24))
display(Markdown("### Last 24 observed months"))
display(monthly_onpromotion.tail(24))
display(Markdown("### Months with the highest missing percentage"))
display(highest_missing_months)
display(Markdown("### Months with zero missing values"))
display(zero_missing_months)

promotion_status,year,true_count,false_count,missing_count,total_row_count,missing_percentage
0,2013,0,0,16322662,16322662,100.000000
1,2014,459114,16477499,5334989,22271602,23.954222
2,2015,1087275,26777369,0,27864644,0.000000
3,2016,3514584,31715287,0,35229871,0.000000
4,2017,2749649,21058612,0,23808261,0.000000


promotion_status,year_month,true_count,false_count,missing_count,total_row_count,missing_percentage
0,2013-01,0,0,1211964,1211964,100.0
1,2013-02,0,0,1160386,1160386,100.0
2,2013-03,0,0,1321927,1321927,100.0
3,2013-04,0,0,1284375,1284375,100.0
4,2013-05,0,0,1360917,1360917,100.0
5,2013-06,0,0,1353864,1353864,100.0
6,2013-07,0,0,1385579,1385579,100.0
7,2013-08,0,0,1421744,1421744,100.0
8,2013-09,0,0,1378438,1378438,100.0
9,2013-10,0,0,1428676,1428676,100.0


### First 24 observed months

promotion_status,year_month,true_count,false_count,missing_count,total_row_count,missing_percentage
0,2013-01,0,0,1211964,1211964,100.0
1,2013-02,0,0,1160386,1160386,100.0
2,2013-03,0,0,1321927,1321927,100.0
3,2013-04,0,0,1284375,1284375,100.0
4,2013-05,0,0,1360917,1360917,100.0
5,2013-06,0,0,1353864,1353864,100.0
6,2013-07,0,0,1385579,1385579,100.0
7,2013-08,0,0,1421744,1421744,100.0
8,2013-09,0,0,1378438,1378438,100.0
9,2013-10,0,0,1428676,1428676,100.0


### Last 24 observed months

promotion_status,year_month,true_count,false_count,missing_count,total_row_count,missing_percentage
32,2015-09,140873,2446532,0,2587405,0.0
33,2015-10,115220,2728484,0,2843704,0.0
34,2015-11,128497,2641593,0,2770090,0.0
35,2015-12,132993,2734933,0,2867926,0.0
36,2016-01,119578,2704107,0,2823685,0.0
37,2016-02,183541,2549613,0,2733154,0.0
38,2016-03,181790,2765061,0,2946851,0.0
39,2016-04,215312,2674093,0,2889405,0.0
40,2016-05,370019,2613445,0,2983464,0.0
41,2016-06,285416,2536448,0,2821864,0.0


### Months with the highest missing percentage

promotion_status,year_month,true_count,false_count,missing_count,total_row_count,missing_percentage
14,2014-03,0,0,2001725,2001725,100.0
12,2014-01,0,0,1933844,1933844,100.0
11,2013-12,0,0,1548395,1548395,100.0
10,2013-11,0,0,1466397,1466397,100.0
9,2013-10,0,0,1428676,1428676,100.0
7,2013-08,0,0,1421744,1421744,100.0
13,2014-02,0,0,1399420,1399420,100.0
6,2013-07,0,0,1385579,1385579,100.0
8,2013-09,0,0,1378438,1378438,100.0
4,2013-05,0,0,1360917,1360917,100.0


### Months with zero missing values

promotion_status,year_month,true_count,false_count,missing_count,total_row_count,missing_percentage
15,2014-04,1939,1482883,0,1484822,0.0
16,2014-05,11575,1526670,0,1538245,0.0
17,2014-06,21496,1524975,0,1546471,0.0
18,2014-07,69665,2023955,0,2093620,0.0
19,2014-08,42940,1623737,0,1666677,0.0
20,2014-09,71659,2014266,0,2085925,0.0
21,2014-10,76899,2091704,0,2168603,0.0
22,2014-11,71034,2055088,0,2126122,0.0
23,2014-12,91907,2134221,0,2226128,0.0
24,2015-01,53148,1704330,0,1757478,0.0


# 16. Daily transition analysis

For descriptive transition evidence, **substantial missingness** is defined as at least **10%** missing on a date, and **near-complete recording** as at most **1%** missing. The material-drop date is not assumed: it is the date with the largest observed day-over-day decline in missing percentage.

In [13]:
SUBSTANTIAL_MISSING_PERCENTAGE = 10.0
NEAR_COMPLETE_MISSING_PERCENTAGE = 1.0

daily_onpromotion["non_missing_count"] = daily_onpromotion["true_count"] + daily_onpromotion["false_count"]
daily_onpromotion["missing_percentage_change"] = daily_onpromotion["missing_percentage"].diff()
first_non_missing_date = daily_onpromotion.loc[daily_onpromotion["non_missing_count"] > 0, "date"].min()
largest_drop_index = daily_onpromotion["missing_percentage_change"].idxmin()
material_drop_date = daily_onpromotion.loc[largest_drop_index, "date"]
material_drop_points = float(-daily_onpromotion.loc[largest_drop_index, "missing_percentage_change"])
last_substantial_missing_date = daily_onpromotion.loc[
    daily_onpromotion["missing_percentage"] >= SUBSTANTIAL_MISSING_PERCENTAGE, "date"
].max()

zero_missing = daily_onpromotion["missing_count"].eq(0)
zero_missing_suffix = zero_missing.iloc[::-1].cummin().iloc[::-1]
permanent_zero_dates = daily_onpromotion.loc[zero_missing_suffix, "date"]
permanent_zero_missing_start = permanent_zero_dates.min() if not permanent_zero_dates.empty else pd.NaT

daily_transition_summary = pd.DataFrame([{
    "first_date_with_any_non_missing": first_non_missing_date.date().isoformat(),
    "largest_daily_missing_drop_date": material_drop_date.date().isoformat(),
    "largest_daily_missing_drop_percentage_points": material_drop_points,
    "first_date_at_or_below_1pct_missing": daily_onpromotion.loc[
        daily_onpromotion["missing_percentage"] <= NEAR_COMPLETE_MISSING_PERCENTAGE, "date"
    ].min().date().isoformat(),
    "last_date_at_or_above_10pct_missing": last_substantial_missing_date.date().isoformat(),
    "permanent_zero_missing_start": (
        permanent_zero_missing_start.date().isoformat() if pd.notna(permanent_zero_missing_start) else None
    ),
}])
display(daily_transition_summary)

transition_window = daily_onpromotion.loc[
    daily_onpromotion["date"].between(material_drop_date - pd.Timedelta(days=7), material_drop_date + pd.Timedelta(days=7)),
    ["date", "total_row_count", "missing_count", "missing_percentage", "true_count", "false_count", "missing_percentage_change"],
]
display(transition_window)

,first_date_with_any_non_missing,largest_daily_missing_drop_date,largest_daily_missing_drop_percentage_points,first_date_at_or_below_1pct_missing,last_date_at_or_above_10pct_missing,permanent_zero_missing_start
0,2014-04-01,2014-04-01,100.0,2014-04-01,2014-03-31,2014-04-01


promotion_status,date,total_row_count,missing_count,missing_percentage,true_count,false_count,missing_percentage_change
447,2014-03-25,62650,62650,100.0,0,0,0.0
448,2014-03-26,62774,62774,100.0,0,0,0.0
449,2014-03-27,61580,61580,100.0,0,0,0.0
450,2014-03-28,63426,63426,100.0,0,0,0.0
451,2014-03-29,68879,68879,100.0,0,0,0.0
452,2014-03-30,67532,67532,100.0,0,0,0.0
453,2014-03-31,65381,65381,100.0,0,0,0.0
454,2014-04-01,51537,0,0.0,8,51529,-100.0
455,2014-04-02,51067,0,0.0,4,51063,0.0
456,2014-04-03,49703,0,0.0,120,49583,0.0


# 17. Missingness by store

In [14]:
stores_affected_by_missingness = int((store_onpromotion["missing_count"] > 0).sum())
store_missing_range = float(store_onpromotion["missing_percentage"].max() - store_onpromotion["missing_percentage"].min())
store_missing_std = float(store_onpromotion["missing_percentage"].std())

display(pd.DataFrame([{
    "stores_affected": stores_affected_by_missingness,
    "observed_stores": len(store_onpromotion),
    "minimum_store_missing_percentage": store_onpromotion["missing_percentage"].min(),
    "maximum_store_missing_percentage": store_onpromotion["missing_percentage"].max(),
    "store_missing_percentage_range": store_missing_range,
    "store_missing_percentage_std": store_missing_std,
}]))
display(Markdown("### Top 20 stores by missing percentage"))
display(store_onpromotion.sort_values(["missing_percentage", "missing_count"], ascending=False).head(20))
display(Markdown("### Bottom 20 stores by missing percentage"))
display(store_onpromotion.sort_values(["missing_percentage", "missing_count"], ascending=True).head(20))

,stores_affected,observed_stores,minimum_store_missing_percentage,maximum_store_missing_percentage,store_missing_percentage_range,store_missing_percentage_std
0,47,54,0.0,20.989503,20.989503,6.393352


### Top 20 stores by missing percentage

promotion_status,store_nbr,true_count,false_count,missing_count,total_row_count,missing_percentage
25,26,111498,1305225,376359,1793082,20.989503
17,18,134087,1617965,444630,2196682,20.240982
24,25,113542,1531019,412459,2057020,20.051288
15,16,114197,1342194,363644,1820035,19.980055
31,32,89932,1001039,271644,1362615,19.935492
4,5,149370,1993704,523617,2666691,19.635458
22,23,128989,1773125,463583,2365697,19.596043
23,24,156253,2076128,537508,2769889,19.405399
9,10,103223,1300141,337118,1740482,19.369232
33,34,150651,1706411,442360,2299422,19.237878


### Bottom 20 stores by missing percentage

promotion_status,store_nbr,true_count,false_count,missing_count,total_row_count,missing_percentage
19,20,157358,1508627,0,1665985,0.000000
20,21,144062,1229357,0,1373419,0.000000
21,22,99543,823940,0,923483,0.000000
28,29,153632,1355327,0,1508959,0.000000
41,42,124670,1337013,0,1461683,0.000000
51,52,30814,259767,0,290581,0.000000
52,53,204016,1734239,0,1938255,0.000000
35,36,158527,1738683,313952,2211162,14.198507
42,43,124340,1480874,334480,1939694,17.243957
39,40,140054,1595845,366900,2102799,17.448173


# 18. Missingness by item

All affected-item counts use the complete item population. Percentage ranking is restricted to items with at least **10,000 observed rows**, a transparent threshold chosen to reduce unstable rates from sparse item histories.

In [15]:
items_affected_by_missingness = int((item_onpromotion["missing_count"] > 0).sum())
eligible_item_rates = item_onpromotion.loc[
    item_onpromotion["total_row_count"] >= ITEM_MIN_OBSERVATIONS_FOR_RATE_RANKING
]

display(pd.DataFrame([{
    "items_affected": items_affected_by_missingness,
    "observed_items": len(item_onpromotion),
    "percentage_of_items_affected": 100.0 * items_affected_by_missingness / len(item_onpromotion),
    "minimum_observations_for_rate_ranking": ITEM_MIN_OBSERVATIONS_FOR_RATE_RANKING,
    "items_eligible_for_rate_ranking": len(eligible_item_rates),
}]))
display(Markdown("### Top 30 items by missing row count"))
display(item_onpromotion.sort_values(["missing_count", "missing_percentage"], ascending=False).head(30))
display(Markdown("### Top 30 sufficiently observed items by missing percentage"))
display(eligible_item_rates.sort_values(["missing_percentage", "missing_count"], ascending=False).head(30))

,items_affected,observed_items,percentage_of_items_affected,minimum_observations_for_rate_ranking,items_eligible_for_rate_ranking
0,2669,4036,66.129832,10000,3236


### Top 30 items by missing row count

promotion_status,item_nbr,true_count,false_count,missing_count,total_row_count,missing_percentage
574,502331,6105,56305,21065,83475,25.235100
302,314384,2914,59484,21052,83450,25.227082
393,364606,1179,61113,21016,83308,25.226869
658,559870,7401,54118,20994,82513,25.443263
240,265559,2684,59389,20974,83047,25.255578
669,564533,444,60692,20950,82086,25.522013
1004,807493,626,58404,20855,79885,26.106278
178,219150,609,44765,20851,66225,31.485089
1497,1057033,2867,49735,20824,73426,28.360526
1554,1084881,914,59655,20755,81324,25.521371


### Top 30 sufficiently observed items by missing percentage

promotion_status,item_nbr,true_count,false_count,missing_count,total_row_count,missing_percentage
194,227728,773,3661,9299,13733,67.712809
1034,819933,283,8796,16133,25212,63.989370
567,476707,743,3589,7374,11706,62.993337
1164,871118,1251,8765,10653,20669,51.540955
1576,1090401,1180,9756,9248,20184,45.818470
1425,1037654,311,11313,8947,20571,43.493267
239,265382,33,8268,5716,14017,40.779054
1593,1094235,880,11564,8117,20561,39.477652
608,514446,33,10592,6834,17459,39.143135
1571,1089820,129,10885,6997,18011,38.848481


# 19. Missingness by product family

In [16]:
family_missing_range = float(family_onpromotion["missing_percentage"].max() - family_onpromotion["missing_percentage"].min())
family_missing_std = float(family_onpromotion["missing_percentage"].std())
display(pd.DataFrame([{
    "family_count": len(family_onpromotion),
    "minimum_family_missing_percentage": family_onpromotion["missing_percentage"].min(),
    "maximum_family_missing_percentage": family_onpromotion["missing_percentage"].max(),
    "family_missing_percentage_range": family_missing_range,
    "family_missing_percentage_std": family_missing_std,
}]))
display(family_onpromotion.sort_values(["missing_percentage", "missing_count"], ascending=False))

,family_count,minimum_family_missing_percentage,maximum_family_missing_percentage,family_missing_percentage_range,family_missing_percentage_std
0,33,0.0,29.62272,29.62272,9.115994


promotion_status,family,true_count,false_count,missing_count,total_row_count,missing_percentage
17,HOME APPLIANCES,58,16917,7145,24120,29.622720
21,LINGERIE,2815,208782,80185,291782,27.481133
24,MEATS,304028,1541919,586101,2432048,24.099072
13,GROCERY II,7347,211166,67145,285658,23.505381
10,EGGS,194922,1015655,370256,1580833,23.421576
14,HARDWARE,163,53664,15708,69535,22.590063
32,SEAFOOD,39052,160826,58017,257895,22.496365
9,DELI,583316,2618499,914581,4116396,22.218003
29,PREPARED FOODS,37524,558767,170297,766588,22.214932
7,CLEANING,661157,12687271,3667172,17015600,21.551823


# 20. Sales behaviour by promotion status

Counts, means, sample standard deviations, minima, maxima, and negative-row percentages are exact streaming aggregates. To preserve memory safety, the median and quartiles are deterministic approximations from rows where `id % 100 == 0` (about a 1% sample); sample sizes are displayed. This comparison is descriptive and does not imply that promotion causes sales differences.

In [17]:
sales_aggregate = pd.concat(sales_parts, ignore_index=True).groupby("promotion_status", as_index=False).agg(
    row_count=("row_count", "sum"),
    unit_sales_sum=("unit_sales_sum", "sum"),
    unit_sales_sum_squares=("unit_sales_sum_squares", "sum"),
    minimum_unit_sales=("minimum_unit_sales", "min"),
    maximum_unit_sales=("maximum_unit_sales", "max"),
    negative_unit_sales_count=("negative_unit_sales_count", "sum"),
)
sales_aggregate["mean_unit_sales"] = sales_aggregate["unit_sales_sum"] / sales_aggregate["row_count"]
sales_aggregate["unit_sales_variance"] = (
    sales_aggregate["unit_sales_sum_squares"]
    - np.square(sales_aggregate["unit_sales_sum"]) / sales_aggregate["row_count"]
) / (sales_aggregate["row_count"] - 1)
sales_aggregate["standard_deviation"] = np.sqrt(sales_aggregate["unit_sales_variance"].clip(lower=0))
sales_aggregate["negative_unit_sales_percentage"] = (
    100.0 * sales_aggregate["negative_unit_sales_count"] / sales_aggregate["row_count"]
)

quantile_rows = []
for status in STATUS_ORDER:
    values = np.concatenate(quantile_samples[status]) if quantile_samples[status] else np.array([], dtype="float64")
    quantile_rows.append({
        "promotion_status": status,
        "quantile_sample_row_count": len(values),
        "p25_unit_sales_approx": float(np.quantile(values, 0.25)),
        "median_unit_sales_approx": float(np.quantile(values, 0.50)),
        "p75_unit_sales_approx": float(np.quantile(values, 0.75)),
    })
quantile_summary = pd.DataFrame(quantile_rows)

sales_behaviour_by_status = sales_aggregate.merge(
    quantile_summary, on="promotion_status", how="left", validate="one_to_one"
)
sales_behaviour_by_status["status_order"] = sales_behaviour_by_status["promotion_status"].map(
    {status: index for index, status in enumerate(STATUS_ORDER)}
)
sales_behaviour_by_status = sales_behaviour_by_status.sort_values("status_order").drop(
    columns=["status_order", "unit_sales_sum", "unit_sales_sum_squares", "unit_sales_variance"]
)
display(sales_behaviour_by_status[[
    "promotion_status", "row_count", "mean_unit_sales", "median_unit_sales_approx",
    "standard_deviation", "p25_unit_sales_approx", "p75_unit_sales_approx",
    "minimum_unit_sales", "maximum_unit_sales", "negative_unit_sales_count",
    "negative_unit_sales_percentage", "quantile_sample_row_count",
]])

,promotion_status,row_count,mean_unit_sales,median_unit_sales_approx,standard_deviation,p25_unit_sales_approx,p75_unit_sales_approx,minimum_unit_sales,maximum_unit_sales,negative_unit_sales_count,negative_unit_sales_percentage,quantile_sample_row_count
1,True,7810622,13.466666,6.0,39.704772,2.99025,13.0,-4.0,17146.0,8,0.000102,78430
0,False,96028767,8.088135,4.0,22.549599,2.00000,8.0,-15372.0,89440.0,6480,0.006748,959964
2,missing,21657651,8.852925,4.0,19.896626,2.00000,9.0,-1344.0,12021.0,1307,0.006035,216577


# 21. Missingness continuity / block structure

Global and representative-store runs use observed calendar dates only. A run breaks when consecutive observed dates are more than one calendar day apart. Missing store-item-date rows are not treated as zero sales or as non-promoted observations.

In [18]:
def longest_calendar_run(summary, condition, date_column="date"):
    dates = summary.loc[condition, date_column].sort_values().drop_duplicates().reset_index(drop=True)
    if dates.empty:
        return {"start_date": None, "end_date": None, "calendar_days": 0, "observed_dates": 0}
    run_id = dates.diff().ne(pd.Timedelta(days=1)).cumsum()
    runs = dates.groupby(run_id).agg(["min", "max", "size"])
    runs["calendar_days"] = (runs["max"] - runs["min"]).dt.days + 1
    longest = runs.sort_values(["calendar_days", "size"], ascending=False).iloc[0]
    return {
        "start_date": longest["min"].date().isoformat(),
        "end_date": longest["max"].date().isoformat(),
        "calendar_days": int(longest["calendar_days"]),
        "observed_dates": int(longest["size"]),
    }

global_substantial_run = longest_calendar_run(
    daily_onpromotion,
    daily_onpromotion["missing_percentage"] >= SUBSTANTIAL_MISSING_PERCENTAGE,
)
global_zero_run = longest_calendar_run(daily_onpromotion, daily_onpromotion["missing_count"].eq(0))
global_continuity = pd.DataFrame([{
    "dates_with_any_missing": int((daily_onpromotion["missing_count"] > 0).sum()),
    "dates_with_100pct_missing": int(daily_onpromotion["missing_percentage"].eq(100).sum()),
    "dates_with_partial_missing": int(daily_onpromotion["missing_percentage"].between(0, 100, inclusive="neither").sum()),
    "longest_substantial_missing_start": global_substantial_run["start_date"],
    "longest_substantial_missing_end": global_substantial_run["end_date"],
    "longest_substantial_missing_calendar_days": global_substantial_run["calendar_days"],
    "longest_zero_missing_start": global_zero_run["start_date"],
    "longest_zero_missing_end": global_zero_run["end_date"],
    "longest_zero_missing_calendar_days": global_zero_run["calendar_days"],
}])
display(global_continuity)

sorted_store_rates = store_onpromotion.sort_values("missing_percentage").reset_index(drop=True)
representative_store_ids = list(dict.fromkeys([
    int(sorted_store_rates.iloc[-1]["store_nbr"]),
    int(sorted_store_rates.iloc[len(sorted_store_rates) // 2]["store_nbr"]),
    int(sorted_store_rates.iloc[0]["store_nbr"]),
]))
representative_store_rows = []
for store_nbr in representative_store_ids:
    store_days = store_daily_onpromotion.loc[store_daily_onpromotion["store_nbr"].eq(store_nbr)].copy()
    substantial_run = longest_calendar_run(
        store_days, store_days["missing_percentage"] >= SUBSTANTIAL_MISSING_PERCENTAGE
    )
    zero_run = longest_calendar_run(store_days, store_days["missing_count"].eq(0))
    representative_store_rows.append({
        "store_nbr": store_nbr,
        "overall_missing_percentage": float(store_onpromotion.loc[
            store_onpromotion["store_nbr"].eq(store_nbr), "missing_percentage"
        ].iloc[0]),
        "dates_with_any_missing": int((store_days["missing_count"] > 0).sum()),
        "dates_with_100pct_missing": int(store_days["missing_percentage"].eq(100).sum()),
        "dates_with_partial_missing": int(store_days["missing_percentage"].between(0, 100, inclusive="neither").sum()),
        "longest_substantial_missing_start": substantial_run["start_date"],
        "longest_substantial_missing_end": substantial_run["end_date"],
        "longest_substantial_missing_calendar_days": substantial_run["calendar_days"],
        "longest_zero_missing_start": zero_run["start_date"],
        "longest_zero_missing_end": zero_run["end_date"],
        "longest_zero_missing_calendar_days": zero_run["calendar_days"],
    })
display(pd.DataFrame(representative_store_rows))

,dates_with_any_missing,dates_with_100pct_missing,dates_with_partial_missing,longest_substantial_missing_start,longest_substantial_missing_end,longest_substantial_missing_calendar_days,longest_zero_missing_start,longest_zero_missing_end,longest_zero_missing_calendar_days
0,454,454,0,2013-01-01,2013-12-24,358,2015-12-26,2016-12-24,365


,store_nbr,overall_missing_percentage,dates_with_any_missing,dates_with_100pct_missing,dates_with_partial_missing,longest_substantial_missing_start,longest_substantial_missing_end,longest_substantial_missing_calendar_days,longest_zero_missing_start,longest_zero_missing_end,longest_zero_missing_calendar_days
0,26,20.989503,452,452,0,2013-01-02,2013-12-24,357,2016-01-02,2016-12-24,358
1,8,18.521003,452,452,0,2013-01-02,2013-12-24,357,2016-01-02,2016-12-24,358
2,29,0.000000,0,0,0,NaN,NaN,0,2016-01-02,2016-12-24,358


# 22. Relationship with data era / history

This evidence addresses whether missing `onpromotion` values are mainly associated with older records. It does not claim that dataset documentation confirms a historical tracking limitation; no such direct source evidence is present in this notebook.

In [19]:
earliest_year = yearly_onpromotion.iloc[0]
latest_year = yearly_onpromotion.iloc[-1]
max_missing_month = monthly_onpromotion.sort_values("missing_percentage", ascending=False).iloc[0]
min_missing_month = monthly_onpromotion.sort_values("missing_percentage", ascending=True).iloc[0]

history_evidence = pd.DataFrame([
    {
        "Evidence area": "Yearly pattern",
        "Observation": f"{int(earliest_year.year)} missing {earliest_year.missing_percentage:.2f}%; {int(latest_year.year)} missing {latest_year.missing_percentage:.2f}% (latest year is partial).",
        "Interpretation": "A large early-versus-late difference would support an era-linked pattern.",
        "Uncertainty": "Calendar year also differs in product/store coverage and the final year is incomplete.",
    },
    {
        "Evidence area": "Monthly pattern",
        "Observation": f"Highest month {max_missing_month.year_month}: {max_missing_month.missing_percentage:.2f}%; lowest {min_missing_month.year_month}: {min_missing_month.missing_percentage:.2f}%.",
        "Interpretation": "Sustained early-high and later-low months support a historical transition more than random missingness.",
        "Uncertainty": "The notebook observes the pattern but does not identify the source-system cause.",
    },
    {
        "Evidence area": "Daily transition",
        "Observation": f"Largest daily decline occurs {material_drop_date.date().isoformat()} ({material_drop_points:.2f} percentage points); last >=10% date is {last_substantial_missing_date.date().isoformat()}.",
        "Interpretation": "An abrupt and persistent decline is consistent with a recording-era change.",
        "Uncertainty": "Thresholds are descriptive and do not prove a system implementation date.",
    },
    {
        "Evidence area": "Store distribution",
        "Observation": f"{stores_affected_by_missingness} of {len(store_onpromotion)} stores are affected; store-rate range {store_missing_range:.2f} points, standard deviation {store_missing_std:.2f}.",
        "Interpretation": "Broad store coverage argues against a single-store failure.",
        "Uncertainty": "Different store operating periods can change lifetime missing percentages.",
    },
    {
        "Evidence area": "Item and family distribution",
        "Observation": f"{items_affected_by_missingness}/{len(item_onpromotion)} items affected; family-rate range {family_missing_range:.2f} points.",
        "Interpretation": "Broad entity coverage can coexist with history-driven missingness because items enter at different dates.",
        "Uncertainty": "Item/family rates are confounded by their observed history lengths.",
    },
])
display(history_evidence)

,Evidence area,Observation,Interpretation,Uncertainty
0,Yearly pattern,2013 missing 100.00%; 2017 missing 0.00% (late...,A large early-versus-late difference would sup...,Calendar year also differs in product/store co...
1,Monthly pattern,Highest month 2013-01: 100.00%; lowest 2014-04...,Sustained early-high and later-low months supp...,The notebook observes the pattern but does not...
2,Daily transition,Largest daily decline occurs 2014-04-01 (100.0...,An abrupt and persistent decline is consistent...,Thresholds are descriptive and do not prove a ...
3,Store distribution,47 of 54 stores are affected; store-rate range...,Broad store coverage argues against a single-s...,Different store operating periods can change l...
4,Item and family distribution,2669/4036 items affected; family-rate range 29...,Broad entity coverage can coexist with history...,Item/family rates are confounded by their obse...


# 23. Evidence summary for `onpromotion`

In [20]:
true_count = int(overall_status_counts["True"])
false_count = int(overall_status_counts["False"])
missing_onpromotion_count = int(overall_status_counts["missing"])
missing_onpromotion_percentage = 100.0 * missing_onpromotion_count / total_merged_rows
dates_with_any_missing = int((daily_onpromotion["missing_count"] > 0).sum())
dates_with_full_missing = int(daily_onpromotion["missing_percentage"].eq(100).sum())
dates_with_partial_missing = int(daily_onpromotion["missing_percentage"].between(0, 100, inclusive="neither").sum())
top_family = family_onpromotion.sort_values("missing_percentage", ascending=False).iloc[0]
sales_missing = sales_behaviour_by_status.loc[
    sales_behaviour_by_status["promotion_status"].eq("missing")
].iloc[0]

onpromotion_evidence_summary = pd.DataFrame([
    {
        "Check": "Overall missingness",
        "Evidence": f"{missing_onpromotion_count:,}/{total_merged_rows:,} rows ({missing_onpromotion_percentage:.6f}%) are missing.",
        "Interpretation": "Missingness is material and cannot be ignored.",
        "Cleaning implication": "Retain unknown status until its semantics are justified.",
    },
    {
        "Check": "Temporal concentration",
        "Evidence": f"Largest daily drop {material_drop_date.date().isoformat()}; last >=10% missing date {last_substantial_missing_date.date().isoformat()}.",
        "Interpretation": "A concentrated transition is more consistent with history/era than broadly random row loss.",
        "Cleaning implication": "Do not infer False from missing; preserve the era signal for later modelling design.",
    },
    {
        "Check": "Store concentration",
        "Evidence": f"{stores_affected_by_missingness}/{len(store_onpromotion)} stores affected; rate range {store_missing_range:.2f} points.",
        "Interpretation": "Missingness is not isolated to one store.",
        "Cleaning implication": "A store-specific fill rule is not supported.",
    },
    {
        "Check": "Item concentration",
        "Evidence": f"{items_affected_by_missingness}/{len(item_onpromotion)} items affected; rate rankings use >= {ITEM_MIN_OBSERVATIONS_FOR_RATE_RANKING:,} rows.",
        "Interpretation": "Item rates reflect both missingness and differing history lengths.",
        "Cleaning implication": "Do not impute from item identity alone.",
    },
    {
        "Check": "Family concentration",
        "Evidence": f"Top family {top_family.family} has {top_family.missing_percentage:.2f}% missing; family range {family_missing_range:.2f} points.",
        "Interpretation": "Family differences require historical-coverage context.",
        "Cleaning implication": "A family-specific fill rule is not yet supported.",
    },
    {
        "Check": "Sales behaviour",
        "Evidence": f"Missing-status exact mean {sales_missing.mean_unit_sales:.3f}; approximate median {sales_missing.median_unit_sales_approx:.3f}; negative share {sales_missing.negative_unit_sales_percentage:.6f}%.",
        "Interpretation": "Sales distributions differ descriptively, but promotion causality is not identified.",
        "Cleaning implication": "Do not use sales values to backfill promotion status in this cleaning stage.",
    },
    {
        "Check": "Continuity/block structure",
        "Evidence": f"{dates_with_any_missing} dates have missing values: {dates_with_full_missing} fully missing and {dates_with_partial_missing} partially missing; longest >=10% run {global_substantial_run['calendar_days']} days.",
        "Interpretation": "Long blocks support structural/historical missingness rather than independent random omissions.",
        "Cleaning implication": "Preserve missing as a distinct unknown state.",
    },
    {
        "Check": "Historical-transition evidence",
        "Evidence": f"Earliest year {earliest_year.missing_percentage:.2f}% missing versus latest partial year {latest_year.missing_percentage:.2f}%; permanent zero start {permanent_zero_missing_start.date().isoformat() if pd.notna(permanent_zero_missing_start) else 'not observed'}.",
        "Interpretation": "The observed time pattern can support an era-linked inference, not a documented source-system fact.",
        "Cleaning implication": "Carry uncertainty explicitly into the final SCRUM-9 rule table.",
    },
])
display(onpromotion_evidence_summary)

,Check,Evidence,Interpretation,Cleaning implication
0,Overall missingness,"21,657,651/125,497,040 rows (17.257499%) are m...",Missingness is material and cannot be ignored.,Retain unknown status until its semantics are ...
1,Temporal concentration,Largest daily drop 2014-04-01; last >=10% miss...,A concentrated transition is more consistent w...,Do not infer False from missing; preserve the ...
2,Store concentration,47/54 stores affected; rate range 20.99 points.,Missingness is not isolated to one store.,A store-specific fill rule is not supported.
3,Item concentration,2669/4036 items affected; rate rankings use >=...,Item rates reflect both missingness and differ...,Do not impute from item identity alone.
4,Family concentration,Top family HOME APPLIANCES has 29.62% missing;...,Family differences require historical-coverage...,A family-specific fill rule is not yet supported.
5,Sales behaviour,Missing-status exact mean 8.853; approximate m...,"Sales distributions differ descriptively, but ...",Do not use sales values to backfill promotion ...
6,Continuity/block structure,454 dates have missing values: 454 fully missi...,Long blocks support structural/historical miss...,Preserve missing as a distinct unknown state.
7,Historical-transition evidence,Earliest year 100.00% missing versus latest pa...,The observed time pattern can support an era-l...,Carry uncertainty explicitly into the final SC...


# 24. Provisional cleaning rule — `onpromotion`

### 1. What the data shows

- Exact status counts are **7,810,622 `True` (6.223750%)**, **96,028,767 `False` (76.518751%)**, and **21,657,651 missing (17.257499%)**; they reconcile to all **125,497,040** merged rows.
- Missingness is a complete historical block: missing values run from **2013-01-01 through 2014-03-31**. All **454 dates** with missing values are **100% missing**, with **zero partially missing dates**.
- `True` and `False` first appear on **2014-04-01**. Missingness drops by exactly **100 percentage points** that day and remains zero afterward.
- Yearly missingness is **100% in 2013**, **23.95% in 2014** because the first quarter is missing, and **0% from 2015 onward**.
- **47 of 54 stores** and **2,669 of 4,036 items** have missing rows. Store, item, and family lifetime percentages differ because entities have different observed history lengths relative to the cutoff.
- Sales differ descriptively by status: exact mean `unit_sales` is 13.467 for `True`, 8.088 for `False`, and 8.853 for missing. The displayed medians/quartiles are deterministic 1% approximations; these differences do not establish promotion causality.

### 2. What is inferred

The exact all-missing-to-all-recorded transition on 2014-04-01 strongly supports a **historical/data-era missingness pattern**, not broadly random row-level omission. It is plausible that promotion status was unavailable or not represented in the earlier source records. That cause is an inference from the data pattern, not a documented source-system fact.

### 3. What remains uncertain

- The notebook has no direct authoritative evidence explaining why pre-2014-04-01 values are missing.
- It does not prove that earlier rows were not promoted; therefore missing cannot be equated with `False`.
- A later missing indicator or explicit unknown category would also act as a pre/post-2014 era marker and must be assessed for temporal confounding or leakage in the modelling design.
- Sales-status comparisons are descriptive, and the median/quartile estimates use a deterministic 1% sample for memory safety.

### 4. Recommended provisional cleaning rule

**Option C is best supported: preserve missing `onpromotion` values and, only during later approved feature engineering, represent them with an explicit unknown category or missing indicator.**

Operationally, this includes **Option B now**: retain missing values as unknown in the cleaned data. Do **not** convert missing values to `False`, and do not impute them from nearby dates or store-item history. No rule is applied in this notebook.

### 5. Carry into the final SCRUM-9 cleaning-rules table

- **Source column:** `onpromotion`
- **Observed issue:** 21,657,651 missing rows, confined to 2013-01-01 through 2014-03-31.
- **Cleaning-stage rule:** preserve nullable values unchanged; missing means unknown, not `False`.
- **Later feature-stage candidate:** explicit unknown category and/or missing indicator, evaluated as a historical-era signal before modelling.
- **Prohibited treatment:** silent conversion to `False` or history-based imputation without authoritative semantics.
- **Validation checks:** retain exact `True`/`False`/missing reconciliation and the 2014-04-01 transition evidence.
- **Open uncertainty:** confirm the historical recording semantics from authoritative Favorita/Kaggle documentation or domain evidence.


# 25. `transactions` missing-value investigation

`transactions` is the number of transactions recorded for a store on a specific date. It is joined at `(date, store_nbr)` grain and repeated across all item rows for that store-date. Therefore, missing item-level rows must be reduced to unique store-date combinations before reasoning about root missingness.

This investigation tests whether missingness is associated with isolated source gaps, store operating boundaries, temporal blocks, holidays, or another observed pattern. No cleaning, imputation, interpolation, filling, row removal, or feature engineering is applied.

# 26. Overall `transactions` coverage

A single selected-column row-group scan builds a compact store-date table. Partial aggregates are combined across row-group boundaries, and consistency checks verify that each store-date has either one repeated transaction value or is entirely missing.

In [21]:
TRANSACTION_SCAN_COLUMNS = [
    "date", "store_nbr", "item_nbr", "unit_sales", "transactions", "is_holiday", "holiday_type"
]
missing_transaction_columns = [column for column in TRANSACTION_SCAN_COLUMNS if column not in parquet_file.schema_arrow.names]
if missing_transaction_columns:
    raise ValueError(f"Missing required transaction investigation columns: {missing_transaction_columns}")

store_date_parts = []
item_row_missing_transactions = 0
item_row_non_missing_transactions = 0

for row_group_index in range(parquet_file.metadata.num_row_groups):
    scan_frame = parquet_file.read_row_group(
        row_group_index,
        columns=TRANSACTION_SCAN_COLUMNS,
    ).to_pandas()
    transaction_missing = scan_frame["transactions"].isna()
    item_row_missing_transactions += int(transaction_missing.sum())
    item_row_non_missing_transactions += int((~transaction_missing).sum())
    scan_frame["_positive_sales_item"] = scan_frame["unit_sales"].gt(0).astype("int64")
    scan_frame["_negative_sales_item"] = scan_frame["unit_sales"].lt(0).astype("int64")
    scan_frame["_transaction_missing_item"] = transaction_missing.astype("int64")

    partial = scan_frame.groupby(["date", "store_nbr"], as_index=False, observed=True).agg(
        number_of_item_rows=("item_nbr", "size"),
        sum_unit_sales=("unit_sales", "sum"),
        positive_sales_item_count=("_positive_sales_item", "sum"),
        negative_sales_item_count=("_negative_sales_item", "sum"),
        transaction_missing_item_rows=("_transaction_missing_item", "sum"),
        transaction_value_min=("transactions", "min"),
        transaction_value_max=("transactions", "max"),
        is_holiday=("is_holiday", "max"),
        holiday_type=("holiday_type", "first"),
    )
    store_date_parts.append(partial)

store_date = pd.concat(store_date_parts, ignore_index=True).groupby(
    ["date", "store_nbr"], as_index=False, observed=True
).agg(
    number_of_item_rows=("number_of_item_rows", "sum"),
    sum_unit_sales=("sum_unit_sales", "sum"),
    positive_sales_item_count=("positive_sales_item_count", "sum"),
    negative_sales_item_count=("negative_sales_item_count", "sum"),
    transaction_missing_item_rows=("transaction_missing_item_rows", "sum"),
    transaction_value_min=("transaction_value_min", "min"),
    transaction_value_max=("transaction_value_max", "max"),
    is_holiday=("is_holiday", "max"),
    holiday_type=("holiday_type", "first"),
).sort_values(["store_nbr", "date"]).reset_index(drop=True)

mixed_missing_store_dates = store_date.loc[
    store_date["transaction_missing_item_rows"].between(1, store_date["number_of_item_rows"] - 1)
]
inconsistent_transaction_values = store_date.loc[
    store_date["transaction_value_min"].notna()
    & store_date["transaction_value_max"].notna()
    & store_date["transaction_value_min"].ne(store_date["transaction_value_max"])
]
if not mixed_missing_store_dates.empty:
    raise ValueError("Some store-dates mix missing and non-missing transaction values across item rows.")
if not inconsistent_transaction_values.empty:
    raise ValueError("Some store-dates contain inconsistent repeated transaction values.")

store_date["transactions"] = store_date["transaction_value_min"]
store_date["transactions_missing"] = store_date["transactions"].isna()
store_date["mean_unit_sales"] = store_date["sum_unit_sales"] / store_date["number_of_item_rows"]
store_date["year"] = store_date["date"].dt.year
store_date["year_month"] = store_date["date"].dt.to_period("M").astype(str)
store_date["weekday"] = store_date["date"].dt.day_name()

total_store_dates = len(store_date)
missing_store_dates_count = int(store_date["transactions_missing"].sum())
non_missing_store_dates_count = total_store_dates - missing_store_dates_count

item_row_coverage = pd.DataFrame([{
    "grain": "item row",
    "total_count": total_merged_rows,
    "non_missing_transactions_count": item_row_non_missing_transactions,
    "missing_transactions_count": item_row_missing_transactions,
    "missing_percentage": 100.0 * item_row_missing_transactions / total_merged_rows,
}])
store_date_coverage = pd.DataFrame([{
    "grain": "unique date + store_nbr",
    "total_count": total_store_dates,
    "non_missing_transactions_count": non_missing_store_dates_count,
    "missing_transactions_count": missing_store_dates_count,
    "missing_percentage": 100.0 * missing_store_dates_count / total_store_dates,
}])
display(item_row_coverage)
display(store_date_coverage)
print(f"Item-row reconciliation: {item_row_missing_transactions + item_row_non_missing_transactions:,} rows")

,grain,total_count,non_missing_transactions_count,missing_transactions_count,missing_percentage
0,item row,125497040,125282415,214625,0.17102


,grain,total_count,non_missing_transactions_count,missing_transactions_count,missing_percentage
0,unique date + store_nbr,83606,83488,118,0.141138


Item-row reconciliation: 125,497,040 rows


# 27. Missing `transactions` store-date table

In [22]:
missing_store_dates = store_date.loc[store_date["transactions_missing"], [
    "date", "store_nbr", "number_of_item_rows", "sum_unit_sales", "mean_unit_sales",
    "positive_sales_item_count", "negative_sales_item_count", "is_holiday", "holiday_type",
    "year", "year_month", "weekday",
]].copy().sort_values(["date", "store_nbr"]).reset_index(drop=True)

display(pd.DataFrame([{"unique_missing_store_date_count": len(missing_store_dates)}]))
display(missing_store_dates.head(30))

,unique_missing_store_date_count
0,118


,date,store_nbr,number_of_item_rows,sum_unit_sales,mean_unit_sales,positive_sales_item_count,negative_sales_item_count,is_holiday,holiday_type,year,year_month,weekday
0,2013-06-19,10,652,3802.2920,5.831736,652,0,False,<NA>,2013,2013-06,Wednesday
1,2013-06-19,35,387,1699.0480,4.390305,387,0,False,<NA>,2013,2013-06,Wednesday
2,2013-06-19,43,645,4642.4950,7.197667,645,0,False,<NA>,2013,2013-06,Wednesday
3,2013-06-19,54,583,2977.8520,5.107808,583,0,False,<NA>,2013,2013-06,Wednesday
4,2014-01-02,32,742,3146.1467,4.240090,742,0,False,<NA>,2014,2014-01,Thursday
5,2014-03-24,25,1288,8014.1600,6.222174,1288,0,False,<NA>,2014,2014-03,Monday
6,2016-01-01,25,1718,16433.3940,9.565421,1718,0,True,Holiday,2016,2016-01,Friday
7,2016-01-02,1,1788,8877.1750,4.964863,1788,0,False,<NA>,2016,2016-01,Saturday
8,2016-01-02,3,2573,47348.5720,18.402088,2573,0,False,<NA>,2016,2016-01,Saturday
9,2016-01-02,4,2152,15849.8190,7.365158,2152,0,False,<NA>,2016,2016-01,Saturday


# 28. Missingness over time

In [23]:
def store_date_missing_summary(frame, keys):
    summary = frame.groupby(keys, as_index=False, observed=True).agg(
        represented_store_date_count=("transactions_missing", "size"),
        missing_store_date_count=("transactions_missing", "sum"),
    )
    summary["missing_percentage"] = (
        100.0 * summary["missing_store_date_count"] / summary["represented_store_date_count"]
    )
    return summary

transaction_missing_by_year = store_date_missing_summary(store_date, ["year"]).sort_values("year")
transaction_missing_by_month = store_date_missing_summary(store_date, ["year_month"]).sort_values("year_month")
transaction_missing_by_date = store_date_missing_summary(store_date, ["date"]).sort_values("date")
top_dates_by_missing_stores = transaction_missing_by_date.sort_values(
    ["missing_store_date_count", "missing_percentage", "date"], ascending=[False, False, True]
).head(20)
all_represented_stores_missing_dates = transaction_missing_by_date.loc[
    transaction_missing_by_date["missing_store_date_count"].eq(
        transaction_missing_by_date["represented_store_date_count"]
    )
]

display(transaction_missing_by_year)
display(transaction_missing_by_month)
display(top_dates_by_missing_stores)
display(Markdown("### Dates where every represented store is missing transactions"))
display(all_represented_stores_missing_dates)

,year,represented_store_date_count,missing_store_date_count,missing_percentage
0,2013,16912,4,0.023652
1,2014,17140,2,0.011669
2,2015,18346,0,0.000000
3,2016,19111,112,0.586050
4,2017,12097,0,0.000000


,year_month,represented_store_date_count,missing_store_date_count,missing_percentage
0,2013-01,1381,0,0.000000
1,2013-02,1288,0,0.000000
2,2013-03,1426,0,0.000000
3,2013-04,1380,0,0.000000
4,2013-05,1449,0,0.000000
5,2013-06,1410,4,0.283688
6,2013-07,1434,0,0.000000
7,2013-08,1457,0,0.000000
8,2013-09,1410,0,0.000000
9,2013-10,1457,0,0.000000


,date,represented_store_date_count,missing_store_date_count,missing_percentage
1094,2016-01-03,53,53,100.000000
1095,2016-01-04,53,39,73.584906
1093,2016-01-02,53,17,32.075472
169,2013-06-19,47,4,8.510638
1362,2016-09-27,51,2,3.921569
1092,2016-01-01,1,1,100.000000
365,2014-01-02,47,1,2.127660
446,2014-03-24,47,1,2.127660
0,2013-01-01,1,0,0.000000
1,2013-01-02,46,0,0.000000


### Dates where every represented store is missing transactions

,date,represented_store_date_count,missing_store_date_count,missing_percentage
1092,2016-01-01,1,1,100.0
1094,2016-01-03,53,53,100.0


# 29. Missingness by store

In [24]:
store_transaction_coverage = store_date.groupby("store_nbr", as_index=False, observed=True).agg(
    represented_date_count=("date", "size"),
    missing_transaction_date_count=("transactions_missing", "sum"),
    first_represented_date=("date", "min"),
    last_represented_date=("date", "max"),
)
store_missing_bounds = missing_store_dates.groupby("store_nbr", as_index=False, observed=True).agg(
    first_missing_date=("date", "min"),
    last_missing_date=("date", "max"),
)
store_transaction_coverage = store_transaction_coverage.merge(
    store_missing_bounds, on="store_nbr", how="left", validate="one_to_one"
)
store_transaction_coverage["missing_percentage"] = (
    100.0 * store_transaction_coverage["missing_transaction_date_count"]
    / store_transaction_coverage["represented_date_count"]
)
stores_affected_by_transaction_missingness = int(
    (store_transaction_coverage["missing_transaction_date_count"] > 0).sum()
)

display(pd.DataFrame([{
    "stores_affected": stores_affected_by_transaction_missingness,
    "observed_stores": len(store_transaction_coverage),
    "percentage_of_stores_affected": 100.0 * stores_affected_by_transaction_missingness / len(store_transaction_coverage),
}]))
display(Markdown("### Top 20 stores by missing date count"))
display(store_transaction_coverage.sort_values(
    ["missing_transaction_date_count", "missing_percentage"], ascending=False
).head(20))
display(Markdown("### Top 20 stores by missing percentage"))
display(store_transaction_coverage.sort_values(
    ["missing_percentage", "missing_transaction_date_count"], ascending=False
).head(20))

,stores_affected,observed_stores,percentage_of_stores_affected
0,53,54,98.148148


### Top 20 stores by missing date count

,store_nbr,represented_date_count,missing_transaction_date_count,first_represented_date,last_represented_date,first_missing_date,last_missing_date,missing_percentage
6,7,1679,4,2013-01-02,2017-08-15,2016-01-02,2016-09-27,0.238237
9,10,1679,4,2013-01-02,2017-08-15,2013-06-19,2016-01-04,0.238237
17,18,1569,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.191205
24,25,1618,3,2013-01-01,2017-08-15,2014-03-24,2016-01-03,0.185414
11,12,1619,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.185300
13,14,1641,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.182815
42,43,1675,3,2013-01-02,2017-08-15,2013-06-19,2016-01-04,0.179104
16,17,1677,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.178891
2,3,1679,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.178678
3,4,1679,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.178678


### Top 20 stores by missing percentage

,store_nbr,represented_date_count,missing_transaction_date_count,first_represented_date,last_represented_date,first_missing_date,last_missing_date,missing_percentage
21,22,673,2,2015-10-09,2017-08-15,2016-01-03,2016-01-04,0.297177
41,42,722,2,2015-08-21,2017-08-15,2016-01-03,2016-01-04,0.277008
20,21,750,2,2015-07-24,2017-08-15,2016-01-03,2016-01-04,0.266667
6,7,1679,4,2013-01-02,2017-08-15,2016-01-02,2016-09-27,0.238237
9,10,1679,4,2013-01-02,2017-08-15,2013-06-19,2016-01-04,0.238237
28,29,876,2,2015-03-20,2017-08-15,2016-01-03,2016-01-04,0.228311
19,20,911,2,2015-02-13,2017-08-15,2016-01-03,2016-01-04,0.219539
17,18,1569,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.191205
24,25,1618,3,2013-01-01,2017-08-15,2014-03-24,2016-01-03,0.185414
11,12,1619,3,2013-01-02,2017-08-15,2016-01-02,2016-01-04,0.185300


# 30. Store operating-boundary investigation

A transparent **7-calendar-day** window is used around each store's first and last represented date. Categories are mutually exclusive: within both boundaries, near start, near end, or interior. Boundary proximity is descriptive and does not prove that a store was closed.

In [25]:
STORE_BOUNDARY_WINDOW_DAYS = 7
missing_with_store_bounds = missing_store_dates.merge(
    store_transaction_coverage[["store_nbr", "first_represented_date", "last_represented_date"]],
    on="store_nbr", how="left", validate="many_to_one",
)
missing_with_store_bounds["days_from_store_start"] = (
    missing_with_store_bounds["date"] - missing_with_store_bounds["first_represented_date"]
).dt.days
missing_with_store_bounds["days_to_store_end"] = (
    missing_with_store_bounds["last_represented_date"] - missing_with_store_bounds["date"]
).dt.days
near_start = missing_with_store_bounds["days_from_store_start"].between(0, STORE_BOUNDARY_WINDOW_DAYS)
near_end = missing_with_store_bounds["days_to_store_end"].between(0, STORE_BOUNDARY_WINDOW_DAYS)
missing_with_store_bounds["boundary_classification"] = np.select(
    [near_start & near_end, near_start, near_end],
    ["near both store start and end", "near store start", "near store end"],
    default="interior",
)
boundary_summary = missing_with_store_bounds["boundary_classification"].value_counts().rename_axis(
    "boundary_classification"
).reset_index(name="missing_store_date_count")
boundary_summary["percentage_of_missing_store_dates"] = (
    100.0 * boundary_summary["missing_store_date_count"] / missing_store_dates_count
)
display(boundary_summary)

,boundary_classification,missing_store_date_count,percentage_of_missing_store_dates
0,interior,118,100.0


# 31. Continuity and gap structure

In [26]:
missing_blocks = []
for store_nbr, group in missing_store_dates.groupby("store_nbr", observed=True):
    dates = group["date"].sort_values().drop_duplicates().reset_index(drop=True)
    block_id = dates.diff().ne(pd.Timedelta(days=1)).cumsum()
    blocks = dates.groupby(block_id).agg(["min", "max", "size"]).reset_index(drop=True)
    for block in blocks.itertuples(index=False):
        missing_blocks.append({
            "store_nbr": int(store_nbr),
            "block_start": block.min,
            "block_end": block.max,
            "block_length_days": int(block.size),
        })
missing_blocks = pd.DataFrame(missing_blocks).sort_values(
    ["block_length_days", "block_start", "store_nbr"], ascending=[False, True, True]
).reset_index(drop=True)

store_gap_summary = missing_blocks.groupby("store_nbr", as_index=False).agg(
    missing_block_count=("block_length_days", "size"),
    longest_consecutive_missing_block=("block_length_days", "max"),
    median_block_length=("block_length_days", "median"),
    isolated_single_day_gaps=("block_length_days", lambda values: int((values == 1).sum())),
    multi_day_gaps=("block_length_days", lambda values: int((values > 1).sum())),
).sort_values(["longest_consecutive_missing_block", "missing_block_count"], ascending=False)

global_gap_summary = pd.DataFrame([{
    "total_missing_blocks": len(missing_blocks),
    "isolated_single_day_blocks": int((missing_blocks["block_length_days"] == 1).sum()),
    "percentage_isolated_single_day": 100.0 * (missing_blocks["block_length_days"] == 1).mean(),
    "multi_day_blocks": int((missing_blocks["block_length_days"] > 1).sum()),
    "longest_global_store_specific_block_days": int(missing_blocks["block_length_days"].max()),
}])
display(global_gap_summary)
display(store_gap_summary.head(20))
display(Markdown("### Longest missing transaction blocks"))
display(missing_blocks.head(30))

,total_missing_blocks,isolated_single_day_blocks,percentage_isolated_single_day,multi_day_blocks,longest_global_store_specific_block_days
0,62,21,33.870968,41,3


,store_nbr,missing_block_count,longest_consecutive_missing_block,median_block_length,isolated_single_day_gaps,multi_day_gaps
6,7,2,3,2.0,1,1
9,10,2,3,2.0,1,1
2,3,1,3,3.0,0,1
3,4,1,3,3.0,0,1
5,6,1,3,3.0,0,1
7,8,1,3,3.0,0,1
8,9,1,3,3.0,0,1
10,11,1,3,3.0,0,1
11,12,1,3,3.0,0,1
12,13,1,3,3.0,0,1


### Longest missing transaction blocks

,store_nbr,block_start,block_end,block_length_days
0,3,2016-01-02,2016-01-04,3
1,4,2016-01-02,2016-01-04,3
2,6,2016-01-02,2016-01-04,3
3,7,2016-01-02,2016-01-04,3
4,8,2016-01-02,2016-01-04,3
5,9,2016-01-02,2016-01-04,3
6,10,2016-01-02,2016-01-04,3
7,11,2016-01-02,2016-01-04,3
8,12,2016-01-02,2016-01-04,3
9,13,2016-01-02,2016-01-04,3


# 32. Nearby transaction context

The deterministic sample contains up to ten isolated missing store-dates and up to ten cases selected from the longest missing blocks. The compact store-date table supplies same-store observed transaction values for ±7 calendar days; no imputed value is calculated.

In [27]:
isolated_cases = missing_blocks.loc[missing_blocks["block_length_days"].eq(1)].sort_values(
    ["block_start", "store_nbr"]
).head(10).rename(columns={"block_start": "selected_missing_date"})
isolated_cases["selection_reason"] = "isolated single-day gap"

longest_block_cases = missing_blocks.sort_values(
    ["block_length_days", "block_start", "store_nbr"], ascending=[False, True, True]
).head(20).copy()
longest_block_cases["selected_missing_date"] = longest_block_cases["block_start"]
longest_block_cases = longest_block_cases.loc[
    ~longest_block_cases.set_index(["store_nbr", "selected_missing_date"]).index.isin(
        isolated_cases.set_index(["store_nbr", "selected_missing_date"]).index
    )
].head(10)
longest_block_cases["selection_reason"] = "case from longest missing block"

selected_transaction_cases = pd.concat([
    isolated_cases[["store_nbr", "selected_missing_date", "block_length_days", "selection_reason"]],
    longest_block_cases[["store_nbr", "selected_missing_date", "block_length_days", "selection_reason"]],
], ignore_index=True).drop_duplicates(["store_nbr", "selected_missing_date"]).reset_index(drop=True)
selected_transaction_cases["case_id"] = np.arange(1, len(selected_transaction_cases) + 1)
display(selected_transaction_cases)

transaction_context_parts = []
for case in selected_transaction_cases.itertuples(index=False):
    context = store_date.loc[
        store_date["store_nbr"].eq(case.store_nbr)
        & store_date["date"].between(
            case.selected_missing_date - pd.Timedelta(days=7),
            case.selected_missing_date + pd.Timedelta(days=7),
        ),
        ["date", "store_nbr", "transactions", "transactions_missing", "number_of_item_rows", "sum_unit_sales"],
    ].copy()
    context["case_id"] = case.case_id
    context["selected_missing_date"] = case.selected_missing_date
    context["relative_calendar_day"] = (context["date"] - case.selected_missing_date).dt.days
    transaction_context_parts.append(context)
transaction_context = pd.concat(transaction_context_parts, ignore_index=True).sort_values(["case_id", "date"])

nearby_context_summary = transaction_context.groupby("case_id", as_index=False).agg(
    observed_store_dates=("date", "size"),
    valid_nearby_transaction_dates=("transactions", "count"),
    missing_transaction_dates=("transactions_missing", "sum"),
    minimum_nearby_transactions=("transactions", "min"),
    median_nearby_transactions=("transactions", "median"),
    maximum_nearby_transactions=("transactions", "max"),
).merge(selected_transaction_cases, on="case_id", how="left", validate="one_to_one")
display(nearby_context_summary)

for case in selected_transaction_cases.itertuples(index=False):
    display(Markdown(
        f"### Case {case.case_id}: store `{case.store_nbr}`, missing date "
        f"`{case.selected_missing_date.date().isoformat()}`, {case.selection_reason}"
    ))
    display(transaction_context.loc[
        transaction_context["case_id"].eq(case.case_id),
        ["date", "relative_calendar_day", "transactions", "transactions_missing", "number_of_item_rows", "sum_unit_sales"],
    ])

,store_nbr,selected_missing_date,block_length_days,selection_reason,case_id
0,10,2013-06-19,1,isolated single-day gap,1
1,35,2013-06-19,1,isolated single-day gap,2
2,43,2013-06-19,1,isolated single-day gap,3
3,54,2013-06-19,1,isolated single-day gap,4
4,32,2014-01-02,1,isolated single-day gap,5
5,25,2014-03-24,1,isolated single-day gap,6
6,25,2016-01-01,1,isolated single-day gap,7
7,23,2016-01-03,1,isolated single-day gap,8
8,24,2016-01-03,1,isolated single-day gap,9
9,25,2016-01-03,1,isolated single-day gap,10


,case_id,observed_store_dates,valid_nearby_transaction_dates,missing_transaction_dates,minimum_nearby_transactions,median_nearby_transactions,maximum_nearby_transactions,store_nbr,selected_missing_date,block_length_days,selection_reason
0,1,15,14,1,912.0,1071.0,1355.0,10,2013-06-19,1,isolated single-day gap
1,2,15,14,1,393.0,523.0,677.0,35,2013-06-19,1,isolated single-day gap
2,3,15,14,1,1012.0,1163.5,1381.0,43,2013-06-19,1,isolated single-day gap
3,4,15,14,1,643.0,762.5,1318.0,54,2013-06-19,1,isolated single-day gap
4,5,14,13,1,523.0,625.0,1465.0,32,2014-01-02,1,isolated single-day gap
5,6,15,14,1,783.0,945.5,1877.0,25,2014-03-24,1,isolated single-day gap
6,7,14,12,2,724.0,1465.0,3178.0,25,2016-01-01,1,isolated single-day gap
7,8,14,13,1,895.0,1101.0,1467.0,23,2016-01-03,1,isolated single-day gap
8,9,14,13,1,1829.0,2486.0,3365.0,24,2016-01-03,1,isolated single-day gap
9,10,15,13,2,724.0,1298.0,3178.0,25,2016-01-03,1,isolated single-day gap


### Case 1: store `10`, missing date `2013-06-19`, isolated single-day gap

,date,relative_calendar_day,transactions,transactions_missing,number_of_item_rows,sum_unit_sales
0,2013-06-12,-7,983.0,False,650,3665.370
1,2013-06-13,-6,961.0,False,666,3236.962
2,2013-06-14,-5,951.0,False,668,3260.324
3,2013-06-15,-4,1355.0,False,772,5889.830
4,2013-06-16,-3,1148.0,False,749,5912.806
5,2013-06-17,-2,1173.0,False,705,5414.866
6,2013-06-18,-1,1117.0,False,687,4365.031
7,2013-06-19,0,NaN,True,652,3802.292
8,2013-06-20,1,1026.0,False,685,4350.513
9,2013-06-21,2,912.0,False,681,3725.361


### Case 2: store `35`, missing date `2013-06-19`, isolated single-day gap

...

,date,relative_calendar_day,transactions,transactions_missing,number_of_item_rows,sum_unit_sales
244,2015-12-26,-7,3222.0,False,1896,23979.269
245,2015-12-27,-6,2772.0,False,1934,20831.158
246,2015-12-28,-5,2473.0,False,1808,18212.795
247,2015-12-29,-4,2750.0,False,1908,20965.192
248,2015-12-30,-3,2850.0,False,1868,21449.776
249,2015-12-31,-2,3443.0,False,1909,23157.599
250,2016-01-02,0,NaN,True,2057,30280.388
251,2016-01-03,1,NaN,True,2029,32740.993
252,2016-01-04,2,NaN,True,1865,20849.429
253,2016-01-05,3,2395.0,False,1879,20233.458


### Case 19: store `12`, missing date `2016-01-02`, case from longest missing block

,date,relative_calendar_day,transactions,transactions_missing,number_of_item_rows,sum_unit_sales
258,2015-12-26,-7,1438.0,False,1441,8982.637
259,2015-12-27,-6,1187.0,False,1359,7014.720
260,2015-12-28,-5,1218.0,False,1325,7228.654
261,2015-12-29,-4,1331.0,False,1384,7808.446
262,2015-12-30,-3,1315.0,False,1327,7391.725
263,2015-12-31,-2,1621.0,False,1344,7758.002
264,2016-01-02,0,NaN,True,1510,9566.088
265,2016-01-03,1,NaN,True,1511,11174.043
266,2016-01-04,2,NaN,True,1403,9214.916
267,2016-01-05,3,1274.0,False,1430,8689.454


### Case 20: store `13`, missing date `2016-01-02`, case from longest missing block

,date,relative_calendar_day,transactions,transactions_missing,number_of_item_rows,sum_unit_sales
272,2015-12-26,-7,1308.0,False,1322,9236.406
273,2015-12-27,-6,894.0,False,1259,6610.831
274,2015-12-28,-5,876.0,False,1191,6514.848
275,2015-12-29,-4,975.0,False,1220,7258.132
276,2015-12-30,-3,1072.0,False,1289,7847.694
277,2015-12-31,-2,1114.0,False,1194,7456.002
278,2016-01-02,0,NaN,True,1476,14200.060
279,2016-01-03,1,NaN,True,1362,9037.255
280,2016-01-04,2,NaN,True,1318,8065.693
281,2016-01-05,3,913.0,False,1241,7984.793


# 33. Relationship between missing `transactions` and `unit_sales`

This is a store-date-level descriptive comparison. Sales values are not used to backfill transactions, and differences are not interpreted causally.

In [28]:
store_date["transaction_coverage_status"] = np.where(
    store_date["transactions_missing"], "missing transactions", "valid transactions"
)
sales_activity_comparison = store_date.groupby("transaction_coverage_status", as_index=False).agg(
    represented_store_dates=("date", "size"),
    mean_daily_total_unit_sales=("sum_unit_sales", "mean"),
    median_daily_total_unit_sales=("sum_unit_sales", "median"),
    mean_item_row_count=("number_of_item_rows", "mean"),
    median_item_row_count=("number_of_item_rows", "median"),
    total_positive_sales_items=("positive_sales_item_count", "sum"),
    total_negative_sales_items=("negative_sales_item_count", "sum"),
    holiday_rate_percentage=("is_holiday", lambda values: 100.0 * values.mean()),
)
display(sales_activity_comparison)
display(missing_store_dates[[
    "date", "store_nbr", "sum_unit_sales", "positive_sales_item_count",
    "negative_sales_item_count", "number_of_item_rows", "is_holiday",
]].sort_values("sum_unit_sales", ascending=False).head(30))

,transaction_coverage_status,represented_store_dates,mean_daily_total_unit_sales,median_daily_total_unit_sales,mean_item_row_count,median_item_row_count,total_positive_sales_items,total_negative_sales_items,holiday_rate_percentage
0,missing transactions,118,20186.728269,15821.1575,1818.855932,1864.5,214616,9,0.847458
1,valid transactions,83488,12830.924622,10014.8675,1500.603859,1417.0,125274629,7786,8.082599


,date,store_nbr,sum_unit_sales,positive_sales_item_count,negative_sales_item_count,number_of_item_rows,is_holiday
68,2016-01-03,45,74597.099,2599,0,2599,False
67,2016-01-03,44,74026.946,2593,0,2593,False
70,2016-01-03,47,67248.303,2556,1,2557,False
69,2016-01-03,46,58926.382,2567,0,2567,False
26,2016-01-03,3,56595.148,2537,1,2538,False
71,2016-01-03,48,54569.517,2503,0,2503,False
72,2016-01-03,49,53416.107,2541,0,2541,False
106,2016-01-04,44,52673.883,2513,0,2513,False
107,2016-01-04,45,50025.920,2530,0,2530,False
78,2016-01-04,3,47439.963,2484,0,2484,False


# 34. Holiday relationship

In [29]:
holiday_transaction_missingness = store_date.groupby("is_holiday", as_index=False).agg(
    represented_store_date_count=("date", "size"),
    missing_store_date_count=("transactions_missing", "sum"),
)
holiday_transaction_missingness["missing_percentage"] = (
    100.0 * holiday_transaction_missingness["missing_store_date_count"]
    / holiday_transaction_missingness["represented_store_date_count"]
)
holiday_transaction_missingness["date_type"] = np.where(
    holiday_transaction_missingness["is_holiday"], "holiday", "non-holiday"
)
display(holiday_transaction_missingness[[
    "date_type", "represented_store_date_count", "missing_store_date_count", "missing_percentage"
]])

missing_by_holiday_type = missing_store_dates.assign(
    holiday_type_display=missing_store_dates["holiday_type"].fillna("No applicable holiday record")
).groupby("holiday_type_display", as_index=False).agg(
    missing_store_date_count=("date", "size")
).sort_values("missing_store_date_count", ascending=False)
display(missing_by_holiday_type)

,date_type,represented_store_date_count,missing_store_date_count,missing_percentage
0,non-holiday,76857,117,0.152231
1,holiday,6749,1,0.014817


,holiday_type_display,missing_store_date_count
1,No applicable holiday record,117
0,Holiday,1


# 35. Valid `transactions` distribution

Distribution statistics use each unique store-date once. Store-specific medians and interquartile ranges describe typical variation but are not applied as imputations.

In [30]:
valid_transaction_store_dates = store_date.loc[~store_date["transactions_missing"]].copy()
valid_transactions = valid_transaction_store_dates["transactions"].astype("float64")
transaction_distribution = pd.DataFrame([{
    "count": int(valid_transactions.count()),
    "minimum": float(valid_transactions.min()),
    "maximum": float(valid_transactions.max()),
    "mean": float(valid_transactions.mean()),
    "median": float(valid_transactions.median()),
    "standard_deviation": float(valid_transactions.std()),
    "p1": float(valid_transactions.quantile(0.01)),
    "p5": float(valid_transactions.quantile(0.05)),
    "p25": float(valid_transactions.quantile(0.25)),
    "p75": float(valid_transactions.quantile(0.75)),
    "p95": float(valid_transactions.quantile(0.95)),
    "p99": float(valid_transactions.quantile(0.99)),
}])
store_transaction_distribution = valid_transaction_store_dates.groupby("store_nbr", as_index=False).agg(
    valid_date_count=("transactions", "size"),
    minimum_transactions=("transactions", "min"),
    p25_transactions=("transactions", lambda values: values.quantile(0.25)),
    median_transactions=("transactions", "median"),
    p75_transactions=("transactions", lambda values: values.quantile(0.75)),
    maximum_transactions=("transactions", "max"),
)
display(transaction_distribution)
display(store_transaction_distribution)

,count,minimum,maximum,mean,median,standard_deviation,p1,p5,p25,p75,p95,p99
0,83488,5.0,8359.0,1694.602158,1393.0,963.286644,516.0,654.0,1046.0,2079.0,3712.0,4862.0


,store_nbr,valid_date_count,minimum_transactions,p25_transactions,median_transactions,p75_transactions,maximum_transactions
0,1,1676,10.0,1325.75,1746.0,1832.00,3023.0
1,2,1677,6.0,1795.00,1889.0,2006.00,4060.0
2,3,1676,2213.0,2888.75,3100.5,3457.00,6085.0
3,4,1676,785.0,1355.00,1455.0,1613.00,3589.0
4,5,1677,721.0,1263.00,1379.0,1496.00,3468.0
5,6,1676,1380.0,1621.00,1729.0,2037.00,4256.0
6,7,1675,1050.0,1684.50,1790.0,1892.00,3023.0
7,8,1676,1360.0,2595.00,2740.5,2887.25,5261.0
8,9,1676,1462.0,1867.00,2045.0,2271.00,4624.0
9,10,1675,615.0,866.50,962.0,1082.00,2242.0


# 36. Candidate cleaning approaches — evidence comparison

In [31]:
candidate_transaction_approaches = pd.DataFrame([
    {
        "Candidate approach": "A. Preserve missing transactions unchanged",
        "Potential benefit": "Maintains provenance and avoids invented operational counts.",
        "Main risk": "Downstream models must handle missingness explicitly or omit the field.",
        "Evidence supporting it": "Missingness is small and source semantics for gaps are unverified.",
        "Evidence against it": "Some algorithms cannot consume nullable numeric inputs directly.",
        "Appropriate for cleaning stage?": "Yes",
    },
    {
        "Candidate approach": "B. Forward fill by store",
        "Potential benefit": "Uses only earlier observed values when applied causally.",
        "Main risk": "Transactions vary by weekday, season, events, and store conditions.",
        "Evidence supporting it": "Nearby context can show whether short isolated gaps have stable predecessors.",
        "Evidence against it": "A prior day's count is not the missing date's observed count.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "C. Backward fill by store",
        "Potential benefit": "Supplies a nearby numeric value.",
        "Main risk": "Uses future information and creates temporal leakage in historical validation.",
        "Evidence supporting it": "Nearby future observations often exist.",
        "Evidence against it": "Future-aware backfill is not deployment-safe and invents source data.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "D. Linear interpolation by store over time",
        "Potential benefit": "Can bridge short numeric gaps smoothly.",
        "Main risk": "Uses future values unless constrained; transaction trajectories are not necessarily linear.",
        "Evidence supporting it": "Could be evaluated later for isolated interior gaps.",
        "Evidence against it": "No evidence proves linearity, and future-aware interpolation leaks information.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "E. Store-specific median",
        "Potential benefit": "Robust to extreme transaction days and store scale.",
        "Main risk": "Ignores date, weekday, season, and store evolution.",
        "Evidence supporting it": "Store-specific distributions have stable summaries.",
        "Evidence against it": "Median replacement may distort genuine daily variation.",
        "Appropriate for cleaning stage?": "TBD",
    },
    {
        "Candidate approach": "F. Date-level median across stores",
        "Potential benefit": "Reflects common calendar conditions.",
        "Main risk": "Stores have different transaction scales and operating contexts.",
        "Evidence supporting it": "Multiple stores are represented on most dates.",
        "Evidence against it": "Cross-store medians are not store-specific observations.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "G. Drop transactions later from modelling",
        "Potential benefit": "Avoids missingness and future-availability risk.",
        "Main risk": "May discard useful historical demand context.",
        "Evidence supporting it": "Actual future transaction counts may be unavailable at forecast serving time.",
        "Evidence against it": "Predictive value has not yet been evaluated with temporal validation.",
        "Appropriate for cleaning stage?": "TBD - modelling decision",
    },
    {
        "Candidate approach": "H. Preserve now; evaluate causal lagged/availability-safe use later",
        "Potential benefit": "Separates source cleaning from leakage-safe feature design.",
        "Main risk": "Requires explicit temporal validation and serving-contract review.",
        "Evidence supporting it": "Current analysis cannot justify an exact imputed count.",
        "Evidence against it": "Leaves a later modelling decision unresolved by design.",
        "Appropriate for cleaning stage?": "Yes",
    },
])
display(candidate_transaction_approaches)
display(Markdown(
    "**Temporal leakage boundary:** backward fill or unconstrained interpolation can use future information. "
    "Cleaning-stage preservation and modelling-stage leakage safety are separate decisions."
))

,Candidate approach,Potential benefit,Main risk,Evidence supporting it,Evidence against it,Appropriate for cleaning stage?
0,A. Preserve missing transactions unchanged,Maintains provenance and avoids invented opera...,Downstream models must handle missingness expl...,Missingness is small and source semantics for ...,Some algorithms cannot consume nullable numeri...,Yes
1,B. Forward fill by store,Uses only earlier observed values when applied...,"Transactions vary by weekday, season, events, ...",Nearby context can show whether short isolated...,A prior day's count is not the missing date's ...,No
2,C. Backward fill by store,Supplies a nearby numeric value.,Uses future information and creates temporal l...,Nearby future observations often exist.,Future-aware backfill is not deployment-safe a...,No
3,D. Linear interpolation by store over time,Can bridge short numeric gaps smoothly.,Uses future values unless constrained; transac...,Could be evaluated later for isolated interior...,"No evidence proves linearity, and future-aware...",No
4,E. Store-specific median,Robust to extreme transaction days and store s...,"Ignores date, weekday, season, and store evolu...",Store-specific distributions have stable summa...,Median replacement may distort genuine daily v...,TBD
5,F. Date-level median across stores,Reflects common calendar conditions.,Stores have different transaction scales and o...,Multiple stores are represented on most dates.,Cross-store medians are not store-specific obs...,No
6,G. Drop transactions later from modelling,Avoids missingness and future-availability risk.,May discard useful historical demand context.,Actual future transaction counts may be unavai...,Predictive value has not yet been evaluated wi...,TBD - modelling decision
7,H. Preserve now; evaluate causal lagged/availa...,Separates source cleaning from leakage-safe fe...,Requires explicit temporal validation and serv...,Current analysis cannot justify an exact imput...,Leaves a later modelling decision unresolved b...,Yes


**Temporal leakage boundary:** backward fill or unconstrained interpolation can use future information. Cleaning-stage preservation and modelling-stage leakage safety are separate decisions.

# 37. Evidence summary for `transactions`

In [32]:
boundary_counts = boundary_summary.set_index("boundary_classification")["missing_store_date_count"].to_dict()
interior_missing_count = int(boundary_counts.get("interior", 0))
isolated_block_count = int((missing_blocks["block_length_days"] == 1).sum())
missing_sales_row = sales_activity_comparison.loc[
    sales_activity_comparison["transaction_coverage_status"].eq("missing transactions")
].iloc[0]
valid_sales_row = sales_activity_comparison.loc[
    sales_activity_comparison["transaction_coverage_status"].eq("valid transactions")
].iloc[0]
holiday_missing_row = holiday_transaction_missingness.loc[
    holiday_transaction_missingness["date_type"].eq("holiday")
].iloc[0]
nonholiday_missing_row = holiday_transaction_missingness.loc[
    holiday_transaction_missingness["date_type"].eq("non-holiday")
].iloc[0]

transaction_evidence_summary = pd.DataFrame([
    {
        "Check": "Item-row missingness",
        "Evidence": f"{item_row_missing_transactions:,}/{total_merged_rows:,} rows ({100.0 * item_row_missing_transactions / total_merged_rows:.6f}%).",
        "Interpretation": "Repeated item rows amplify store-date gaps.",
        "Cleaning implication": "Reason and validate at store-date grain.",
    },
    {
        "Check": "Unique store-date missingness",
        "Evidence": f"{missing_store_dates_count:,}/{total_store_dates:,} store-dates ({100.0 * missing_store_dates_count / total_store_dates:.6f}%).",
        "Interpretation": "This is the root missingness population.",
        "Cleaning implication": "Do not treat item-level rows as independent failures.",
    },
    {
        "Check": "Temporal concentration",
        "Evidence": f"Highest year has {int(transaction_missing_by_year['missing_store_date_count'].max()):,} missing store-dates; {len(all_represented_stores_missing_dates)} dates have every represented store missing.",
        "Interpretation": "Year/month/date tables show whether gaps cluster in particular periods.",
        "Cleaning implication": "A global fill rule is not justified by low prevalence alone.",
    },
    {
        "Check": "Store concentration",
        "Evidence": f"{stores_affected_by_transaction_missingness}/{len(store_transaction_coverage)} stores affected.",
        "Interpretation": "The store table distinguishes widespread gaps from a few-store problem.",
        "Cleaning implication": "Any later estimator must respect store-specific scale and history.",
    },
    {
        "Check": "Operating-boundary relationship",
        "Evidence": f"{interior_missing_count:,}/{missing_store_dates_count:,} missing store-dates are interior under the ±{STORE_BOUNDARY_WINDOW_DAYS}-day rule.",
        "Interpretation": "Boundary proximity alone does not explain all gaps.",
        "Cleaning implication": "Do not label boundary gaps as store closure without source evidence.",
    },
    {
        "Check": "Continuity/gap structure",
        "Evidence": f"{len(missing_blocks):,} blocks; {isolated_block_count:,} isolated; longest {int(missing_blocks['block_length_days'].max())} days.",
        "Interpretation": "The mix of isolated and multi-day blocks guides later sensitivity tests.",
        "Cleaning implication": "No single interpolation assumption is proven.",
    },
    {
        "Check": "Nearby transaction context",
        "Evidence": f"Deterministic {len(selected_transaction_cases)}-case context shows valid-neighbor medians and ranges without imputing gaps.",
        "Interpretation": "Nearby values establish availability, not the missing true count.",
        "Cleaning implication": "Do not forward/backfill during source cleaning.",
    },
    {
        "Check": "Sales activity on missing dates",
        "Evidence": f"Missing-date median daily sales {missing_sales_row.median_daily_total_unit_sales:.2f} vs valid-date {valid_sales_row.median_daily_total_unit_sales:.2f}.",
        "Interpretation": "Missing transaction dates can still contain recorded sales activity.",
        "Cleaning implication": "Do not drop sales rows solely because transactions is missing.",
    },
    {
        "Check": "Holiday relationship",
        "Evidence": f"Holiday missing {holiday_missing_row.missing_percentage:.3f}% vs non-holiday {nonholiday_missing_row.missing_percentage:.3f}%.",
        "Interpretation": "The comparison indicates whether holidays are disproportionately represented.",
        "Cleaning implication": "Holiday status alone does not supply an exact transaction value.",
    },
    {
        "Check": "Valid transaction distribution",
        "Evidence": f"Median {valid_transactions.median():.1f}; p5 {valid_transactions.quantile(0.05):.1f}; p95 {valid_transactions.quantile(0.95):.1f} across {len(valid_transactions):,} valid store-dates.",
        "Interpretation": "Transaction scale varies and store-specific ranges are visible.",
        "Cleaning implication": "Distribution plausibility is insufficient to choose an imputation method.",
    },
])
display(transaction_evidence_summary)

,Check,Evidence,Interpretation,Cleaning implication
0,Item-row missingness,"214,625/125,497,040 rows (0.171020%).",Repeated item rows amplify store-date gaps.,Reason and validate at store-date grain.
1,Unique store-date missingness,"118/83,606 store-dates (0.141138%).",This is the root missingness population.,Do not treat item-level rows as independent fa...
2,Temporal concentration,Highest year has 112 missing store-dates; 2 da...,Year/month/date tables show whether gaps clust...,A global fill rule is not justified by low pre...
3,Store concentration,53/54 stores affected.,The store table distinguishes widespread gaps ...,Any later estimator must respect store-specifi...
4,Operating-boundary relationship,118/118 missing store-dates are interior under...,Boundary proximity alone does not explain all ...,Do not label boundary gaps as store closure wi...
5,Continuity/gap structure,62 blocks; 21 isolated; longest 3 days.,The mix of isolated and multi-day blocks guide...,No single interpolation assumption is proven.
6,Nearby transaction context,Deterministic 20-case context shows valid-neig...,"Nearby values establish availability, not the ...",Do not forward/backfill during source cleaning.
7,Sales activity on missing dates,Missing-date median daily sales 15821.16 vs va...,Missing transaction dates can still contain re...,Do not drop sales rows solely because transact...
8,Holiday relationship,Holiday missing 0.015% vs non-holiday 0.152%.,The comparison indicates whether holidays are ...,Holiday status alone does not supply an exact ...
9,Valid transaction distribution,"Median 1393.0; p5 654.0; p95 3712.0 across 83,...",Transaction scale varies and store-specific ra...,Distribution plausibility is insufficient to c...


# 38. Provisional cleaning rule — 	transactions

### 1. What the data shows

- Exactly **214,625 of 125,497,040 item rows (0.171020%)** have missing 	transactions. Because 	transactions is a store-date measure repeated across item rows, these reduce to **118 of 83,606 represented store-dates (0.141138%)**.
- Missingness affects **53 of 54 stores**. It is strongly time-concentrated: **112 of the 118** missing store-dates occur in 2016, including broad gaps on 2016-01-02 through 2016-01-04; two dates have missing transactions for every represented store.
- All **118 missing store-dates are interior** under the displayed ±7-day boundary check. The gaps form **62 store-specific blocks**: 21 isolated one-day blocks and 41 multi-day blocks, with a maximum length of three days.
- Missing-transaction store-dates still contain substantial recorded sales: median daily total unit_sales is **15,821.16**, compared with **10,014.87** for store-dates with valid transactions. This is descriptive context, not evidence that missingness raises sales.
- Holiday status does not explain the pattern: the store-date missing rate is **0.014817% on holidays** and **0.152231% on non-holidays**.
- Valid store-date transaction counts vary widely (median **1,393**, 5th percentile **654**, 95th percentile **3,712**) and their scale also differs by store.

### 2. What is inferred

- The item-row count amplifies a much smaller set of root gaps at the store-date grain.
- The combination of a short multi-store temporal disruption and a few isolated exceptions is more consistent with incomplete transaction coverage than with stores having no sales activity.
- Nearby valid observations provide useful context, but they do not identify the exact missing transaction counts or prove that forward fill, backward fill, interpolation, zero fill, or a store median is correct.

### 3. What remains uncertain

- The source does not establish why the 118 store-date values are absent.
- The true counts cannot be recovered exactly from sales totals, holidays, or adjacent transaction observations.
- Whether same-day 	ransactions will be available at forecasting time depends on the eventual prediction contract and must not be assumed during cleaning.

### 4. Recommended provisional cleaning rule

**Preserve missing 	ransactions values unchanged in the cleaned base dataset. Do not impute, zero-fill, interpolate, forward-fill, backward-fill, or drop affected sales rows during SCRUM-9 cleaning.** This is option **F: preserve missing values now and decide modelling treatment later**. The raw column remains auditable, and no unsupported transaction count is invented.

No rule is applied in this analysis notebook.

### 5. Later feature-engineering and temporal-validation implications

- Treat any missingness indicator, lag, rolling statistic, or estimator as later feature engineering, not source cleaning.
- Any estimator must be fitted only on past training data, operate at store-date grain, respect store-specific scale, and be recomputed within each temporal validation fold.
- Backward fill and unconstrained interpolation are prohibited because they can use future information.
- Confirm the inference-time availability contract before allowing contemporaneous 	ransactions into a forecasting feature set.

### 6. Final cleaning-rule table entry

| Field | Decision |
|---|---|
| Column | 	ransactions |
| Observed issue | 214,625 missing item rows representing 118 missing store-dates |
| Grain for reasoning | (date, store_nbr) |
| Provisional action | Preserve missing values unchanged |
| Explicitly prohibited in cleaning | Zero fill, forward/backward fill, interpolation, median fill, row deletion |
| Rationale | Exact values are unknown; gaps contain recorded sales; no imputation method is proven |
| Later dependency | Availability-safe feature design and fold-local temporal validation |
| Rule applied here | No |


# 39. `dcoilwtico` missing-value investigation

`dcoilwtico` is the daily West Texas Intermediate (WTI) crude-oil price. It is a date-level external time series, so the same value is repeated across every represented store-item row on a date. Root missingness must therefore be assessed at unique-date grain rather than treating repeated item-row nulls as independent oil-price failures.

This investigation tests whether missingness follows weekends, holidays, source-data gaps, dataset boundaries, or another temporal pattern. It is analysis-only: no oil value is filled, interpolated, replaced, removed, or otherwise cleaned here.

# 40. Overall `dcoilwtico` coverage

Scan only the required Parquet columns, reconcile item-row and unique-date coverage, and verify that no date contains conflicting non-null oil prices.

In [33]:
OIL_SCAN_COLUMNS = ["date", "dcoilwtico", "is_holiday", "holiday_type"]
missing_oil_columns = [column for column in OIL_SCAN_COLUMNS if column not in parquet_file.schema_arrow.names]
if missing_oil_columns:
    raise ValueError(f"Missing required oil investigation columns: {missing_oil_columns}")

oil_date_parts = []
oil_item_row_missing_count = 0
oil_item_row_non_missing_count = 0

for row_group_index in range(parquet_file.metadata.num_row_groups):
    oil_scan = parquet_file.read_row_group(
        row_group_index,
        columns=OIL_SCAN_COLUMNS,
    ).to_pandas()
    oil_missing = oil_scan["dcoilwtico"].isna()
    oil_item_row_missing_count += int(oil_missing.sum())
    oil_item_row_non_missing_count += int((~oil_missing).sum())

    oil_scan["_oil_missing_item_row"] = oil_missing.astype("int64")
    holiday_type_partial = oil_scan.loc[
        oil_scan["holiday_type"].notna(), ["date", "holiday_type"]
    ].drop_duplicates().groupby("date", observed=True)["holiday_type"].agg(
        lambda values: " | ".join(sorted(set(map(str, values))))
    ).rename("holiday_type").reset_index()
    partial = oil_scan.groupby("date", as_index=False, observed=True).agg(
        item_row_count=("date", "size"),
        missing_oil_item_rows=("_oil_missing_item_row", "sum"),
        oil_value_min=("dcoilwtico", "min"),
        oil_value_max=("dcoilwtico", "max"),
        is_holiday=("is_holiday", "max"),
    ).merge(holiday_type_partial, on="date", how="left", validate="one_to_one")
    oil_date_parts.append(partial)

def combine_holiday_types(values):
    parts = {
        part.strip()
        for value in values.dropna()
        for part in str(value).split(" | ")
        if part.strip()
    }
    return " | ".join(sorted(parts)) if parts else None

oil_date = pd.concat(oil_date_parts, ignore_index=True).groupby(
    "date", as_index=False, observed=True
).agg(
    item_row_count=("item_row_count", "sum"),
    missing_oil_item_rows=("missing_oil_item_rows", "sum"),
    oil_value_min=("oil_value_min", "min"),
    oil_value_max=("oil_value_max", "max"),
    is_holiday=("is_holiday", "max"),
    holiday_type=("holiday_type", combine_holiday_types),
).sort_values("date").reset_index(drop=True)

conflicting_oil_dates = oil_date.loc[
    oil_date["oil_value_min"].notna()
    & oil_date["oil_value_max"].notna()
    & oil_date["oil_value_min"].ne(oil_date["oil_value_max"])
].copy()
oil_date["dcoilwtico"] = oil_date["oil_value_min"]
oil_date["is_missing"] = oil_date["dcoilwtico"].isna()

mixed_oil_dates = oil_date.loc[
    oil_date["missing_oil_item_rows"].between(1, oil_date["item_row_count"] - 1)
].copy()
if not conflicting_oil_dates.empty:
    raise ValueError("At least one date contains conflicting non-null dcoilwtico values.")
if not mixed_oil_dates.empty:
    raise ValueError("At least one date mixes missing and non-missing dcoilwtico item rows.")

total_oil_dates = len(oil_date)
missing_oil_date_count = int(oil_date["is_missing"].sum())
valid_oil_date_count = total_oil_dates - missing_oil_date_count

oil_item_row_coverage = pd.DataFrame([{
    "grain": "item row",
    "total_count": total_merged_rows,
    "non_missing_dcoilwtico_count": oil_item_row_non_missing_count,
    "missing_dcoilwtico_count": oil_item_row_missing_count,
    "missing_percentage": 100.0 * oil_item_row_missing_count / total_merged_rows,
}])
oil_date_coverage = pd.DataFrame([{
    "grain": "unique date",
    "total_count": total_oil_dates,
    "non_missing_dcoilwtico_count": valid_oil_date_count,
    "missing_dcoilwtico_count": missing_oil_date_count,
    "missing_percentage": 100.0 * missing_oil_date_count / total_oil_dates,
}])
oil_consistency_check = pd.DataFrame([{
    "dates_with_conflicting_non_null_values": len(conflicting_oil_dates),
    "dates_mixing_null_and_non_null_values": len(mixed_oil_dates),
    "maximum_non_null_values_per_date": 0 if valid_oil_date_count == 0 else 1,
}])

if oil_item_row_missing_count + oil_item_row_non_missing_count != total_merged_rows:
    raise ValueError("Oil item-row counts do not reconcile to Parquet metadata.")

display(oil_item_row_coverage)
display(oil_date_coverage)
display(oil_consistency_check)

,grain,total_count,non_missing_dcoilwtico_count,missing_dcoilwtico_count,missing_percentage
0,item row,125497040,84974110,40522930,32.289949


,grain,total_count,non_missing_dcoilwtico_count,missing_dcoilwtico_count,missing_percentage
0,unique date,1684,1163,521,30.938242


,dates_with_conflicting_non_null_values,dates_mixing_null_and_non_null_values,maximum_non_null_values_per_date
0,0,0,1


# 41. Unique date-level oil table

Build the compact analytical table at the correct date grain and show bounded previews of the full sequence and missing dates.

In [34]:
oil_date["year"] = oil_date["date"].dt.year
oil_date["year_month"] = oil_date["date"].dt.to_period("M").astype(str)
oil_date["weekday"] = oil_date["date"].dt.weekday
oil_date["weekday_name"] = oil_date["date"].dt.day_name()
oil_date["is_weekend"] = oil_date["weekday"].ge(5)
oil_date["date_has_holiday"] = oil_date["is_holiday"].fillna(False).astype(bool)

oil_date_table = oil_date[[
    "date", "dcoilwtico", "is_missing", "year", "year_month",
    "weekday", "weekday_name", "is_weekend", "date_has_holiday", "holiday_type",
]].copy()
missing_oil_dates = oil_date_table.loc[oil_date_table["is_missing"]].copy()

display(Markdown("### First 30 unique dates"))
display(oil_date_table.head(30))
display(Markdown("### First 30 missing-oil dates"))
display(missing_oil_dates.head(30))
display(Markdown("### Last 30 missing-oil dates"))
display(missing_oil_dates.tail(30))

### First 30 unique dates

,date,dcoilwtico,is_missing,year,year_month,weekday,weekday_name,is_weekend,date_has_holiday,holiday_type
0,2013-01-01,NaN,True,2013,2013-01,1,Tuesday,False,True,Holiday
1,2013-01-02,93.14,False,2013,2013-01,2,Wednesday,False,False,<NA>
2,2013-01-03,92.97,False,2013,2013-01,3,Thursday,False,False,<NA>
3,2013-01-04,93.12,False,2013,2013-01,4,Friday,False,False,<NA>
4,2013-01-05,NaN,True,2013,2013-01,5,Saturday,True,False,Work Day
5,2013-01-06,NaN,True,2013,2013-01,6,Sunday,True,False,<NA>
6,2013-01-07,93.20,False,2013,2013-01,0,Monday,False,False,<NA>
7,2013-01-08,93.21,False,2013,2013-01,1,Tuesday,False,False,<NA>
8,2013-01-09,93.08,False,2013,2013-01,2,Wednesday,False,False,<NA>
9,2013-01-10,93.81,False,2013,2013-01,3,Thursday,False,False,<NA>


### First 30 missing-oil dates

,date,dcoilwtico,is_missing,year,year_month,weekday,weekday_name,is_weekend,date_has_holiday,holiday_type
0,2013-01-01,NaN,True,2013,2013-01,1,Tuesday,False,True,Holiday
4,2013-01-05,NaN,True,2013,2013-01,5,Saturday,True,False,Work Day
5,2013-01-06,NaN,True,2013,2013-01,6,Sunday,True,False,<NA>
11,2013-01-12,NaN,True,2013,2013-01,5,Saturday,True,False,Work Day
12,2013-01-13,NaN,True,2013,2013-01,6,Sunday,True,False,<NA>
18,2013-01-19,NaN,True,2013,2013-01,5,Saturday,True,False,<NA>
19,2013-01-20,NaN,True,2013,2013-01,6,Sunday,True,False,<NA>
20,2013-01-21,NaN,True,2013,2013-01,0,Monday,False,False,<NA>
25,2013-01-26,NaN,True,2013,2013-01,5,Saturday,True,False,<NA>
26,2013-01-27,NaN,True,2013,2013-01,6,Sunday,True,False,<NA>


### Last 30 missing-oil dates

,date,dcoilwtico,is_missing,year,year_month,weekday,weekday_name,is_weekend,date_has_holiday,holiday_type
1590,2017-05-14,NaN,True,2017,2017-05,6,Sunday,True,True,Event
1596,2017-05-20,NaN,True,2017,2017-05,5,Saturday,True,False,<NA>
1597,2017-05-21,NaN,True,2017,2017-05,6,Sunday,True,False,<NA>
1603,2017-05-27,NaN,True,2017,2017-05,5,Saturday,True,False,<NA>
1604,2017-05-28,NaN,True,2017,2017-05,6,Sunday,True,False,<NA>
1605,2017-05-29,NaN,True,2017,2017-05,0,Monday,False,False,<NA>
1610,2017-06-03,NaN,True,2017,2017-06,5,Saturday,True,False,<NA>
1611,2017-06-04,NaN,True,2017,2017-06,6,Sunday,True,False,<NA>
1617,2017-06-10,NaN,True,2017,2017-06,5,Saturday,True,False,<NA>
1618,2017-06-11,NaN,True,2017,2017-06,6,Sunday,True,False,<NA>


# 42. Missingness by weekday

Compare all seven weekdays and quantify how much of the missing-date population occurs on weekends versus weekdays.

In [35]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
oil_missing_by_weekday = oil_date_table.groupby(
    ["weekday", "weekday_name"], as_index=False, observed=True
).agg(
    total_dates=("date", "size"),
    missing_oil_dates=("is_missing", "sum"),
)
oil_missing_by_weekday["valid_oil_dates"] = (
    oil_missing_by_weekday["total_dates"] - oil_missing_by_weekday["missing_oil_dates"]
)
oil_missing_by_weekday["missing_percentage"] = (
    100.0 * oil_missing_by_weekday["missing_oil_dates"] / oil_missing_by_weekday["total_dates"]
)
oil_missing_by_weekday["weekday_name"] = pd.Categorical(
    oil_missing_by_weekday["weekday_name"], categories=weekday_order, ordered=True
)
oil_missing_by_weekday = oil_missing_by_weekday.sort_values("weekday_name").reset_index(drop=True)

weekend_missing_date_count = int(missing_oil_dates["is_weekend"].sum())
weekday_missing_date_count = missing_oil_date_count - weekend_missing_date_count
weekend_share_of_missing = 100.0 * weekend_missing_date_count / missing_oil_date_count
weekday_share_of_missing = 100.0 * weekday_missing_date_count / missing_oil_date_count
weekend_concentration = pd.DataFrame([{
    "all_missing_oil_dates": missing_oil_date_count,
    "missing_weekend_dates": weekend_missing_date_count,
    "percentage_of_missing_dates_on_weekends": weekend_share_of_missing,
    "missing_weekday_dates": weekday_missing_date_count,
    "percentage_of_missing_dates_on_weekdays": weekday_share_of_missing,
    "saturday_and_sunday_account_for_majority": weekend_missing_date_count > weekday_missing_date_count,
}])

display(oil_missing_by_weekday)
display(weekend_concentration)

,weekday,weekday_name,total_dates,missing_oil_dates,valid_oil_dates,missing_percentage
0,0,Monday,241,23,218,9.543568
1,1,Tuesday,242,2,240,0.826446
2,2,Wednesday,240,1,239,0.416667
3,3,Thursday,240,6,234,2.500000
4,4,Friday,240,8,232,3.333333
5,5,Saturday,241,241,0,100.000000
6,6,Sunday,240,240,0,100.000000


,all_missing_oil_dates,missing_weekend_dates,percentage_of_missing_dates_on_weekends,missing_weekday_dates,percentage_of_missing_dates_on_weekdays,saturday_and_sunday_account_for_majority
0,521,481,92.322457,40,7.677543,True


# 43. Missingness over time

Summarize unique-date coverage by year and month to test whether the missing rate is stable or concentrated in particular periods.

In [36]:
def oil_date_missing_summary(frame, keys):
    summary = frame.groupby(keys, as_index=False, observed=True).agg(
        total_dates=("date", "size"),
        missing_oil_dates=("is_missing", "sum"),
    )
    summary["valid_oil_dates"] = summary["total_dates"] - summary["missing_oil_dates"]
    summary["missing_percentage"] = 100.0 * summary["missing_oil_dates"] / summary["total_dates"]
    return summary

oil_missing_by_year = oil_date_missing_summary(oil_date_table, ["year"]).sort_values("year")
oil_missing_by_month = oil_date_missing_summary(oil_date_table, ["year_month"]).sort_values("year_month")
highest_missing_oil_months = oil_missing_by_month.sort_values(
    ["missing_percentage", "missing_oil_dates", "year_month"], ascending=[False, False, True]
).head(20)
zero_missing_oil_months = oil_missing_by_month.loc[
    oil_missing_by_month["missing_oil_dates"].eq(0)
].copy()

display(Markdown("### Yearly summary"))
display(oil_missing_by_year)
display(Markdown("### Full monthly summary"))
display(oil_missing_by_month)
display(Markdown("### Months with the highest missing percentages"))
display(highest_missing_oil_months)
display(Markdown("### Months with zero missing dates"))
display(zero_missing_oil_months)

### Yearly summary

,year,total_dates,missing_oil_dates,valid_oil_dates,missing_percentage
0,2013,364,112,252,30.769231
1,2014,364,112,252,30.769231
2,2015,364,112,252,30.769231
3,2016,365,113,252,30.958904
4,2017,227,72,155,31.718062


### Full monthly summary

,year_month,total_dates,missing_oil_dates,valid_oil_dates,missing_percentage
0,2013-01,31,10,21,32.258065
1,2013-02,28,9,19,32.142857
2,2013-03,31,11,20,35.483871
3,2013-04,30,8,22,26.666667
4,2013-05,31,9,22,29.032258
5,2013-06,30,10,20,33.333333
6,2013-07,31,9,22,29.032258
7,2013-08,31,9,22,29.032258
8,2013-09,30,10,20,33.333333
9,2013-10,31,8,23,25.806452


### Months with the highest missing percentages

,year_month,total_dates,missing_oil_dates,valid_oil_dates,missing_percentage
36,2016-01,31,12,19,38.709677
54,2017-07,31,12,19,38.709677
22,2014-11,30,11,19,36.666667
51,2017-04,30,11,19,36.666667
2,2013-03,31,11,20,35.483871
24,2015-01,31,11,20,35.483871
28,2015-05,31,11,20,35.483871
42,2016-07,31,11,20,35.483871
48,2017-01,31,11,20,35.483871
5,2013-06,30,10,20,33.333333


### Months with zero missing dates

,year_month,total_dates,missing_oil_dates,valid_oil_dates,missing_percentage


# 44. Consecutive missing-date block structure

Identify consecutive calendar-date runs to distinguish routine weekend gaps from longer or unusual source gaps.

In [37]:
sorted_missing_oil_dates = missing_oil_dates["date"].sort_values().drop_duplicates().reset_index(drop=True)
oil_missing_block_id = sorted_missing_oil_dates.diff().ne(pd.Timedelta(days=1)).cumsum()
oil_missing_blocks = sorted_missing_oil_dates.groupby(oil_missing_block_id).agg(
    start_date="min", end_date="max", length_days="size"
).reset_index(drop=True)

def weekdays_in_block(row):
    return ", ".join(pd.date_range(row.start_date, row.end_date).day_name().tolist())

oil_missing_blocks["weekdays_involved"] = [
    weekdays_in_block(row) for row in oil_missing_blocks.itertuples(index=False)
]
oil_missing_blocks = oil_missing_blocks.sort_values(
    ["length_days", "start_date"], ascending=[False, True]
).reset_index(drop=True)

oil_block_summary = pd.DataFrame([{
    "total_missing_date_blocks": len(oil_missing_blocks),
    "single_day_gaps": int(oil_missing_blocks["length_days"].eq(1).sum()),
    "two_day_gaps": int(oil_missing_blocks["length_days"].eq(2).sum()),
    "three_day_gaps": int(oil_missing_blocks["length_days"].eq(3).sum()),
    "gaps_longer_than_three_days": int(oil_missing_blocks["length_days"].gt(3).sum()),
    "longest_consecutive_missing_block_days": int(oil_missing_blocks["length_days"].max()),
    "median_block_length_days": float(oil_missing_blocks["length_days"].median()),
}])

display(oil_block_summary)
display(Markdown("### Thirty longest missing-oil blocks"))
display(oil_missing_blocks.head(30))

,total_missing_date_blocks,single_day_gaps,two_day_gaps,three_day_gaps,gaps_longer_than_three_days,longest_consecutive_missing_block_days,median_block_length_days
0,250,10,210,29,1,4,2.0


### Thirty longest missing-oil blocks

,start_date,end_date,length_days,weekdays_involved
0,2017-07-01,2017-07-04,4,"Saturday, Sunday, Monday, Tuesday"
1,2013-01-19,2013-01-21,3,"Saturday, Sunday, Monday"
2,2013-02-16,2013-02-18,3,"Saturday, Sunday, Monday"
3,2013-03-29,2013-03-31,3,"Friday, Saturday, Sunday"
4,2013-05-25,2013-05-27,3,"Saturday, Sunday, Monday"
5,2013-08-31,2013-09-02,3,"Saturday, Sunday, Monday"
6,2014-01-18,2014-01-20,3,"Saturday, Sunday, Monday"
7,2014-02-15,2014-02-17,3,"Saturday, Sunday, Monday"
8,2014-04-18,2014-04-20,3,"Friday, Saturday, Sunday"
9,2014-05-24,2014-05-26,3,"Saturday, Sunday, Monday"


# 45. Weekend-pattern investigation

Test complete Friday-valid / Saturday-missing / Sunday-missing / Monday-valid sequences, incomplete weekend boundaries, and missing blocks that involve weekdays. No value is imputed.

In [38]:
oil_status_by_date = oil_date_table.set_index("date")["is_missing"]
dataset_dates = oil_status_by_date.index
saturday_dates = dataset_dates[dataset_dates.weekday == 5]
weekend_pattern_rows = []
for saturday in saturday_dates:
    friday = saturday - pd.Timedelta(days=1)
    sunday = saturday + pd.Timedelta(days=1)
    monday = saturday + pd.Timedelta(days=2)
    if sunday not in oil_status_by_date.index:
        continue
    friday_available = friday in oil_status_by_date.index
    monday_available = monday in oil_status_by_date.index
    friday_valid = friday_available and not bool(oil_status_by_date.loc[friday])
    saturday_missing = bool(oil_status_by_date.loc[saturday])
    sunday_missing = bool(oil_status_by_date.loc[sunday])
    monday_valid = monday_available and not bool(oil_status_by_date.loc[monday])
    weekend_pattern_rows.append({
        "friday": friday,
        "saturday": saturday,
        "sunday": sunday,
        "monday": monday,
        "friday_available": friday_available,
        "friday_valid": friday_valid,
        "saturday_missing": saturday_missing,
        "sunday_missing": sunday_missing,
        "monday_available": monday_available,
        "monday_valid": monday_valid,
    })
weekend_patterns = pd.DataFrame(weekend_pattern_rows)
missing_weekend_pairs = weekend_patterns.loc[
    weekend_patterns["saturday_missing"] | weekend_patterns["sunday_missing"]
].copy()
missing_weekend_pairs["pattern_classification"] = np.select(
    [
        missing_weekend_pairs["saturday_missing"]
        & missing_weekend_pairs["sunday_missing"]
        & missing_weekend_pairs["friday_valid"]
        & missing_weekend_pairs["monday_valid"],
        ~missing_weekend_pairs["friday_available"] | ~missing_weekend_pairs["monday_available"],
        missing_weekend_pairs["saturday_missing"] & missing_weekend_pairs["sunday_missing"],
    ],
    [
        "both weekend days missing; valid Friday and Monday",
        "missing weekend with one dataset side unavailable",
        "both weekend days missing; at least one boundary not valid",
    ],
    default="partial weekend missingness",
)

oil_missing_blocks["contains_weekend_date"] = oil_missing_blocks.apply(
    lambda row: bool((pd.date_range(row["start_date"], row["end_date"]).weekday >= 5).any()), axis=1
)
oil_missing_blocks["contains_weekday_date"] = oil_missing_blocks.apply(
    lambda row: bool((pd.date_range(row["start_date"], row["end_date"]).weekday < 5).any()), axis=1
)
weekend_pattern_summary = pd.DataFrame([{
    "weekends_with_any_missing_oil_date": len(missing_weekend_pairs),
    "both_weekend_days_missing_bounded_by_valid_friday_monday": int(
        missing_weekend_pairs["pattern_classification"].eq(
            "both weekend days missing; valid Friday and Monday"
        ).sum()
    ),
    "weekend_patterns_with_one_dataset_side_unavailable": int(
        missing_weekend_pairs["pattern_classification"].eq(
            "missing weekend with one dataset side unavailable"
        ).sum()
    ),
    "missing_blocks_containing_weekends": int(oil_missing_blocks["contains_weekend_date"].sum()),
    "missing_blocks_containing_weekdays": int(oil_missing_blocks["contains_weekday_date"].sum()),
    "pure_weekday_missing_blocks": int(
        (oil_missing_blocks["contains_weekday_date"] & ~oil_missing_blocks["contains_weekend_date"]).sum()
    ),
}])

display(weekend_pattern_summary)
display(missing_weekend_pairs["pattern_classification"].value_counts().rename_axis(
    "pattern_classification"
).reset_index(name="weekend_count"))
display(Markdown("### Sample weekend patterns"))
display(missing_weekend_pairs.head(30))
display(Markdown("### Missing blocks containing at least one weekday"))
display(oil_missing_blocks.loc[oil_missing_blocks["contains_weekday_date"]].head(30))

,weekends_with_any_missing_oil_date,both_weekend_days_missing_bounded_by_valid_friday_monday,weekend_patterns_with_one_dataset_side_unavailable,missing_blocks_containing_weekends,missing_blocks_containing_weekdays,pure_weekday_missing_blocks
0,240,209,1,241,39,9


,pattern_classification,weekend_count
0,both weekend days missing; valid Friday and Mo...,209
1,both weekend days missing; at least one bounda...,30
2,missing weekend with one dataset side unavailable,1


### Sample weekend patterns

,friday,saturday,sunday,monday,friday_available,friday_valid,saturday_missing,sunday_missing,monday_available,monday_valid,pattern_classification
0,2013-01-04,2013-01-05,2013-01-06,2013-01-07,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...
1,2013-01-11,2013-01-12,2013-01-13,2013-01-14,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...
2,2013-01-18,2013-01-19,2013-01-20,2013-01-21,True,True,True,True,True,False,both weekend days missing; at least one bounda...
3,2013-01-25,2013-01-26,2013-01-27,2013-01-28,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...
4,2013-02-01,2013-02-02,2013-02-03,2013-02-04,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...
5,2013-02-08,2013-02-09,2013-02-10,2013-02-11,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...
6,2013-02-15,2013-02-16,2013-02-17,2013-02-18,True,True,True,True,True,False,both weekend days missing; at least one bounda...
7,2013-02-22,2013-02-23,2013-02-24,2013-02-25,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...
8,2013-03-01,2013-03-02,2013-03-03,2013-03-04,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...
9,2013-03-08,2013-03-09,2013-03-10,2013-03-11,True,True,True,True,True,True,both weekend days missing; valid Friday and Mo...


### Missing blocks containing at least one weekday

,start_date,end_date,length_days,weekdays_involved,contains_weekend_date,contains_weekday_date
0,2017-07-01,2017-07-04,4,"Saturday, Sunday, Monday, Tuesday",True,True
1,2013-01-19,2013-01-21,3,"Saturday, Sunday, Monday",True,True
2,2013-02-16,2013-02-18,3,"Saturday, Sunday, Monday",True,True
3,2013-03-29,2013-03-31,3,"Friday, Saturday, Sunday",True,True
4,2013-05-25,2013-05-27,3,"Saturday, Sunday, Monday",True,True
5,2013-08-31,2013-09-02,3,"Saturday, Sunday, Monday",True,True
6,2014-01-18,2014-01-20,3,"Saturday, Sunday, Monday",True,True
7,2014-02-15,2014-02-17,3,"Saturday, Sunday, Monday",True,True
8,2014-04-18,2014-04-20,3,"Friday, Saturday, Sunday",True,True
9,2014-05-24,2014-05-26,3,"Saturday, Sunday, Monday",True,True


# 46. Holiday relationship

Compare missingness on dates with any applicable merged holiday indicator against other dates. This is descriptive evidence and does not assume holidays cause oil-price gaps.

In [39]:
oil_holiday_missingness = oil_date_table.groupby("date_has_holiday", as_index=False, observed=True).agg(
    total_dates=("date", "size"),
    missing_oil_dates=("is_missing", "sum"),
)
oil_holiday_missingness["valid_oil_dates"] = (
    oil_holiday_missingness["total_dates"] - oil_holiday_missingness["missing_oil_dates"]
)
oil_holiday_missingness["missing_percentage"] = (
    100.0 * oil_holiday_missingness["missing_oil_dates"] / oil_holiday_missingness["total_dates"]
)
oil_holiday_missingness["date_type"] = np.where(
    oil_holiday_missingness["date_has_holiday"], "holiday", "non-holiday"
)

missing_oil_by_holiday_type = missing_oil_dates.assign(
    holiday_type_display=missing_oil_dates["holiday_type"].fillna("No applicable holiday record")
).groupby("holiday_type_display", as_index=False, observed=True).agg(
    missing_oil_date_count=("date", "size")
).sort_values("missing_oil_date_count", ascending=False)

display(oil_holiday_missingness[[
    "date_type", "total_dates", "missing_oil_dates", "valid_oil_dates", "missing_percentage"
]])
display(missing_oil_by_holiday_type)

,date_type,total_dates,missing_oil_dates,valid_oil_dates,missing_percentage
0,non-holiday,1451,443,1008,30.530669
1,holiday,233,78,155,33.476395


,holiday_type_display,missing_oil_date_count
8,No applicable holiday record,438
6,Holiday,43
4,Event,19
0,Additional,11
10,Work Day,4
2,Additional | Holiday,1
1,Additional | Event,1
5,Event | Holiday,1
3,Additional | Transfer,1
7,Holiday | Work Day,1


# 47. Boundary-date investigation

Classify missing dates within seven calendar days of the first or last represented dataset date, with all remaining cases labelled interior.

In [40]:
OIL_BOUNDARY_WINDOW_DAYS = 7
first_oil_dataset_date = oil_date_table["date"].min()
last_oil_dataset_date = oil_date_table["date"].max()
missing_oil_with_boundary = missing_oil_dates.copy()
missing_oil_with_boundary["days_from_dataset_start"] = (
    missing_oil_with_boundary["date"] - first_oil_dataset_date
).dt.days
missing_oil_with_boundary["days_to_dataset_end"] = (
    last_oil_dataset_date - missing_oil_with_boundary["date"]
).dt.days
oil_near_start = missing_oil_with_boundary["days_from_dataset_start"].between(
    0, OIL_BOUNDARY_WINDOW_DAYS
)
oil_near_end = missing_oil_with_boundary["days_to_dataset_end"].between(
    0, OIL_BOUNDARY_WINDOW_DAYS
)
missing_oil_with_boundary["boundary_classification"] = np.select(
    [oil_near_start & oil_near_end, oil_near_start, oil_near_end],
    ["near both beginning and end", "near beginning", "near end"],
    default="interior",
)
oil_boundary_summary = missing_oil_with_boundary["boundary_classification"].value_counts().rename_axis(
    "boundary_classification"
).reset_index(name="missing_oil_date_count")
oil_boundary_summary["percentage_of_missing_oil_dates"] = (
    100.0 * oil_boundary_summary["missing_oil_date_count"] / missing_oil_date_count
)
display(pd.DataFrame([{
    "first_dataset_date": first_oil_dataset_date,
    "last_dataset_date": last_oil_dataset_date,
    "boundary_window_days": OIL_BOUNDARY_WINDOW_DAYS,
}]))
display(oil_boundary_summary)

,first_dataset_date,last_dataset_date,boundary_window_days
0,2013-01-01,2017-08-15,7


,boundary_classification,missing_oil_date_count,percentage_of_missing_oil_dates
0,interior,516,99.040307
1,near beginning,3,0.575816
2,near end,2,0.383877


# 48. Nearby oil-price context

Inspect deterministic ±5-calendar-day windows for weekend, weekday, longest-gap, and boundary cases. The tables describe nearby observations only; they do not calculate replacement values.

In [41]:
weekend_oil_cases = missing_oil_dates.loc[missing_oil_dates["is_weekend"], ["date"]].head(4).copy()
weekend_oil_cases["selection_reason"] = "weekend missing date"
weekday_oil_cases = missing_oil_dates.loc[~missing_oil_dates["is_weekend"], ["date"]].head(4).copy()
weekday_oil_cases["selection_reason"] = "weekday missing date"
longest_oil_cases = oil_missing_blocks.head(4)[["start_date"]].rename(
    columns={"start_date": "date"}
)
longest_oil_cases["selection_reason"] = "start of longest missing block"
boundary_oil_cases = missing_oil_with_boundary.loc[
    ~missing_oil_with_boundary["boundary_classification"].eq("interior"), ["date"]
].head(4).copy()
boundary_oil_cases["selection_reason"] = "dataset-boundary missing date"

selected_oil_cases = pd.concat([
    weekend_oil_cases, weekday_oil_cases, longest_oil_cases, boundary_oil_cases
], ignore_index=True).drop_duplicates("date").sort_values("date").reset_index(drop=True)
selected_oil_cases["case_id"] = np.arange(1, len(selected_oil_cases) + 1)

oil_context_parts = []
for case in selected_oil_cases.itertuples(index=False):
    context = oil_date_table.loc[
        oil_date_table["date"].between(
            case.date - pd.Timedelta(days=5), case.date + pd.Timedelta(days=5)
        ),
        ["date", "weekday_name", "dcoilwtico", "is_missing", "date_has_holiday"],
    ].copy()
    context["case_id"] = case.case_id
    context["selected_missing_date"] = case.date
    context["relative_calendar_day"] = (context["date"] - case.date).dt.days
    oil_context_parts.append(context)
nearby_oil_context = pd.concat(oil_context_parts, ignore_index=True).sort_values(["case_id", "date"])

nearby_oil_context_summary_rows = []
for case in selected_oil_cases.itertuples(index=False):
    case_context = nearby_oil_context.loc[nearby_oil_context["case_id"].eq(case.case_id)]
    previous_valid = case_context.loc[
        (case_context["date"] < case.date) & case_context["dcoilwtico"].notna()
    ].tail(1)
    next_valid = case_context.loc[
        (case_context["date"] > case.date) & case_context["dcoilwtico"].notna()
    ].head(1)
    previous_value = None if previous_valid.empty else float(previous_valid["dcoilwtico"].iloc[0])
    next_value = None if next_valid.empty else float(next_valid["dcoilwtico"].iloc[0])
    nearby_oil_context_summary_rows.append({
        "case_id": case.case_id,
        "missing_date": case.date,
        "selection_reason": case.selection_reason,
        "previous_valid_date": None if previous_valid.empty else previous_valid["date"].iloc[0],
        "previous_valid_price": previous_value,
        "next_valid_date": None if next_valid.empty else next_valid["date"].iloc[0],
        "next_valid_price": next_value,
        "absolute_surrounding_price_difference": (
            np.nan if previous_value is None or next_value is None else abs(next_value - previous_value)
        ),
    })
nearby_oil_context_summary = pd.DataFrame(nearby_oil_context_summary_rows)

display(selected_oil_cases)
display(nearby_oil_context_summary)
for case in selected_oil_cases.itertuples(index=False):
    display(Markdown(
        f"### Case {case.case_id}: `{case.date.date().isoformat()}` — {case.selection_reason}"
    ))
    display(nearby_oil_context.loc[
        nearby_oil_context["case_id"].eq(case.case_id),
        ["date", "relative_calendar_day", "weekday_name", "dcoilwtico", "is_missing", "date_has_holiday"],
    ])

,date,selection_reason,case_id
0,2013-01-01,weekday missing date,1
1,2013-01-05,weekend missing date,2
2,2013-01-06,weekend missing date,3
3,2013-01-12,weekend missing date,4
4,2013-01-13,weekend missing date,5
5,2013-01-19,start of longest missing block,6
6,2013-01-21,weekday missing date,7
7,2013-02-16,start of longest missing block,8
8,2013-02-18,weekday missing date,9
9,2013-03-29,weekday missing date,10


,case_id,missing_date,selection_reason,previous_valid_date,previous_valid_price,next_valid_date,next_valid_price,absolute_surrounding_price_difference
0,1,2013-01-01,weekday missing date,NaT,NaN,2013-01-02,93.14,NaN
1,2,2013-01-05,weekend missing date,2013-01-04,93.12,2013-01-07,93.20,0.08
2,3,2013-01-06,weekend missing date,2013-01-04,93.12,2013-01-07,93.20,0.08
3,4,2013-01-12,weekend missing date,2013-01-11,93.60,2013-01-14,94.27,0.67
4,5,2013-01-13,weekend missing date,2013-01-11,93.60,2013-01-14,94.27,0.67
5,6,2013-01-19,start of longest missing block,2013-01-18,95.61,2013-01-22,96.09,0.48
6,7,2013-01-21,weekday missing date,2013-01-18,95.61,2013-01-22,96.09,0.48
7,8,2013-02-16,start of longest missing block,2013-02-15,95.95,2013-02-19,96.69,0.74
8,9,2013-02-18,weekday missing date,2013-02-15,95.95,2013-02-19,96.69,0.74
9,10,2013-03-29,weekday missing date,2013-03-28,97.24,2013-04-01,97.10,0.14


### Case 1: `2013-01-01` — weekday missing date

,date,relative_calendar_day,weekday_name,dcoilwtico,is_missing,date_has_holiday
0,2013-01-01,0,Tuesday,NaN,True,True
1,2013-01-02,1,Wednesday,93.14,False,False
2,2013-01-03,2,Thursday,92.97,False,False
3,2013-01-04,3,Friday,93.12,False,False
4,2013-01-05,4,Saturday,NaN,True,False
5,2013-01-06,5,Sunday,NaN,True,False


### Case 2: `2013-01-05` — weekend missing date

...

,date,relative_calendar_day,weekday_name,dcoilwtico,is_missing,date_has_holiday
93,2013-03-24,-5,Sunday,NaN,True,False
94,2013-03-25,-4,Monday,94.55,False,False
95,2013-03-26,-3,Tuesday,95.99,False,False
96,2013-03-27,-2,Wednesday,96.53,False,False
97,2013-03-28,-1,Thursday,97.24,False,False
98,2013-03-29,0,Friday,NaN,True,False
99,2013-03-30,1,Saturday,NaN,True,False
100,2013-03-31,2,Sunday,NaN,True,False
101,2013-04-01,3,Monday,97.10,False,True
102,2013-04-02,4,Tuesday,97.23,False,False


### Case 11: `2017-07-01` — start of longest missing block

,date,relative_calendar_day,weekday_name,dcoilwtico,is_missing,date_has_holiday
104,2017-06-26,-5,Monday,43.24,False,False
105,2017-06-27,-4,Tuesday,44.25,False,False
106,2017-06-28,-3,Wednesday,44.74,False,False
107,2017-06-29,-2,Thursday,44.88,False,False
108,2017-06-30,-1,Friday,46.02,False,False
109,2017-07-01,0,Saturday,NaN,True,False
110,2017-07-02,1,Sunday,NaN,True,False
111,2017-07-03,2,Monday,NaN,True,True
112,2017-07-04,3,Tuesday,NaN,True,False
113,2017-07-05,4,Wednesday,45.11,False,False


### Case 12: `2017-08-12` — dataset-boundary missing date

,date,relative_calendar_day,weekday_name,dcoilwtico,is_missing,date_has_holiday
115,2017-08-07,-5,Monday,49.37,False,False
116,2017-08-08,-4,Tuesday,49.07,False,False
117,2017-08-09,-3,Wednesday,49.59,False,False
118,2017-08-10,-2,Thursday,48.54,False,False
119,2017-08-11,-1,Friday,48.81,False,True
120,2017-08-12,0,Saturday,NaN,True,False
121,2017-08-13,1,Sunday,NaN,True,False
122,2017-08-14,2,Monday,47.59,False,False
123,2017-08-15,3,Tuesday,47.57,False,True


# 49. Valid oil-price distribution

Summarize observed prices and absolute price changes only when two valid observations occur on consecutive calendar dates.

In [42]:
valid_oil_dates = oil_date_table.loc[~oil_date_table["is_missing"], ["date", "dcoilwtico"]].copy()
valid_oil_prices = valid_oil_dates["dcoilwtico"].astype("float64")
valid_oil_distribution = pd.DataFrame([{
    "count": int(valid_oil_prices.count()),
    "minimum": float(valid_oil_prices.min()),
    "maximum": float(valid_oil_prices.max()),
    "mean": float(valid_oil_prices.mean()),
    "median": float(valid_oil_prices.median()),
    "standard_deviation": float(valid_oil_prices.std()),
    "p1": float(valid_oil_prices.quantile(0.01)),
    "p5": float(valid_oil_prices.quantile(0.05)),
    "p25": float(valid_oil_prices.quantile(0.25)),
    "p75": float(valid_oil_prices.quantile(0.75)),
    "p95": float(valid_oil_prices.quantile(0.95)),
    "p99": float(valid_oil_prices.quantile(0.99)),
}])

valid_oil_dates["calendar_day_gap"] = valid_oil_dates["date"].diff().dt.days
valid_oil_dates["absolute_daily_change"] = valid_oil_dates["dcoilwtico"].diff().abs()
consecutive_valid_oil_changes = valid_oil_dates.loc[
    valid_oil_dates["calendar_day_gap"].eq(1), "absolute_daily_change"
]
oil_daily_change_summary = pd.DataFrame([{
    "consecutive_valid_calendar_day_pairs": len(consecutive_valid_oil_changes),
    "median_absolute_daily_change": float(consecutive_valid_oil_changes.median()),
    "p95_absolute_daily_change": float(consecutive_valid_oil_changes.quantile(0.95)),
    "maximum_absolute_daily_change": float(consecutive_valid_oil_changes.max()),
}])

display(valid_oil_distribution)
display(oil_daily_change_summary)

,count,minimum,maximum,mean,median,standard_deviation,p1,p5,p25,p75,p95,p99
0,1163,26.19,110.62,67.925589,53.33,25.677366,29.8278,36.82,46.39,95.79,105.225,107.9614


,consecutive_valid_calendar_day_pairs,median_absolute_daily_change,p95_absolute_daily_change,maximum_absolute_daily_change
0,912,0.735,2.4535,6.06


# 50. Candidate cleaning approaches — evidence comparison

Compare possible directions without automatically selecting or applying a method. Cleaning-stage provenance and modelling-stage leakage safety remain separate decisions.

In [43]:
candidate_oil_approaches = pd.DataFrame([
    {
        "Candidate approach": "A. Preserve missing oil values unchanged",
        "Potential benefit": "Preserves source provenance and avoids invented market prices.",
        "Main risk": "Downstream algorithms must handle missingness or omit the field.",
        "Evidence supporting it": "Root gaps are identifiable at date grain and their true prices are not observed.",
        "Evidence against it": "Leaves a nullable numeric field for later pipeline design.",
        "Appropriate for cleaning stage?": "Yes",
    },
    {
        "Candidate approach": "B. Forward-fill from previous valid date",
        "Potential benefit": "Can be causal when only earlier information is used.",
        "Main risk": "Carries stale prices across real market movements and long closures.",
        "Evidence supporting it": "Many calendar gaps may follow non-trading days.",
        "Evidence against it": "A previous quote is not the missing date's observed quote.",
        "Appropriate for cleaning stage?": "TBD",
    },
    {
        "Candidate approach": "C. Backward-fill from next valid date",
        "Potential benefit": "Supplies a nearby observed price.",
        "Main risk": "Uses future information and can leak across temporal validation boundaries.",
        "Evidence supporting it": "A later valid quote often exists.",
        "Evidence against it": "Not historically available on the missing date and invents source data.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "D. Linear interpolation between surrounding valid dates",
        "Potential benefit": "Creates a smooth numeric bridge across short gaps.",
        "Main risk": "Uses future values and assumes linear movement through unobserved dates.",
        "Evidence supporting it": "Short calendar gaps have valid observations on both sides in many cases.",
        "Evidence against it": "Observed daily changes demonstrate that oil prices need not move linearly.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "E. Time-aware interpolation only for short gaps",
        "Potential benefit": "Bounds the duration over which prices are estimated.",
        "Main risk": "Still invents values and can use future information unless designed causally.",
        "Evidence supporting it": "Block analysis identifies gap lengths explicitly.",
        "Evidence against it": "No executed evidence proves a safe maximum gap or interpolation rule.",
        "Appropriate for cleaning stage?": "TBD",
    },
    {
        "Candidate approach": "F. Global mean or median",
        "Potential benefit": "Simple complete numeric column.",
        "Main risk": "Destroys temporal structure and replaces market-specific prices with a static value.",
        "Evidence supporting it": "The valid distribution supplies stable summary statistics.",
        "Evidence against it": "Oil is a time series with material level changes and daily variation.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "G. Drop dcoilwtico later from modelling",
        "Potential benefit": "Avoids missingness and uncertain feature availability.",
        "Main risk": "May discard useful external demand context.",
        "Evidence supporting it": "Predictive value has not yet been demonstrated.",
        "Evidence against it": "No temporal model comparison has tested its incremental value.",
        "Appropriate for cleaning stage?": "TBD — modelling decision",
    },
    {
        "Candidate approach": "H. Preserve source values; evaluate causal transformations later",
        "Potential benefit": "Separates auditable cleaning from leakage-safe feature design.",
        "Main risk": "Defers a pipeline decision to temporal modelling experiments.",
        "Evidence supporting it": "Current evidence describes gaps but cannot recover true prices.",
        "Evidence against it": "Requires explicit later validation and serving-time design.",
        "Appropriate for cleaning stage?": "Yes",
    },
])
display(candidate_oil_approaches)
display(Markdown(
    "**Temporal leakage boundary:** backward fill and two-sided interpolation use future prices. "
    "They can invalidate historical model evaluation. Preserving source values during cleaning "
    "does not prevent later causal, fold-local feature experiments."
))

,Candidate approach,Potential benefit,Main risk,Evidence supporting it,Evidence against it,Appropriate for cleaning stage?
0,A. Preserve missing oil values unchanged,Preserves source provenance and avoids invente...,Downstream algorithms must handle missingness ...,Root gaps are identifiable at date grain and t...,Leaves a nullable numeric field for later pipe...,Yes
1,B. Forward-fill from previous valid date,Can be causal when only earlier information is...,Carries stale prices across real market moveme...,Many calendar gaps may follow non-trading days.,A previous quote is not the missing date's obs...,TBD
2,C. Backward-fill from next valid date,Supplies a nearby observed price.,Uses future information and can leak across te...,A later valid quote often exists.,Not historically available on the missing date...,No
3,D. Linear interpolation between surrounding va...,Creates a smooth numeric bridge across short g...,Uses future values and assumes linear movement...,Short calendar gaps have valid observations on...,Observed daily changes demonstrate that oil pr...,No
4,E. Time-aware interpolation only for short gaps,Bounds the duration over which prices are esti...,Still invents values and can use future inform...,Block analysis identifies gap lengths explicitly.,No executed evidence proves a safe maximum gap...,TBD
5,F. Global mean or median,Simple complete numeric column.,Destroys temporal structure and replaces marke...,The valid distribution supplies stable summary...,Oil is a time series with material level chang...,No
6,G. Drop dcoilwtico later from modelling,Avoids missingness and uncertain feature avail...,May discard useful external demand context.,Predictive value has not yet been demonstrated.,No temporal model comparison has tested its in...,TBD — modelling decision
7,H. Preserve source values; evaluate causal tra...,Separates auditable cleaning from leakage-safe...,Defers a pipeline decision to temporal modelli...,Current evidence describes gaps but cannot rec...,Requires explicit later validation and serving...,Yes


**Temporal leakage boundary:** backward fill and two-sided interpolation use future prices. They can invalidate historical model evaluation. Preserving source values during cleaning does not prevent later causal, fold-local feature experiments.

# 51. Evidence summary for `dcoilwtico`

Consolidate the executed evidence and its cleaning implications at the correct date-level root grain.

In [44]:
oil_boundary_counts = oil_boundary_summary.set_index(
    "boundary_classification"
)["missing_oil_date_count"].to_dict()
oil_holiday_rows = oil_holiday_missingness.set_index("date_type")
weekday_missing_share = 100.0 - weekend_share_of_missing
longest_oil_block_days = int(oil_missing_blocks["length_days"].max())
weekday_blocks_count = int(oil_missing_blocks["contains_weekday_date"].sum())

oil_evidence_summary = pd.DataFrame([
    {
        "Check": "Row-level missingness",
        "Evidence": f"{oil_item_row_missing_count:,}/{total_merged_rows:,} rows ({100.0 * oil_item_row_missing_count / total_merged_rows:.6f}%).",
        "Interpretation": "Repeated item rows amplify date-level oil gaps.",
        "Cleaning implication": "Reason and validate at unique-date grain.",
    },
    {
        "Check": "Date-level missingness",
        "Evidence": f"{missing_oil_date_count:,}/{total_oil_dates:,} dates ({100.0 * missing_oil_date_count / total_oil_dates:.6f}%).",
        "Interpretation": "This is the root missingness population.",
        "Cleaning implication": "Do not treat item-row nulls as independent failures.",
    },
    {
        "Check": "Weekday concentration",
        "Evidence": f"{weekday_missing_date_count:,} missing dates ({weekday_missing_share:.3f}%) occur Monday-Friday.",
        "Interpretation": "Weekday gaps test whether weekends are a complete explanation.",
        "Cleaning implication": "Do not assume every missing date is a market weekend.",
    },
    {
        "Check": "Weekend concentration",
        "Evidence": f"{weekend_missing_date_count:,} missing dates ({weekend_share_of_missing:.3f}%) occur Saturday-Sunday.",
        "Interpretation": "Calendar closure is a major pattern only to the degree shown by this share.",
        "Cleaning implication": "Calendar evidence alone does not recover a price.",
    },
    {
        "Check": "Temporal concentration",
        "Evidence": f"Yearly range {oil_missing_by_year['missing_percentage'].min():.3f}% to {oil_missing_by_year['missing_percentage'].max():.3f}%; monthly maxima are displayed.",
        "Interpretation": "The tables show stability or period-specific concentration.",
        "Cleaning implication": "Any later treatment must respect time order.",
    },
    {
        "Check": "Block structure",
        "Evidence": f"{len(oil_missing_blocks):,} blocks; longest {longest_oil_block_days} days; {weekday_blocks_count} blocks contain weekdays.",
        "Interpretation": "The population includes the displayed mix of routine and unusual gaps.",
        "Cleaning implication": "No universal interpolation rule is proven.",
    },
    {
        "Check": "Holiday relationship",
        "Evidence": f"Holiday missing {oil_holiday_rows.loc['holiday', 'missing_percentage']:.3f}% vs non-holiday {oil_holiday_rows.loc['non-holiday', 'missing_percentage']:.3f}%.",
        "Interpretation": "Holiday association is descriptive, not causal.",
        "Cleaning implication": "Holiday status cannot supply the exact missing quote.",
    },
    {
        "Check": "Boundary relationship",
        "Evidence": f"{int(oil_boundary_counts.get('near beginning', 0))} near beginning, {int(oil_boundary_counts.get('near end', 0))} near end, {int(oil_boundary_counts.get('interior', 0))} interior.",
        "Interpretation": "Boundary gaps are separated from the interior population.",
        "Cleaning implication": "Do not apply an edge-specific rule to interior gaps.",
    },
    {
        "Check": "Nearby price continuity",
        "Evidence": f"Deterministic {len(selected_oil_cases)}-case ±5-day windows show surrounding observed prices without filling gaps.",
        "Interpretation": "Nearby values describe continuity but do not reveal true missing prices.",
        "Cleaning implication": "Do not convert context inspection into automatic imputation.",
    },
    {
        "Check": "Valid price variation",
        "Evidence": f"Median consecutive-day absolute change {consecutive_valid_oil_changes.median():.3f}; p95 {consecutive_valid_oil_changes.quantile(0.95):.3f}; maximum {consecutive_valid_oil_changes.max():.3f}.",
        "Interpretation": "Observed prices can move materially between adjacent valid dates.",
        "Cleaning implication": "Global summaries and linear paths can distort temporal behaviour.",
    },
])
display(oil_evidence_summary)

,Check,Evidence,Interpretation,Cleaning implication
0,Row-level missingness,"40,522,930/125,497,040 rows (32.289949%).",Repeated item rows amplify date-level oil gaps.,Reason and validate at unique-date grain.
1,Date-level missingness,"521/1,684 dates (30.938242%).",This is the root missingness population.,Do not treat item-row nulls as independent fai...
2,Weekday concentration,40 missing dates (7.678%) occur Monday-Friday.,Weekday gaps test whether weekends are a compl...,Do not assume every missing date is a market w...
3,Weekend concentration,481 missing dates (92.322%) occur Saturday-Sun...,Calendar closure is a major pattern only to th...,Calendar evidence alone does not recover a price.
4,Temporal concentration,Yearly range 30.769% to 31.718%; monthly maxim...,The tables show stability or period-specific c...,Any later treatment must respect time order.
5,Block structure,250 blocks; longest 4 days; 39 blocks contain ...,The population includes the displayed mix of r...,No universal interpolation rule is proven.
6,Holiday relationship,Holiday missing 33.476% vs non-holiday 30.531%.,"Holiday association is descriptive, not causal.",Holiday status cannot supply the exact missing...
7,Boundary relationship,"3 near beginning, 2 near end, 516 interior.",Boundary gaps are separated from the interior ...,Do not apply an edge-specific rule to interior...
8,Nearby price continuity,Deterministic 12-case ±5-day windows show surr...,Nearby values describe continuity but do not r...,Do not convert context inspection into automat...
9,Valid price variation,Median consecutive-day absolute change 0.735; ...,Observed prices can move materially between ad...,Global summaries and linear paths can distort ...


# 52. Provisional cleaning rule — `dcoilwtico`

### 1. What the data shows

- Exactly **40,522,930 of 125,497,040 item rows (32.289949%)** have missing `dcoilwtico`. Because the oil price is repeated at date grain, these rows reduce to **521 of 1,684 unique dates (30.938242%)**.
- Date consistency is intact: no date contains conflicting non-null oil prices, and no date mixes missing and non-missing oil values across item rows.
- All **481 represented weekend dates are missing**, accounting for **92.322457%** of missing oil dates. The remaining **40 dates (7.677543%)** occur Monday through Friday: Monday 23, Tuesday 2, Wednesday 1, Thursday 6, and Friday 8.
- Missingness is temporally stable at roughly 31% annually rather than isolated to one era. No represented month has zero missing dates.
- The 521 missing dates form **250 blocks**: 10 one-day, 210 two-day, 29 three-day, and one four-day block. The longest block is **2017-07-01 through 2017-07-04**. There are 39 blocks containing at least one weekday, including nine pure-weekday blocks.
- Of 240 weekends with missing oil values, **209** have both weekend days missing and valid Friday/Monday prices. Another 30 have at least one surrounding Friday or Monday that is not valid, and one lies against a dataset boundary.
- Holiday dates have a **33.476395%** missing rate versus **30.530669%** for non-holidays. This modest association does not explain the universal weekend pattern or recover any missing quote.
- Boundary effects are minor: 3 missing dates are near the beginning, 2 near the end, and **516 are interior** under the seven-day rule.
- Across 1,163 valid dates, WTI ranges from **26.19 to 110.62**. For consecutive valid calendar-day pairs, the median absolute change is **0.735**, the 95th percentile is **2.4535**, and the maximum is **6.06**.

### 2. What is inferred

- Missingness is mainly calendar-driven: the source has no Saturday or Sunday quotes, and most missing blocks are weekend pairs bounded by valid weekday observations.
- Weekends are not a complete explanation. The 40 weekday gaps and longer blocks are consistent with market holidays or other source-calendar gaps, but the merged data does not establish a definitive cause for every date.
- Nearby prices often look continuous, but observed day-to-day changes show that the exact path through a gap cannot be recovered from surrounding values alone.

### 3. What remains uncertain

- The true oil price for each missing date is not present in the source.
- This analysis does not establish whether a non-trading-day carry-forward convention is appropriate for the eventual forecasting contract.
- The incremental predictive value of oil, and whether it should remain in modelling at all, require temporal validation.

### 4. Recommended provisional cleaning rule

**Preserve original missing `dcoilwtico` values unchanged during SCRUM-9 cleaning and decide feature treatment later.** Do not forward-fill, backward-fill, interpolate, use a global mean or median, or delete affected sales rows in the cleaned base dataset. This follows direction **E** and retains direction **F** as a later modelling option if oil does not add validated value. No rule is applied in this analysis notebook.

### 5. Feature-engineering implications

- Any later carry-forward, lag, missingness indicator, trading-calendar mapping, or bounded time-aware transformation is feature engineering rather than source cleaning.
- Such transformations must be fitted or calculated inside each temporal training fold and evaluated against preserving missingness or excluding oil.

### 6. Temporal leakage concerns

- Backward fill and two-sided interpolation use future prices and are unsuitable for historical validation unless the forecast contract proves those prices were available at prediction time.
- A causal forward-fill experiment may use only prices available before the forecast origin, but it still invents a non-observed daily value and is therefore deferred from cleaning.

### 7. Final SCRUM-9 cleaning-rules table entry

| Field | Decision |
|---|---|
| Column | `dcoilwtico` |
| Root grain | Unique `date` |
| Observed issue | 40,522,930 missing item rows representing 521 missing dates |
| Provisional cleaning action | Preserve missing values unchanged |
| Explicitly prohibited during cleaning | Forward/backward fill, interpolation, global mean/median, row deletion |
| Evidence basis | 92.322457% weekend concentration, but 40 weekday gaps and material valid-price variation remain |
| Later dependency | Leakage-safe temporal feature comparison and oil-value ablation |
| Rule applied here | No |


# 53. Holiday-column consistency and structural-null investigation

Holiday information is attached at store-date level. National holidays can apply to every represented store, regional holidays depend on store state, and local holidays depend on store city. A normal store-date with no applicable event should naturally have no holiday type, locale, or description; those nulls may mean “not applicable” rather than corrupted data.

This investigation distinguishes expected structural nulls from genuine inconsistencies, while retaining transferred-event and `Work Day` context. No cleaning rule is applied here.

# 54. Store-date holiday table

Scan only holiday and store-scope columns. Before resolving repeated item rows to `(date, store_nbr)`, test each holiday attribute for conflicting non-null values or mixed null/non-null values within a store-date.

In [45]:
HOLIDAY_COLUMNS = [
    "is_holiday", "holiday_type", "holiday_locale", "holiday_description",
    "holiday_transferred", "holiday_event_count",
]
HOLIDAY_DETAIL_COLUMNS = [
    "holiday_type", "holiday_locale", "holiday_description",
    "holiday_transferred", "holiday_event_count",
]
HOLIDAY_SCAN_COLUMNS = ["date", "store_nbr", "city", "state", *HOLIDAY_COLUMNS]
missing_holiday_columns = [column for column in HOLIDAY_SCAN_COLUMNS if column not in parquet_file.schema_arrow.names]
if missing_holiday_columns:
    raise ValueError(f"Missing required holiday investigation columns: {missing_holiday_columns}")

holiday_store_date_parts = []
holiday_item_row_null_counts = {column: 0 for column in HOLIDAY_DETAIL_COLUMNS}

for row_group_index in range(parquet_file.metadata.num_row_groups):
    holiday_scan = parquet_file.read_row_group(
        row_group_index, columns=HOLIDAY_SCAN_COLUMNS
    ).to_pandas()
    for column in HOLIDAY_DETAIL_COLUMNS:
        holiday_item_row_null_counts[column] += int(holiday_scan[column].isna().sum())

    aggregation = {
        "item_row_count": ("date", "size"),
        "city_min": ("city", "min"), "city_max": ("city", "max"),
        "state_min": ("state", "min"), "state_max": ("state", "max"),
    }
    for column in HOLIDAY_COLUMNS:
        aggregation[f"{column}_min"] = (column, "min")
        aggregation[f"{column}_max"] = (column, "max")
        aggregation[f"{column}_non_null_count"] = (column, "count")
    partial = holiday_scan.groupby(
        ["date", "store_nbr"], as_index=False, observed=True
    ).agg(**aggregation)
    for column in HOLIDAY_COLUMNS:
        partial[f"{column}_null_count"] = (
            partial["item_row_count"] - partial[f"{column}_non_null_count"]
        )
    holiday_store_date_parts.append(partial)

combined_aggregation = {
    "item_row_count": ("item_row_count", "sum"),
    "city_min": ("city_min", "min"), "city_max": ("city_max", "max"),
    "state_min": ("state_min", "min"), "state_max": ("state_max", "max"),
}
for column in HOLIDAY_COLUMNS:
    combined_aggregation[f"{column}_min"] = (f"{column}_min", "min")
    combined_aggregation[f"{column}_max"] = (f"{column}_max", "max")
    combined_aggregation[f"{column}_non_null_count"] = (f"{column}_non_null_count", "sum")
    combined_aggregation[f"{column}_null_count"] = (f"{column}_null_count", "sum")

holiday_store_date_audit = pd.concat(holiday_store_date_parts, ignore_index=True).groupby(
    ["date", "store_nbr"], as_index=False, observed=True
).agg(**combined_aggregation).sort_values(["date", "store_nbr"]).reset_index(drop=True)

holiday_conflict_parts = []
for column in HOLIDAY_COLUMNS:
    different_non_null_values = (
        holiday_store_date_audit[f"{column}_min"].notna()
        & holiday_store_date_audit[f"{column}_max"].notna()
        & holiday_store_date_audit[f"{column}_min"].ne(holiday_store_date_audit[f"{column}_max"])
    )
    mixed_null_and_non_null = (
        holiday_store_date_audit[f"{column}_null_count"].gt(0)
        & holiday_store_date_audit[f"{column}_non_null_count"].gt(0)
    )
    conflicts = holiday_store_date_audit.loc[
        different_non_null_values | mixed_null_and_non_null,
        ["date", "store_nbr", f"{column}_min", f"{column}_max",
         f"{column}_null_count", f"{column}_non_null_count"],
    ].copy()
    conflicts.insert(2, "column", column)
    holiday_conflict_parts.append(conflicts)
holiday_attribute_conflicts = pd.concat(holiday_conflict_parts, ignore_index=True)

holiday_store_date = holiday_store_date_audit[[
    "date", "store_nbr", "item_row_count", "city_min", "state_min",
    *[f"{column}_min" for column in HOLIDAY_COLUMNS],
]].rename(columns={
    "city_min": "city", "state_min": "state",
    **{f"{column}_min": column for column in HOLIDAY_COLUMNS},
})
holiday_store_date["is_holiday"] = holiday_store_date["is_holiday"].astype(bool)
holiday_store_date["holiday_transferred"] = holiday_store_date["holiday_transferred"].astype("boolean")
holiday_store_date["holiday_event_count"] = holiday_store_date["holiday_event_count"].astype("Int64")
holiday_store_date["year"] = holiday_store_date["date"].dt.year
holiday_store_date["year_month"] = holiday_store_date["date"].dt.to_period("M").astype(str)
holiday_store_date["weekday"] = holiday_store_date["date"].dt.day_name()

store_scope_conflicts = holiday_store_date_audit.loc[
    holiday_store_date_audit["city_min"].ne(holiday_store_date_audit["city_max"])
    | holiday_store_date_audit["state_min"].ne(holiday_store_date_audit["state_max"])
]
if not store_scope_conflicts.empty:
    raise ValueError("Store city/state values conflict within a store-date.")

holiday_conflict_summary = pd.DataFrame([
    {"column": column, "conflicting_store_dates": int((holiday_attribute_conflicts["column"] == column).sum())}
    for column in HOLIDAY_COLUMNS
])
display(pd.DataFrame([{
    "unique_store_dates": len(holiday_store_date),
    "holiday_attribute_conflicting_store_dates": int(
        holiday_attribute_conflicts[["date", "store_nbr"]].drop_duplicates().shape[0]
    ),
    "store_scope_conflicting_store_dates": len(store_scope_conflicts),
}]))
display(holiday_conflict_summary)
display(holiday_store_date.head(30))
if not holiday_attribute_conflicts.empty:
    display(holiday_attribute_conflicts.head(30))

,unique_store_dates,holiday_attribute_conflicting_store_dates,store_scope_conflicting_store_dates
0,83606,0,0


,column,conflicting_store_dates
0,is_holiday,0
1,holiday_type,0
2,holiday_locale,0
3,holiday_description,0
4,holiday_transferred,0
5,holiday_event_count,0


,date,store_nbr,item_row_count,city,state,is_holiday,holiday_type,holiday_locale,holiday_description,holiday_transferred,holiday_event_count,year,year_month,weekday
0,2013-01-01,25,578,Salinas,Santa Elena,True,Holiday,National,Primer dia del ano,False,1,2013,2013-01,Tuesday
1,2013-01-02,1,1018,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
2,2013-01-02,2,1103,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
3,2013-01-02,3,1201,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
4,2013-01-02,4,1049,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
5,2013-01-02,5,1035,Santo Domingo,Santo Domingo de los Tsachilas,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
6,2013-01-02,6,1125,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
7,2013-01-02,7,1077,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
8,2013-01-02,8,1203,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday
9,2013-01-02,9,1021,Quito,Pichincha,False,<NA>,<NA>,<NA>,False,0,2013,2013-01,Wednesday


# 55. Overall holiday coverage

Separate item-row reconciliation from the root store-date population, then profile each holiday detail field at both grains.

In [46]:
total_holiday_store_dates = len(holiday_store_date)
holiday_store_date_count = int(holiday_store_date["is_holiday"].sum())
nonholiday_store_date_count = total_holiday_store_dates - holiday_store_date_count
holiday_overall_coverage = pd.DataFrame([{
    "total_represented_store_dates": total_holiday_store_dates,
    "is_holiday_true_store_dates": holiday_store_date_count,
    "is_holiday_false_store_dates": nonholiday_store_date_count,
    "holiday_percentage": 100.0 * holiday_store_date_count / total_holiday_store_dates,
}])

holiday_store_date_detail_coverage = pd.DataFrame([
    {
        "column": column,
        "grain": "unique date + store_nbr",
        "total_count": total_holiday_store_dates,
        "non_null_count": int(holiday_store_date[column].notna().sum()),
        "null_count": int(holiday_store_date[column].isna().sum()),
        "null_percentage": 100.0 * holiday_store_date[column].isna().mean(),
    }
    for column in HOLIDAY_DETAIL_COLUMNS
])
holiday_item_row_detail_coverage = pd.DataFrame([
    {
        "column": column,
        "grain": "item row",
        "total_count": total_merged_rows,
        "non_null_count": total_merged_rows - holiday_item_row_null_counts[column],
        "null_count": holiday_item_row_null_counts[column],
        "null_percentage": 100.0 * holiday_item_row_null_counts[column] / total_merged_rows,
    }
    for column in HOLIDAY_DETAIL_COLUMNS
])

display(holiday_overall_coverage)
display(Markdown("### Unique store-date detail coverage"))
display(holiday_store_date_detail_coverage)
display(Markdown("### Item-row detail coverage for Notebook 04 reconciliation"))
display(holiday_item_row_detail_coverage)

,total_represented_store_dates,is_holiday_true_store_dates,is_holiday_false_store_dates,holiday_percentage
0,83606,6749,76857,8.072387


### Unique store-date detail coverage

,column,grain,total_count,non_null_count,null_count,null_percentage
0,holiday_type,unique date + store_nbr,83606,7308,76298,91.259001
1,holiday_locale,unique date + store_nbr,83606,7308,76298,91.259001
2,holiday_description,unique date + store_nbr,83606,7308,76298,91.259001
3,holiday_transferred,unique date + store_nbr,83606,83606,0,0.000000
4,holiday_event_count,unique date + store_nbr,83606,83606,0,0.000000


### Item-row detail coverage for Notebook 04 reconciliation

,column,grain,total_count,non_null_count,null_count,null_percentage
0,holiday_type,item row,125497040,11655544,113841496,90.712495
1,holiday_locale,item row,125497040,11655544,113841496,90.712495
2,holiday_description,item row,125497040,11655544,113841496,90.712495
3,holiday_transferred,item row,125497040,125497040,0,0.000000
4,holiday_event_count,item row,125497040,125497040,0,0.000000


# 56. Structural-null consistency

Cross-check null and non-null holiday detail values separately for `is_holiday=False` and `is_holiday=True`. This distinguishes “no applicable detail” from missing detail on an actual holiday.

In [47]:
structural_null_rows = []
for is_holiday_value, group in holiday_store_date.groupby("is_holiday", observed=True):
    for column in HOLIDAY_DETAIL_COLUMNS:
        null_count = int(group[column].isna().sum())
        structural_null_rows.append({
            "is_holiday": bool(is_holiday_value),
            "column": column,
            "store_date_count": len(group),
            "null_count": null_count,
            "non_null_count": len(group) - null_count,
            "null_percentage": 100.0 * null_count / len(group),
        })
holiday_structural_null_summary = pd.DataFrame(structural_null_rows)

explicit_structural_checks = pd.DataFrame([
    {
        "check": f"is_holiday = {status} AND {column} is {null_status}",
        "store_date_count": int(
            holiday_store_date["is_holiday"].eq(status)
            .loc[holiday_store_date[column].isna() if null_status == "null" else holiday_store_date[column].notna()]
            .sum()
        ),
    }
    for column in ["holiday_type", "holiday_locale", "holiday_description"]
    for status in [False, True]
    for null_status in ["null", "non-null"]
])

display(holiday_structural_null_summary)
display(explicit_structural_checks)

,is_holiday,column,store_date_count,null_count,non_null_count,null_percentage
0,False,holiday_type,76857,76298,559,99.272675
1,False,holiday_locale,76857,76298,559,99.272675
2,False,holiday_description,76857,76298,559,99.272675
3,False,holiday_transferred,76857,0,76857,0.000000
4,False,holiday_event_count,76857,0,76857,0.000000
5,True,holiday_type,6749,0,6749,0.000000
6,True,holiday_locale,6749,0,6749,0.000000
7,True,holiday_description,6749,0,6749,0.000000
8,True,holiday_transferred,6749,0,6749,0.000000
9,True,holiday_event_count,6749,0,6749,0.000000


,check,store_date_count
0,is_holiday = False AND holiday_type is null,76298
1,is_holiday = False AND holiday_type is non-null,559
2,is_holiday = True AND holiday_type is null,0
3,is_holiday = True AND holiday_type is non-null,6749
4,is_holiday = False AND holiday_locale is null,76298
5,is_holiday = False AND holiday_locale is non-null,559
6,is_holiday = True AND holiday_locale is null,0
7,is_holiday = True AND holiday_locale is non-null,6749
8,is_holiday = False AND holiday_description is ...,76298
9,is_holiday = False AND holiday_description is ...,559


# 57. `holiday_event_count` consistency

Test event-count semantics against `is_holiday`, including zero-event non-holidays, transferred or `Work Day` context, and multi-event store-dates. Records are inspected but not changed.

In [48]:
event_count = holiday_store_date["holiday_event_count"]
event_count_bucket = pd.Series(np.select(
    [event_count.isna(), event_count.lt(0), event_count.eq(0), event_count.eq(1), event_count.gt(1)],
    ["missing", "negative", "0", "1", ">1"], default="other"
), index=holiday_store_date.index)
holiday_store_date["event_count_bucket"] = event_count_bucket

holiday_event_count_combinations = holiday_store_date.groupby(
    ["is_holiday", "event_count_bucket"], as_index=False, observed=True
).agg(store_date_count=("date", "size"))
holiday_event_count_distribution = holiday_store_date["holiday_event_count"].value_counts(
    dropna=False
).sort_index().rename_axis("holiday_event_count").reset_index(name="store_date_count")
multi_event_store_date_count = int(event_count.gt(1).sum())
event_count_consistency_summary = pd.DataFrame([{
    "minimum_event_count": event_count.min(),
    "maximum_event_count": event_count.max(),
    "missing_event_count_store_dates": int(event_count.isna().sum()),
    "negative_event_count_store_dates": int(event_count.lt(0).sum()),
    "multi_event_store_dates": multi_event_store_date_count,
}])

display(holiday_event_count_combinations)
display(event_count_consistency_summary)
display(holiday_event_count_distribution)
display(Markdown("### Examples where holiday_event_count > 1"))
display(holiday_store_date.loc[event_count.gt(1), [
    "date", "store_nbr", "city", "state", "is_holiday", "holiday_event_count",
    "holiday_type", "holiday_locale", "holiday_description", "holiday_transferred",
]].head(30))

,is_holiday,event_count_bucket,store_date_count
0,False,0,76298
1,False,1,559
2,True,1,6520
3,True,>1,229


,minimum_event_count,maximum_event_count,missing_event_count_store_dates,negative_event_count_store_dates,multi_event_store_dates
0,0,2,0,0,229


,holiday_event_count,store_date_count
0,0,76298
1,1,7079
2,2,229


### Examples where holiday_event_count > 1

,date,store_nbr,city,state,is_holiday,holiday_event_count,holiday_type,holiday_locale,holiday_description,holiday_transferred
16510,2013-12-22,25,Salinas,Santa Elena,True,2,Additional | Holiday,Local | National,Cantonizacion de Salinas | Navidad-3,False
25058,2014-06-25,12,Latacunga,Cotopaxi,True,2,Event | Holiday,Local | National,Cantonizacion de Latacunga | Mundial de futbol...,False
25059,2014-06-25,13,Latacunga,Cotopaxi,True,2,Event | Holiday,Local | National,Cantonizacion de Latacunga | Mundial de futbol...,False
25061,2014-06-25,15,Ibarra,Imbabura,True,2,Event | Holiday,National | Regional,Mundial de futbol Brasil: Ecuador-Francia | Pr...,False
25081,2014-06-25,40,Machala,El Oro,True,2,Event | Holiday,Local | National,Fundacion de Machala | Mundial de futbol Brasi...,False
25082,2014-06-25,41,Machala,El Oro,True,2,Event | Holiday,Local | National,Fundacion de Machala | Mundial de futbol Brasi...,False
33641,2014-12-22,25,Salinas,Santa Elena,True,2,Additional | Holiday,Local | National,Cantonizacion de Salinas | Navidad-3,False
33764,2014-12-26,1,Quito,Pichincha,True,2,Additional | Bridge,National,Navidad+1 | Puente Navidad,False
33765,2014-12-26,2,Quito,Pichincha,True,2,Additional | Bridge,National,Navidad+1 | Puente Navidad,False
33766,2014-12-26,3,Quito,Pichincha,True,2,Additional | Bridge,National,Navidad+1 | Puente Navidad,False


# 58. `holiday_transferred` consistency

Profile transferred status and its relationship with holiday applicability, type, and event count. A transferred source record can intentionally retain context while not being an observed holiday on that date.

In [49]:
transferred_status = holiday_store_date["holiday_transferred"].map(
    {True: "True", False: "False"}
).fillna("null")
holiday_transferred_counts = transferred_status.value_counts().rename_axis(
    "holiday_transferred"
).reset_index(name="store_date_count")
holiday_transferred_by_status = holiday_store_date.assign(
    transferred_status=transferred_status
).groupby(["is_holiday", "transferred_status"], as_index=False, observed=True).agg(
    store_date_count=("date", "size")
)
holiday_transferred_context = holiday_store_date.assign(
    transferred_status=transferred_status
).groupby(
    ["is_holiday", "holiday_type", "holiday_event_count", "transferred_status"],
    as_index=False, observed=True, dropna=False,
).agg(store_date_count=("date", "size")).sort_values("store_date_count", ascending=False)
transferred_holiday_examples = holiday_store_date.loc[
    holiday_store_date["holiday_transferred"].fillna(False),
    ["date", "store_nbr", "city", "state", "is_holiday", "holiday_type",
     "holiday_event_count", "holiday_transferred", "holiday_description"],
].head(30)

display(holiday_transferred_counts)
display(holiday_transferred_by_status)
display(holiday_transferred_context.head(30))
display(Markdown("### Examples with transferred holiday/event context"))
display(transferred_holiday_examples)

,holiday_transferred,store_date_count
0,False,83285
1,True,321


,is_holiday,transferred_status,store_date_count
0,False,False,76536
1,False,True,321
2,True,False,6749


,is_holiday,holiday_type,holiday_event_count,transferred_status,store_date_count
2,False,<NA>,0,False,76298
9,True,Event,1,False,2637
12,True,Holiday,1,False,1969
3,True,Additional,1,False,1449
15,True,Transfer,1,False,365
0,False,Holiday,1,True,321
1,False,Work Day,1,False,238
8,True,Bridge,1,False,100
11,True,Event | Holiday,2,False,60
10,True,Event,2,False,53


### Examples with transferred holiday/event context

,date,store_nbr,city,state,is_holiday,holiday_type,holiday_event_count,holiday_transferred,holiday_description
13011,2013-10-09,1,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13012,2013-10-09,2,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13013,2013-10-09,3,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13014,2013-10-09,4,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13015,2013-10-09,5,Santo Domingo,Santo Domingo de los Tsachilas,False,Holiday,1,True,Independencia de Guayaquil
13016,2013-10-09,6,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13017,2013-10-09,7,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13018,2013-10-09,8,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13019,2013-10-09,9,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil
13020,2013-10-09,10,Quito,Pichincha,False,Holiday,1,True,Independencia de Guayaquil


# 59. Holiday type distribution

Treat `holiday_event_count > 0` as applicable event context so transferred records and `Work Day` rows remain visible. Report exact observed combined type values without merging or renaming them.

In [50]:
event_context_store_dates = holiday_store_date.loc[
    holiday_store_date["holiday_event_count"].gt(0)
].copy()
holiday_type_distribution = event_context_store_dates.groupby(
    "holiday_type", as_index=False, observed=True, dropna=False
).agg(
    store_date_count=("date", "size"),
    minimum_event_count=("holiday_event_count", "min"),
    maximum_event_count=("holiday_event_count", "max"),
    transferred_true_count=("holiday_transferred", lambda values: int(values.eq(True).sum())),
    transferred_false_count=("holiday_transferred", lambda values: int(values.eq(False).sum())),
    transferred_null_count=("holiday_transferred", lambda values: int(values.isna().sum())),
).sort_values("store_date_count", ascending=False)
holiday_type_distribution["percentage_of_event_context_store_dates"] = (
    100.0 * holiday_type_distribution["store_date_count"] / len(event_context_store_dates)
)
display(holiday_type_distribution)

,holiday_type,store_date_count,minimum_event_count,maximum_event_count,transferred_true_count,transferred_false_count,transferred_null_count,percentage_of_event_context_store_dates
6,Event,2690,1,2,0,2690,0,36.808976
8,Holiday,2291,1,2,321,1970,0,31.349206
0,Additional,1449,1,1,0,1449,0,19.827586
10,Transfer,365,1,1,0,365,0,4.994527
11,Work Day,238,1,1,0,238,0,3.256705
5,Bridge,100,1,1,0,100,0,1.368363
7,Event | Holiday,60,2,2,0,60,0,0.821018
2,Additional | Event,53,2,2,0,53,0,0.725233
1,Additional | Bridge,48,2,2,0,48,0,0.656814
4,Additional | Transfer,8,2,2,0,8,0,0.109469


# 60. Holiday locale distribution

Report exact observed locale combinations, then test internal scope expansion: National tokens should cover all represented stores on a date, Regional tokens all represented stores in an affected state, and Local tokens all represented stores in an affected city.

In [51]:
holiday_locale_distribution = event_context_store_dates.groupby(
    "holiday_locale", as_index=False, observed=True, dropna=False
).agg(store_date_count=("date", "size")).sort_values("store_date_count", ascending=False)
holiday_locale_distribution["percentage_of_event_context_store_dates"] = (
    100.0 * holiday_locale_distribution["store_date_count"] / len(event_context_store_dates)
)

locale_tokens = holiday_store_date["holiday_locale"].fillna("").str.split(" | ", regex=False)
for locale_name in ["National", "Regional", "Local"]:
    holiday_store_date[f"has_{locale_name.lower()}_locale"] = locale_tokens.apply(
        lambda values: locale_name in values
    )

represented_stores_by_date = holiday_store_date.groupby("date", as_index=False).agg(
    represented_store_count=("store_nbr", "size")
)
national_scope = holiday_store_date.loc[
    holiday_store_date["has_national_locale"]
].groupby("date", as_index=False).agg(national_store_count=("store_nbr", "size")).merge(
    represented_stores_by_date, on="date", how="left", validate="one_to_one"
)
national_scope["scope_consistent"] = national_scope["national_store_count"].eq(
    national_scope["represented_store_count"]
)

represented_stores_by_date_state = holiday_store_date.groupby(
    ["date", "state"], as_index=False, observed=True
).agg(represented_store_count=("store_nbr", "size"))
regional_scope = holiday_store_date.loc[
    holiday_store_date["has_regional_locale"]
].groupby(["date", "state"], as_index=False, observed=True).agg(
    regional_store_count=("store_nbr", "size")
).merge(
    represented_stores_by_date_state, on=["date", "state"], how="left", validate="one_to_one"
)
regional_scope["scope_consistent"] = regional_scope["regional_store_count"].eq(
    regional_scope["represented_store_count"]
)

represented_stores_by_date_city = holiday_store_date.groupby(
    ["date", "city"], as_index=False, observed=True
).agg(represented_store_count=("store_nbr", "size"))
local_scope = holiday_store_date.loc[
    holiday_store_date["has_local_locale"]
].groupby(["date", "city"], as_index=False, observed=True).agg(
    local_store_count=("store_nbr", "size")
).merge(
    represented_stores_by_date_city, on=["date", "city"], how="left", validate="one_to_one"
)
local_scope["scope_consistent"] = local_scope["local_store_count"].eq(
    local_scope["represented_store_count"]
)

locale_scope_summary = pd.DataFrame([
    {"locale_scope": "National", "tested_scope_groups": len(national_scope), "inconsistent_scope_groups": int((~national_scope["scope_consistent"]).sum())},
    {"locale_scope": "Regional", "tested_scope_groups": len(regional_scope), "inconsistent_scope_groups": int((~regional_scope["scope_consistent"]).sum())},
    {"locale_scope": "Local", "tested_scope_groups": len(local_scope), "inconsistent_scope_groups": int((~local_scope["scope_consistent"]).sum())},
])
locale_scope_anomalies = pd.concat([
    national_scope.loc[~national_scope["scope_consistent"]].assign(locale_scope="National"),
    regional_scope.loc[~regional_scope["scope_consistent"]].assign(locale_scope="Regional"),
    local_scope.loc[~local_scope["scope_consistent"]].assign(locale_scope="Local"),
], ignore_index=True)

display(holiday_locale_distribution)
display(locale_scope_summary)
if not locale_scope_anomalies.empty:
    display(locale_scope_anomalies.head(30))

,holiday_locale,store_date_count,percentage_of_event_context_store_dates
2,National,6937,94.923372
0,Local,330,4.515599
4,Regional,27,0.369458
1,Local | National,13,0.177887
3,National | Regional,1,0.013684


,locale_scope,tested_scope_groups,inconsistent_scope_groups
0,National,143,0
1,Regional,18,0
2,Local,109,0


# 61. Holiday description consistency

Inspect exact description strings without cleaning accents, punctuation, case, or names. Test description associations with type and locale, and check detail completeness on actual holidays.

In [52]:
holiday_description_count = int(event_context_store_dates["holiday_description"].nunique(dropna=True))
frequent_holiday_descriptions = event_context_store_dates["holiday_description"].value_counts(
    dropna=False
).rename_axis("holiday_description").reset_index(name="store_date_count").head(30)
description_associations = event_context_store_dates.groupby(
    "holiday_description", as_index=False, observed=True, dropna=False
).agg(
    observed_holiday_type_count=("holiday_type", "nunique"),
    observed_holiday_locale_count=("holiday_locale", "nunique"),
    store_date_count=("date", "size"),
)
descriptions_with_multiple_types = description_associations.loc[
    description_associations["observed_holiday_type_count"].gt(1)
].sort_values("store_date_count", ascending=False)
descriptions_with_multiple_locales = description_associations.loc[
    description_associations["observed_holiday_locale_count"].gt(1)
].sort_values("store_date_count", ascending=False)
actual_holiday_null_descriptions = holiday_store_date.loc[
    holiday_store_date["is_holiday"] & holiday_store_date["holiday_description"].isna()
]

display(pd.DataFrame([{
    "unique_non_null_holiday_descriptions": holiday_description_count,
    "descriptions_associated_with_multiple_types": len(descriptions_with_multiple_types),
    "descriptions_associated_with_multiple_locales": len(descriptions_with_multiple_locales),
    "actual_holiday_store_dates_with_null_description": len(actual_holiday_null_descriptions),
}]))
display(frequent_holiday_descriptions)
display(Markdown("### Descriptions associated with multiple exact holiday types"))
display(descriptions_with_multiple_types.head(30))
display(Markdown("### Descriptions associated with multiple exact locale values"))
display(descriptions_with_multiple_locales.head(30))

,unique_non_null_holiday_descriptions,descriptions_associated_with_multiple_types,descriptions_associated_with_multiple_locales,actual_holiday_store_dates_with_null_description
0,108,3,0,0


,holiday_description,store_date_count
0,Carnaval,496
1,Primer Grito de Independencia,252
2,Batalla de Pichincha,249
3,Viernes Santo,246
4,Navidad-4,201
5,Navidad-2,201
6,Navidad-1,201
7,Primer dia del ano-1,201
8,Dia de Difuntos,200
9,Independencia de Cuenca,200


### Descriptions associated with multiple exact holiday types

,holiday_description,observed_holiday_type_count,observed_holiday_locale_count,store_date_count
27,Fundacion de Guayaquil,2,1,37
28,Fundacion de Guayaquil-1,2,1,29
25,Fundacion de Cuenca,2,1,15


### Descriptions associated with multiple exact locale values

,holiday_description,observed_holiday_type_count,observed_holiday_locale_count,store_date_count


# 62. Multiple-event store-date investigation

Inspect the complete multi-event store-date DataFrame without splitting or expanding rows. The merged grain remains one row per `(date, store_nbr, item_nbr)`.

In [53]:
multi_event_store_dates = holiday_store_date.loc[
    holiday_store_date["holiday_event_count"].gt(1),
    ["date", "store_nbr", "city", "state", "holiday_event_count", "holiday_type",
     "holiday_locale", "holiday_description", "holiday_transferred", "is_holiday"],
].sort_values(["holiday_event_count", "date", "store_nbr"], ascending=[False, True, True])
multi_event_summary = pd.DataFrame([{
    "multi_event_store_dates": len(multi_event_store_dates),
    "maximum_events_on_one_store_date": multi_event_store_dates["holiday_event_count"].max(),
    "multi_event_rows_with_combined_types": int(multi_event_store_dates["holiday_type"].str.contains(" | ", regex=False, na=False).sum()),
    "multi_event_rows_with_combined_locales": int(multi_event_store_dates["holiday_locale"].str.contains(" | ", regex=False, na=False).sum()),
    "multi_event_rows_with_combined_descriptions": int(multi_event_store_dates["holiday_description"].str.contains(" | ", regex=False, na=False).sum()),
    "attribute_conflicts_across_item_rows": int(
        holiday_attribute_conflicts[["date", "store_nbr"]].drop_duplicates().shape[0]
    ),
}])
display(multi_event_summary)
display(multi_event_store_dates)

,multi_event_store_dates,maximum_events_on_one_store_date,multi_event_rows_with_combined_types,multi_event_rows_with_combined_locales,multi_event_rows_with_combined_descriptions,attribute_conflicts_across_item_rows
0,229,2,175,14,229,0


,date,store_nbr,city,state,holiday_event_count,holiday_type,holiday_locale,holiday_description,holiday_transferred,is_holiday
16510,2013-12-22,25,Salinas,Santa Elena,2,Additional | Holiday,Local | National,Cantonizacion de Salinas | Navidad-3,False,True
25058,2014-06-25,12,Latacunga,Cotopaxi,2,Event | Holiday,Local | National,Cantonizacion de Latacunga | Mundial de futbol...,False,True
25059,2014-06-25,13,Latacunga,Cotopaxi,2,Event | Holiday,Local | National,Cantonizacion de Latacunga | Mundial de futbol...,False,True
25061,2014-06-25,15,Ibarra,Imbabura,2,Event | Holiday,National | Regional,Mundial de futbol Brasil: Ecuador-Francia | Pr...,False,True
25081,2014-06-25,40,Machala,El Oro,2,Event | Holiday,Local | National,Fundacion de Machala | Mundial de futbol Brasi...,False,True
...,...,...,...,...,...,...,...,...,...,...
63259,2016-07-24,51,Guayaquil,Guayas,2,Additional | Transfer,Local,Fundacion de Guayaquil-1 | Traslado Fundacion ...,False,True
68958,2016-11-12,23,Ambato,Tungurahua,2,Holiday | Work Day,Local | National,Independencia de Ambato | Recupero Puente Dia ...,False,True
68985,2016-11-12,50,Ambato,Tungurahua,2,Holiday | Work Day,Local | National,Independencia de Ambato | Recupero Puente Dia ...,False,True
71056,2016-12-22,25,Salinas,Santa Elena,2,Additional | Holiday,Local | National,Cantonizacion de Salinas | Navidad-3,False,True


# 63. Suspicious holiday consistency cases

Apply every requested anomaly rule, then distinguish unexplained inconsistencies from semantically valid transferred-event or `Work Day` context.

In [54]:
anomaly_conditions = {
    "is_holiday True but event_count <= 0 or missing": (
        holiday_store_date["is_holiday"]
        & (holiday_store_date["holiday_event_count"].isna() | holiday_store_date["holiday_event_count"].le(0))
    ),
    "is_holiday True but holiday_type null": holiday_store_date["is_holiday"] & holiday_store_date["holiday_type"].isna(),
    "is_holiday True but holiday_locale null": holiday_store_date["is_holiday"] & holiday_store_date["holiday_locale"].isna(),
    "is_holiday True but holiday_description null": holiday_store_date["is_holiday"] & holiday_store_date["holiday_description"].isna(),
    "is_holiday False but event_count > 0": ~holiday_store_date["is_holiday"] & holiday_store_date["holiday_event_count"].gt(0),
    "is_holiday False but holiday_type non-null": ~holiday_store_date["is_holiday"] & holiday_store_date["holiday_type"].notna(),
    "negative holiday_event_count": holiday_store_date["holiday_event_count"].lt(0),
}
explained_nonholiday_context = (
    ~holiday_store_date["is_holiday"]
    & holiday_store_date["holiday_event_count"].gt(0)
    & (
        holiday_store_date["holiday_transferred"].fillna(False)
        | holiday_store_date["holiday_type"].fillna("").str.contains("Work Day", regex=False)
    )
)
anomaly_summary_rows = []
anomaly_examples = []
unresolved_mask = pd.Series(False, index=holiday_store_date.index)
for anomaly_type, mask in anomaly_conditions.items():
    semantically_explained = anomaly_type in {
        "is_holiday False but event_count > 0",
        "is_holiday False but holiday_type non-null",
    }
    unexplained_for_type = mask & (~explained_nonholiday_context if semantically_explained else True)
    unresolved_mask |= unexplained_for_type
    anomaly_summary_rows.append({
        "anomaly_type": anomaly_type,
        "triggered_store_date_count": int(mask.sum()),
        "semantically_explained_store_date_count": int((mask & explained_nonholiday_context).sum()) if semantically_explained else 0,
        "unresolved_suspicious_store_date_count": int(unexplained_for_type.sum()),
        "assessment": (
            "Transferred source records and/or Work Day context; not automatically an error."
            if semantically_explained else "No valid exception is assumed."
        ),
    })
    examples = holiday_store_date.loc[mask, [
        "date", "store_nbr", "city", "state", "is_holiday", "holiday_event_count",
        "holiday_type", "holiday_locale", "holiday_description", "holiday_transferred",
    ]].head(5).copy()
    examples.insert(0, "anomaly_type", anomaly_type)
    anomaly_examples.append(examples)

conflict_store_dates = holiday_attribute_conflicts[["date", "store_nbr"]].drop_duplicates()
anomaly_summary_rows.append({
    "anomaly_type": "conflicting holiday attributes across item rows",
    "triggered_store_date_count": len(conflict_store_dates),
    "semantically_explained_store_date_count": 0,
    "unresolved_suspicious_store_date_count": len(conflict_store_dates),
    "assessment": "Repeated item rows should carry identical store-date holiday values.",
})
if not conflict_store_dates.empty:
    unresolved_mask |= holiday_store_date.set_index(["date", "store_nbr"]).index.isin(
        conflict_store_dates.set_index(["date", "store_nbr"]).index
    )

holiday_anomaly_summary = pd.DataFrame(anomaly_summary_rows)
holiday_anomaly_examples = pd.concat(anomaly_examples, ignore_index=True)
unresolved_suspicious_store_date_count = int(unresolved_mask.sum())
display(holiday_anomaly_summary)
display(pd.DataFrame([{
    "unique_unresolved_suspicious_store_dates": unresolved_suspicious_store_date_count
}]))
display(holiday_anomaly_examples)

,anomaly_type,triggered_store_date_count,semantically_explained_store_date_count,unresolved_suspicious_store_date_count,assessment
0,is_holiday True but event_count <= 0 or missing,0,0,0,No valid exception is assumed.
1,is_holiday True but holiday_type null,0,0,0,No valid exception is assumed.
2,is_holiday True but holiday_locale null,0,0,0,No valid exception is assumed.
3,is_holiday True but holiday_description null,0,0,0,No valid exception is assumed.
4,is_holiday False but event_count > 0,559,559,0,Transferred source records and/or Work Day con...
5,is_holiday False but holiday_type non-null,559,559,0,Transferred source records and/or Work Day con...
6,negative holiday_event_count,0,0,0,No valid exception is assumed.
7,conflicting holiday attributes across item rows,0,0,0,Repeated item rows should carry identical stor...


,unique_unresolved_suspicious_store_dates
0,0


,anomaly_type,date,store_nbr,city,state,is_holiday,holiday_event_count,holiday_type,holiday_locale,holiday_description,holiday_transferred
0,is_holiday False but event_count > 0,2013-01-05,1,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
1,is_holiday False but event_count > 0,2013-01-05,2,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
2,is_holiday False but event_count > 0,2013-01-05,3,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
3,is_holiday False but event_count > 0,2013-01-05,4,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
4,is_holiday False but event_count > 0,2013-01-05,5,Santo Domingo,Santo Domingo de los Tsachilas,False,1,Work Day,National,Recupero puente Navidad,False
5,is_holiday False but holiday_type non-null,2013-01-05,1,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
6,is_holiday False but holiday_type non-null,2013-01-05,2,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
7,is_holiday False but holiday_type non-null,2013-01-05,3,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
8,is_holiday False but holiday_type non-null,2013-01-05,4,Quito,Pichincha,False,1,Work Day,National,Recupero puente Navidad,False
9,is_holiday False but holiday_type non-null,2013-01-05,5,Santo Domingo,Santo Domingo de los Tsachilas,False,1,Work Day,National,Recupero puente Navidad,False


# 64. Structural null versus genuine missing classification

Classify each detail field from executed store-date evidence, retaining deliberate `False` and `0` representations separately from true nulls.

In [55]:
structural_classification_rows = []
for column in HOLIDAY_DETAIL_COLUMNS:
    null_count = int(holiday_store_date[column].isna().sum())
    null_on_nonholiday = int((~holiday_store_date["is_holiday"] & holiday_store_date[column].isna()).sum())
    null_on_holiday = int((holiday_store_date["is_holiday"] & holiday_store_date[column].isna()).sum())
    conflict_count = int((holiday_attribute_conflicts["column"] == column).sum())
    if column in {"holiday_type", "holiday_locale", "holiday_description"}:
        context = f"{null_on_nonholiday:,} nulls on non-holidays; {null_on_holiday:,} on actual holidays"
        structural = "Yes, when no applicable event detail exists" if null_on_holiday == 0 else "Partly"
        implication = "Preserve null as not-applicable; investigate only actual-holiday nulls"
    elif column == "holiday_transferred":
        context = "No nulls; False is the deliberate no-record/default representation"
        structural = "Not applicable — represented as False, not null"
        implication = "Preserve the Boolean source semantics"
    else:
        context = "No nulls; 0 is the deliberate no-event/default representation"
        structural = "Not applicable — represented as 0, not null"
        implication = "Preserve the integer source semantics"
    structural_classification_rows.append({
        "Column": column,
        "Null count": null_count,
        "Main context of nulls": context,
        "Structural/expected?": structural,
        "Genuine missing-data evidence?": "Yes" if null_on_holiday > 0 or conflict_count > 0 else "No",
        "Cleaning implication": implication,
    })
holiday_structural_classification = pd.DataFrame(structural_classification_rows)
display(holiday_structural_classification)

,Column,Null count,Main context of nulls,Structural/expected?,Genuine missing-data evidence?,Cleaning implication
0,holiday_type,76298,"76,298 nulls on non-holidays; 0 on actual holi...","Yes, when no applicable event detail exists",No,Preserve null as not-applicable; investigate o...
1,holiday_locale,76298,"76,298 nulls on non-holidays; 0 on actual holi...","Yes, when no applicable event detail exists",No,Preserve null as not-applicable; investigate o...
2,holiday_description,76298,"76,298 nulls on non-holidays; 0 on actual holi...","Yes, when no applicable event detail exists",No,Preserve null as not-applicable; investigate o...
3,holiday_transferred,0,No nulls; False is the deliberate no-record/de...,"Not applicable — represented as False, not null",No,Preserve the Boolean source semantics
4,holiday_event_count,0,No nulls; 0 is the deliberate no-event/default...,"Not applicable — represented as 0, not null",No,Preserve the integer source semantics


# 65. Candidate cleaning approaches — holiday columns

Compare source-cleaning choices separately from later model-friendly feature derivation. A useful modelling category does not require overwriting the auditable source column.

In [56]:
candidate_holiday_approaches = pd.DataFrame([
    {
        "Candidate approach": "A. Preserve structural nulls unchanged",
        "Potential benefit": "Retains not-applicable source semantics and provenance.",
        "Main risk": "Some modelling libraries require explicit null handling.",
        "Evidence supporting it": "Detail nulls align with store-dates lacking applicable event context.",
        "Evidence against it": "Raw nulls are less convenient for some encoders.",
        "Appropriate for cleaning stage?": "Yes",
    },
    {
        "Candidate approach": "B. Replace detail nulls with No Holiday",
        "Potential benefit": "Creates an explicit category for modelling and reporting.",
        "Main risk": "Overwrites not-applicable semantics and conflates cleaning with feature engineering.",
        "Evidence supporting it": "Most null detail rows correspond to no event context.",
        "Evidence against it": "The original columns correctly preserve absence of applicable detail.",
        "Appropriate for cleaning stage?": "No",
    },
    {
        "Candidate approach": "C. Replace null holiday_transferred with False",
        "Potential benefit": "Would produce a complete Boolean field.",
        "Main risk": "Could erase unknown status if true nulls existed.",
        "Evidence supporting it": "The merged base already deliberately represents no record as False.",
        "Evidence against it": "There are no remaining nulls to replace.",
        "Appropriate for cleaning stage?": "No action required",
    },
    {
        "Candidate approach": "D. Replace missing holiday_event_count with 0",
        "Potential benefit": "Would represent no applicable events numerically.",
        "Main risk": "Could hide genuine missing counts if nulls existed.",
        "Evidence supporting it": "The merged base already deliberately represents no record as 0.",
        "Evidence against it": "There are no remaining nulls to replace.",
        "Appropriate for cleaning stage?": "No action required",
    },
    {
        "Candidate approach": "E. Preserve raw semantics; derive explicit modelling features later",
        "Potential benefit": "Keeps cleaning auditable while allowing model-friendly categories or indicators.",
        "Main risk": "Requires documented downstream transformations.",
        "Evidence supporting it": "The representation is internally consistent at store-date grain.",
        "Evidence against it": "Feature pipelines must handle combined multi-event strings deliberately.",
        "Appropriate for cleaning stage?": "Yes",
    },
    {
        "Candidate approach": "F. Split multi-event strings into additional rows",
        "Potential benefit": "Would normalize individual events for event-centric analysis.",
        "Main risk": "Breaks the merged forecasting grain and multiplies store-item rows.",
        "Evidence supporting it": "Multi-event context is visibly combined.",
        "Evidence against it": "The required merged grain must remain one row per date, store, and item.",
        "Appropriate for cleaning stage?": "No",
    },
])
display(candidate_holiday_approaches)

,Candidate approach,Potential benefit,Main risk,Evidence supporting it,Evidence against it,Appropriate for cleaning stage?
0,A. Preserve structural nulls unchanged,Retains not-applicable source semantics and pr...,Some modelling libraries require explicit null...,Detail nulls align with store-dates lacking ap...,Raw nulls are less convenient for some encoders.,Yes
1,B. Replace detail nulls with No Holiday,Creates an explicit category for modelling and...,Overwrites not-applicable semantics and confla...,Most null detail rows correspond to no event c...,The original columns correctly preserve absenc...,No
2,C. Replace null holiday_transferred with False,Would produce a complete Boolean field.,Could erase unknown status if true nulls existed.,The merged base already deliberately represent...,There are no remaining nulls to replace.,No action required
3,D. Replace missing holiday_event_count with 0,Would represent no applicable events numerically.,Could hide genuine missing counts if nulls exi...,The merged base already deliberately represent...,There are no remaining nulls to replace.,No action required
4,E. Preserve raw semantics; derive explicit mod...,Keeps cleaning auditable while allowing model-...,Requires documented downstream transformations.,The representation is internally consistent at...,Feature pipelines must handle combined multi-e...,Yes
5,F. Split multi-event strings into additional rows,Would normalize individual events for event-ce...,Breaks the merged forecasting grain and multip...,Multi-event context is visibly combined.,The required merged grain must remain one row ...,No


# 66. Evidence summary — holiday columns

Consolidate structural-null, applicability, multi-event, and anomaly evidence at store-date grain.

In [57]:
detail_null_rates = holiday_store_date_detail_coverage.set_index("column")["null_percentage"]
actual_holiday_detail_nulls = {
    column: int((holiday_store_date["is_holiday"] & holiday_store_date[column].isna()).sum())
    for column in ["holiday_type", "holiday_locale", "holiday_description"]
}
nonholiday_detail_null_share = 100.0 * (
    ~holiday_store_date["is_holiday"]
    & holiday_store_date["holiday_type"].isna()
).sum() / holiday_store_date["holiday_type"].isna().sum()
locale_inconsistent_scope_groups = int(locale_scope_summary["inconsistent_scope_groups"].sum())
transferred_true_count = int(holiday_store_date["holiday_transferred"].eq(True).sum())

holiday_evidence_summary = pd.DataFrame([
    {
        "Check": "Overall holiday rate",
        "Evidence": f"{holiday_store_date_count:,}/{total_holiday_store_dates:,} store-dates ({100.0 * holiday_store_date_count / total_holiday_store_dates:.6f}%).",
        "Interpretation": "Actual holidays are a minority of represented store-dates.",
        "Cleaning implication": "Use store-date denominators, not repeated item rows.",
    },
    {
        "Check": "Detail-column null rates",
        "Evidence": "; ".join(f"{column} {detail_null_rates[column]:.6f}%" for column in ["holiday_type", "holiday_locale", "holiday_description"]),
        "Interpretation": "Large string-null rates require applicability context.",
        "Cleaning implication": "Do not label null strings corrupted without cross-checking event scope.",
    },
    {
        "Check": "Non-holiday structural-null consistency",
        "Evidence": f"{nonholiday_detail_null_share:.6f}% of holiday_type nulls occur on is_holiday=False store-dates.",
        "Interpretation": "Null type predominantly represents no applicable detail.",
        "Cleaning implication": "Preserve structurally meaningful nulls.",
    },
    {
        "Check": "Actual-holiday detail completeness",
        "Evidence": "; ".join(f"{column} nulls {count}" for column, count in actual_holiday_detail_nulls.items()),
        "Interpretation": "Executed counts show whether actual holidays retain required detail.",
        "Cleaning implication": "Only actual-holiday nulls would indicate genuine missing-detail risk.",
    },
    {
        "Check": "holiday_event_count consistency",
        "Evidence": f"Minimum {event_count.min()}; maximum {event_count.max()}; {multi_event_store_date_count:,} multi-event store-dates; {int(event_count.lt(0).sum())} negative.",
        "Interpretation": "Zero encodes no event and positive counts retain event multiplicity.",
        "Cleaning implication": "Preserve counts and do not expand the forecasting grain.",
    },
    {
        "Check": "Transferred-status consistency",
        "Evidence": f"{transferred_true_count:,} True; {int(holiday_store_date['holiday_transferred'].eq(False).sum()):,} False; {int(holiday_store_date['holiday_transferred'].isna().sum())} null.",
        "Interpretation": "Transferred context is retained separately from actual-holiday status.",
        "Cleaning implication": "Do not reinterpret transferred as missing or overwrite it.",
    },
    {
        "Check": "Locale consistency",
        "Evidence": f"{locale_inconsistent_scope_groups} inconsistent National/Regional/Local scope groups across executed internal checks.",
        "Interpretation": "Scope tokens align, or exceptions are explicitly displayed.",
        "Cleaning implication": "Do not remap locale scope without an evidenced inconsistency.",
    },
    {
        "Check": "Multi-event handling",
        "Evidence": f"{len(multi_event_store_dates):,} store-dates have >1 event; maximum {multi_event_store_dates['holiday_event_count'].max()}.",
        "Interpretation": "Combined strings retain context within one store-date row.",
        "Cleaning implication": "Do not split rows or multiply the merged grain.",
    },
    {
        "Check": "Unresolved anomalies",
        "Evidence": f"{unresolved_suspicious_store_date_count:,} unique unresolved suspicious store-dates.",
        "Interpretation": "Transferred and Work Day exceptions are separated from genuine inconsistency.",
        "Cleaning implication": "Preserve valid exceptions; review only unresolved cases.",
    },
])
display(holiday_evidence_summary)

,Check,Evidence,Interpretation,Cleaning implication
0,Overall holiday rate,"6,749/83,606 store-dates (8.072387%).",Actual holidays are a minority of represented ...,"Use store-date denominators, not repeated item..."
1,Detail-column null rates,holiday_type 91.259001%; holiday_locale 91.259...,Large string-null rates require applicability ...,Do not label null strings corrupted without cr...
2,Non-holiday structural-null consistency,100.000000% of holiday_type nulls occur on is_...,Null type predominantly represents no applicab...,Preserve structurally meaningful nulls.
3,Actual-holiday detail completeness,holiday_type nulls 0; holiday_locale nulls 0; ...,Executed counts show whether actual holidays r...,Only actual-holiday nulls would indicate genui...
4,holiday_event_count consistency,Minimum 0; maximum 2; 229 multi-event store-da...,Zero encodes no event and positive counts reta...,Preserve counts and do not expand the forecast...
5,Transferred-status consistency,"321 True; 83,285 False; 0 null.",Transferred context is retained separately fro...,Do not reinterpret transferred as missing or o...
6,Locale consistency,0 inconsistent National/Regional/Local scope g...,"Scope tokens align, or exceptions are explicit...",Do not remap locale scope without an evidenced...
7,Multi-event handling,229 store-dates have >1 event; maximum 2.,Combined strings retain context within one sto...,Do not split rows or multiply the merged grain.
8,Unresolved anomalies,0 unique unresolved suspicious store-dates.,Transferred and Work Day exceptions are separa...,Preserve valid exceptions; review only unresol...


# 67. Provisional cleaning rule — holiday columns

### 1. What the data shows

- The merged dataset contains **83,606 unique store-dates**. Exactly **6,749 (8.072387%)** have `is_holiday=True`; **76,857** have `is_holiday=False`.
- At store-date grain, `holiday_type`, `holiday_locale`, and `holiday_description` each contain **76,298 nulls (91.259001%)**. At repeated item-row grain, each contains **113,841,496 nulls (90.712495%)**, reconciling the earlier Notebook 04 observation.
- All **6,749 actual holiday store-dates** have non-null type, locale, description, transferred status, and event count. No holiday attribute conflicts across repeated item rows were found.
- There are **7,308 event-context store-dates**: 6,749 actual holidays plus 559 `is_holiday=False` rows with valid context. Those 559 are fully explained by **321 transferred source records** and **238 `Work Day` records**.
- `holiday_transferred` has **321 True**, **83,285 False**, and **0 null** store-dates. `holiday_event_count` has **76,298 zeros**, **7,079 ones**, **229 twos**, no nulls, no negatives, and a maximum of two.
- Exactly **229 store-dates** contain multiple events. Their descriptions are combined within one store-date row; 175 also have combined type strings and 14 have combined locale strings. No item-row inconsistency or grain multiplication was detected.
- Internal scope checks found **zero inconsistent National, Regional, or Local applicability groups**. There are 108 exact non-null description values; three descriptions occur with multiple exact types and none with multiple locale values.
- All requested anomaly checks reconcile to **zero unresolved suspicious store-dates**.

### 2. What is structurally expected

- Null `holiday_type`, `holiday_locale`, and `holiday_description` values on the 76,298 no-event store-dates correctly mean that holiday detail is not applicable. They are structural nulls, not missing observations requiring repair.
- The merged base deliberately represents no applicable event as `holiday_transferred=False` and `holiday_event_count=0`; these columns are complete rather than null.
- `is_holiday=False` does not require all detail fields to be null: transferred source records and `Work Day` context intentionally retain type, locale, description, and event count without being classified as an actual holiday.

### 3. Genuine inconsistencies found

No genuine inconsistency was found in the executed checks: actual holidays have complete details and positive event counts, event counts are non-negative, locale expansion is internally consistent, and repeated item rows agree within every store-date.

### 4. What remains uncertain

- The predictive value of individual holiday types, locale scopes, descriptions, transferred context, and multi-event combinations has not been tested.
- Description variants associated with multiple types may be historically valid; this notebook does not establish a text-normalization error.
- Any model-friendly encoding must be evaluated later without changing the auditable merged-source representation.

### 5. Recommended provisional cleaning rule

**Preserve all holiday columns unchanged during SCRUM-9 cleaning.** Keep structural nulls in `holiday_type`, `holiday_locale`, and `holiday_description`; preserve the deliberate Boolean semantics of `is_holiday` and `holiday_transferred`; preserve `holiday_event_count=0` for no-event store-dates and positive counts for applicable context. Do not insert `"No Holiday"`, overwrite transferred status, rename or merge observed categories, split multi-event rows, or delete records. No cleaning rule is applied in this analysis notebook.

### 6. Later feature-engineering implications

- Explicit categories such as `"No Holiday"`, applicability indicators, transferred-event indicators, locale encodings, multi-event flags, and text-derived features may be created later as separate modelling features.
- Downstream transformations must preserve the one-row `(date, store_nbr, item_nbr)` forecasting grain and be evaluated through temporal validation.
- Combined multi-event strings should not be expanded into additional merged rows.

### 7. Final SCRUM-9 cleaning-rules table entries

| Column | Observed representation | Provisional cleaning rule |
|---|---|---|
| `is_holiday` | Complete Boolean; false for transferred and `Work Day`-only context | Preserve unchanged |
| `holiday_type` | Structural null when no applicable event; exact combined values for multi-events | Preserve nulls and observed values unchanged |
| `holiday_locale` | Structural null when no applicable event; internally consistent scope values | Preserve nulls and observed values unchanged |
| `holiday_description` | Structural null when no applicable event; exact combined descriptions for multi-events | Preserve nulls and text unchanged; no normalization yet |
| `holiday_transferred` | Complete Boolean; deliberate False default and 321 transferred contexts | Preserve unchanged |
| `holiday_event_count` | Complete integer; 0 means no event, positive values retain multiplicity | Preserve unchanged; do not split rows |
| Merged grain | One row per `(date, store_nbr, item_nbr)` | Preserve exactly |


# 68. Earthquake-related event investigation

This focused investigation evaluates observed earthquake-related event records and their relationship with `unit_sales` before the semantic breadth of `is_holiday` receives final review. It identifies records from actual holiday descriptions—not from every row labelled `holiday_type="Event"`—and applies no cleaning decision, deletion, transformation, or causal interpretation.

## 68.1 Identify earthquake-related descriptions

First inspect distinct event metadata whose description contains the observed Spanish earthquake term `Terremoto`. The exact discovered description strings then become the downstream filter.

In [58]:
EARTHQUAKE_DISCOVERY_COLUMNS = [
    "date", "holiday_description", "holiday_type", "holiday_locale"
]
missing_earthquake_columns = [
    column for column in EARTHQUAKE_DISCOVERY_COLUMNS
    if column not in parquet_file.schema_arrow.names
]
if missing_earthquake_columns:
    raise ValueError(f"Missing required earthquake investigation columns: {missing_earthquake_columns}")

EARTHQUAKE_DISCOVERY_TERM = "terremoto"
earthquake_metadata_parts = []
for row_group_index in range(parquet_file.metadata.num_row_groups):
    metadata_scan = parquet_file.read_row_group(
        row_group_index, columns=EARTHQUAKE_DISCOVERY_COLUMNS
    ).to_pandas()
    matching_description = metadata_scan["holiday_description"].astype("string").str.contains(
        EARTHQUAKE_DISCOVERY_TERM, case=False, regex=False, na=False
    )
    if matching_description.any():
        earthquake_metadata_parts.append(
            metadata_scan.loc[matching_description, EARTHQUAKE_DISCOVERY_COLUMNS].drop_duplicates()
        )

if not earthquake_metadata_parts:
    raise ValueError("No earthquake-related holiday descriptions were found.")

earthquake_event_metadata = pd.concat(
    earthquake_metadata_parts, ignore_index=True
).drop_duplicates().sort_values(
    ["date", "holiday_description", "holiday_type", "holiday_locale"]
).reset_index(drop=True)
EARTHQUAKE_DESCRIPTIONS = sorted(
    earthquake_event_metadata["holiday_description"].dropna().unique().tolist()
)
earthquake_event_dates = pd.DatetimeIndex(
    sorted(earthquake_event_metadata["date"].drop_duplicates())
)

earthquake_description_inventory = earthquake_event_metadata.groupby(
    ["holiday_description", "holiday_type", "holiday_locale"],
    as_index=False, observed=True, dropna=False,
).agg(
    first_date=("date", "min"),
    last_date=("date", "max"),
    unique_date_count=("date", "nunique"),
).sort_values(["first_date", "holiday_description"]).reset_index(drop=True)

display(pd.DataFrame([{
    "discovery_term": EARTHQUAKE_DISCOVERY_TERM,
    "exact_matching_descriptions": len(EARTHQUAKE_DESCRIPTIONS),
    "matching_metadata_rows": len(earthquake_event_metadata),
    "unique_matching_dates": len(earthquake_event_dates),
}]))
display(Markdown("### Distinct matching descriptions and observed metadata"))
display(earthquake_description_inventory)
display(Markdown("### Matching descriptions by date"))
display(earthquake_event_metadata)

,discovery_term,exact_matching_descriptions,matching_metadata_rows,unique_matching_dates
0,terremoto,33,33,31


### Distinct matching descriptions and observed metadata

,holiday_description,holiday_type,holiday_locale,first_date,last_date,unique_date_count
0,Terremoto Manabi,Event,National,2016-04-16,2016-04-16,1
1,Terremoto Manabi+1,Event,National,2016-04-17,2016-04-17,1
2,Terremoto Manabi+2,Event,National,2016-04-18,2016-04-18,1
3,Terremoto Manabi+3,Event,National,2016-04-19,2016-04-19,1
4,Terremoto Manabi+4,Event,National,2016-04-20,2016-04-20,1
5,Cantonizacion de Riobamba | Terremoto Manabi+5,Event | Holiday,Local | National,2016-04-21,2016-04-21,1
6,Terremoto Manabi+5,Event,National,2016-04-21,2016-04-21,1
7,Terremoto Manabi+6,Event,National,2016-04-22,2016-04-22,1
8,Terremoto Manabi+7,Event,National,2016-04-23,2016-04-23,1
9,Terremoto Manabi+8,Event,National,2016-04-24,2016-04-24,1


### Matching descriptions by date

,date,holiday_description,holiday_type,holiday_locale
0,2016-04-16,Terremoto Manabi,Event,National
1,2016-04-17,Terremoto Manabi+1,Event,National
2,2016-04-18,Terremoto Manabi+2,Event,National
3,2016-04-19,Terremoto Manabi+3,Event,National
4,2016-04-20,Terremoto Manabi+4,Event,National
5,2016-04-21,Cantonizacion de Riobamba | Terremoto Manabi+5,Event | Holiday,Local | National
6,2016-04-21,Terremoto Manabi+5,Event,National
7,2016-04-22,Terremoto Manabi+6,Event,National
8,2016-04-23,Terremoto Manabi+7,Event,National
9,2016-04-24,Terremoto Manabi+8,Event,National


## 68.2 Earthquake-related record counts and `unit_sales`

Use the exact observed description inventory to scan only the earthquake calendar span and equal-length preceding/following windows. Collect only numeric earthquake sales arrays needed for exact medians; all wider data remains row-group bounded.

In [59]:
EARTHQUAKE_SALES_COLUMNS = [
    "date", "store_nbr", "item_nbr", "unit_sales",
    "holiday_description", "holiday_type", "holiday_locale", "is_holiday",
]
earthquake_start_date = pd.Timestamp(earthquake_event_dates.min())
earthquake_end_date = pd.Timestamp(earthquake_event_dates.max())
earthquake_calendar_span_days = int((earthquake_end_date - earthquake_start_date).days + 1)
nearby_start_date = earthquake_start_date - pd.Timedelta(days=earthquake_calendar_span_days)
nearby_end_date = earthquake_end_date + pd.Timedelta(days=earthquake_calendar_span_days)

parquet_date_column_index = parquet_file.schema_arrow.names.index("date")
daily_all_sales = {}
daily_all_row_counts = {}
earthquake_daily_sales_parts = {}
earthquake_daily_stores = {}
earthquake_daily_items = {}
earthquake_daily_descriptions = {}
earthquake_daily_types = {}
earthquake_daily_locales = {}
earthquake_store_set = set()
earthquake_item_set = set()

for row_group_index in range(parquet_file.metadata.num_row_groups):
    date_statistics = parquet_file.metadata.row_group(row_group_index).column(
        parquet_date_column_index
    ).statistics
    if date_statistics is not None and date_statistics.has_min_max:
        row_group_min_date = pd.Timestamp(date_statistics.min)
        row_group_max_date = pd.Timestamp(date_statistics.max)
        if row_group_max_date < nearby_start_date or row_group_min_date > nearby_end_date:
            continue

    window_scan = parquet_file.read_row_group(
        row_group_index, columns=EARTHQUAKE_SALES_COLUMNS
    ).to_pandas()
    window_scan = window_scan.loc[
        window_scan["date"].between(nearby_start_date, nearby_end_date)
    ]
    if window_scan.empty:
        continue

    for date, date_group in window_scan.groupby("date", observed=True):
        date_key = pd.Timestamp(date)
        daily_all_sales[date_key] = daily_all_sales.get(date_key, 0.0) + float(
            date_group["unit_sales"].sum()
        )
        daily_all_row_counts[date_key] = daily_all_row_counts.get(date_key, 0) + len(date_group)

    earthquake_mask = (
        window_scan["date"].isin(earthquake_event_dates)
        & window_scan["holiday_description"].isin(EARTHQUAKE_DESCRIPTIONS)
    )
    earthquake_chunk = window_scan.loc[earthquake_mask]
    if earthquake_chunk.empty:
        continue

    earthquake_store_set.update(int(value) for value in earthquake_chunk["store_nbr"].unique())
    earthquake_item_set.update(int(value) for value in earthquake_chunk["item_nbr"].unique())
    for date, date_group in earthquake_chunk.groupby("date", observed=True):
        date_key = pd.Timestamp(date)
        earthquake_daily_sales_parts.setdefault(date_key, []).append(
            date_group["unit_sales"].to_numpy(dtype="float64", copy=True)
        )
        earthquake_daily_stores.setdefault(date_key, set()).update(
            int(value) for value in date_group["store_nbr"].unique()
        )
        earthquake_daily_items.setdefault(date_key, set()).update(
            int(value) for value in date_group["item_nbr"].unique()
        )
        earthquake_daily_descriptions.setdefault(date_key, set()).update(
            str(value) for value in date_group["holiday_description"].dropna().unique()
        )
        earthquake_daily_types.setdefault(date_key, set()).update(
            str(value) for value in date_group["holiday_type"].dropna().unique()
        )
        earthquake_daily_locales.setdefault(date_key, set()).update(
            str(value) for value in date_group["holiday_locale"].dropna().unique()
        )

missing_scanned_earthquake_dates = set(earthquake_event_dates) - set(earthquake_daily_sales_parts)
if missing_scanned_earthquake_dates:
    raise ValueError(f"Matching earthquake dates were absent from the bounded sales scan: {missing_scanned_earthquake_dates}")

earthquake_sales_values = np.concatenate([
    values for date in sorted(earthquake_daily_sales_parts)
    for values in earthquake_daily_sales_parts[date]
])
earthquake_negative_sales_count = int((earthquake_sales_values < 0).sum())
earthquake_fractional_sales_count = int(
    (~np.isclose(earthquake_sales_values, np.round(earthquake_sales_values), atol=1e-9)).sum()
)
earthquake_sales_summary = pd.DataFrame([{
    "total_item_rows": len(earthquake_sales_values),
    "unique_dates": len(earthquake_daily_sales_parts),
    "first_date": min(earthquake_daily_sales_parts),
    "last_date": max(earthquake_daily_sales_parts),
    "unique_stores": len(earthquake_store_set),
    "unique_items": len(earthquake_item_set),
    "total_unit_sales": float(earthquake_sales_values.sum()),
    "mean_unit_sales": float(earthquake_sales_values.mean()),
    "median_unit_sales": float(np.median(earthquake_sales_values)),
    "minimum_unit_sales": float(earthquake_sales_values.min()),
    "maximum_unit_sales": float(earthquake_sales_values.max()),
    "negative_unit_sales_count": earthquake_negative_sales_count,
    "fractional_unit_sales_count": earthquake_fractional_sales_count,
    "fractional_unit_sales_percentage": (
        100.0 * earthquake_fractional_sales_count / len(earthquake_sales_values)
    ),
}])
display(earthquake_sales_summary)

,total_item_rows,unique_dates,first_date,last_date,unique_stores,unique_items,total_unit_sales,mean_unit_sales,median_unit_sales,minimum_unit_sales,maximum_unit_sales,negative_unit_sales_count,fractional_unit_sales_count,fractional_unit_sales_percentage
0,2995997,31,2016-04-16,2016-05-16,53,3548,2.732052e+07,9.119008,4.0,-4673.0,44142.0,271,205086,6.845334


## 68.3 Aggregated daily earthquake-event analysis

Summarize only rows carrying one of the exact earthquake descriptions, retaining observed combined event metadata without expanding the merged grain.

In [60]:
earthquake_daily_rows = []
for date in sorted(earthquake_daily_sales_parts):
    date_sales = np.concatenate(earthquake_daily_sales_parts[date])
    earthquake_daily_rows.append({
        "date": date,
        "earthquake_event_description": " || ".join(sorted(earthquake_daily_descriptions[date])),
        "holiday_type": " || ".join(sorted(earthquake_daily_types[date])),
        "holiday_locale": " || ".join(sorted(earthquake_daily_locales[date])),
        "number_of_sales_rows": len(date_sales),
        "unique_stores": len(earthquake_daily_stores[date]),
        "unique_items": len(earthquake_daily_items[date]),
        "total_unit_sales": float(date_sales.sum()),
        "mean_unit_sales": float(date_sales.mean()),
        "median_unit_sales": float(np.median(date_sales)),
    })
earthquake_daily_analysis = pd.DataFrame(earthquake_daily_rows)
display(earthquake_daily_analysis)

,date,earthquake_event_description,holiday_type,holiday_locale,number_of_sales_rows,unique_stores,unique_items,total_unit_sales,mean_unit_sales,median_unit_sales
0,2016-04-16,Terremoto Manabi,Event,National,100966,53,3484,862121.492,8.538731,4.0000
1,2016-04-17,Terremoto Manabi+1,Event,National,100910,53,3441,1271833.739,12.603644,5.0000
2,2016-04-18,Terremoto Manabi+2,Event,National,98729,53,3477,1345920.605,13.632475,5.0000
3,2016-04-19,Terremoto Manabi+3,Event,National,97114,53,3479,1152089.208,11.863266,4.0000
4,2016-04-20,Terremoto Manabi+4,Event,National,96776,53,3481,1062426.290,10.978200,4.0000
5,2016-04-21,Cantonizacion de Riobamba | Terremoto Manabi+5...,Event || Event | Holiday,Local | National || National,93466,53,3481,998132.782,10.679100,4.0000
6,2016-04-22,Terremoto Manabi+6,Event,National,93996,53,3475,857059.244,9.118040,4.0000
7,2016-04-23,Terremoto Manabi+7,Event,National,101232,53,3487,1022143.135,10.097036,5.0000
8,2016-04-24,Terremoto Manabi+8,Event,National,99272,53,3442,1039369.932,10.469920,5.0000
9,2016-04-25,Terremoto Manabi+9,Event,National,91658,53,3466,680000.754,7.418891,4.0000


## 68.4 Nearby-period comparison

Compare daily total sales on earthquake-event dates with represented non-earthquake dates in equal-length calendar windows immediately before and after the event span. This local baseline is descriptive: it does not control for seasonality, promotions, store/item coverage changes, holidays, or other concurrent events.

In [61]:
nearby_daily_sales = pd.DataFrame([
    {
        "date": date,
        "daily_total_unit_sales": total_sales,
        "sales_row_count": daily_all_row_counts[date],
    }
    for date, total_sales in sorted(daily_all_sales.items())
])
earthquake_date_set = set(pd.Timestamp(value) for value in earthquake_event_dates)
previous_window_mask = nearby_daily_sales["date"].between(
    nearby_start_date, earthquake_start_date - pd.Timedelta(days=1)
)
following_window_mask = nearby_daily_sales["date"].between(
    earthquake_end_date + pd.Timedelta(days=1), nearby_end_date
)
earthquake_date_mask = nearby_daily_sales["date"].isin(earthquake_date_set)
comparison_date_mask = (previous_window_mask | following_window_mask) & ~earthquake_date_mask

def summarize_daily_period(label, frame):
    return {
        "period": label,
        "first_represented_date": frame["date"].min(),
        "last_represented_date": frame["date"].max(),
        "represented_date_count": len(frame),
        "mean_daily_total_unit_sales": float(frame["daily_total_unit_sales"].mean()),
        "median_daily_total_unit_sales": float(frame["daily_total_unit_sales"].median()),
        "minimum_daily_total_unit_sales": float(frame["daily_total_unit_sales"].min()),
        "maximum_daily_total_unit_sales": float(frame["daily_total_unit_sales"].max()),
    }

earthquake_period_daily = nearby_daily_sales.loc[earthquake_date_mask].copy()
nearby_comparison_daily = nearby_daily_sales.loc[comparison_date_mask].copy()
previous_comparison_daily = nearby_daily_sales.loc[previous_window_mask & ~earthquake_date_mask].copy()
following_comparison_daily = nearby_daily_sales.loc[following_window_mask & ~earthquake_date_mask].copy()
earthquake_vs_nearby_comparison = pd.DataFrame([
    summarize_daily_period("earthquake-event dates", earthquake_period_daily),
    summarize_daily_period("nearby non-earthquake comparison dates", nearby_comparison_daily),
])
comparison_mean = earthquake_vs_nearby_comparison.loc[
    earthquake_vs_nearby_comparison["period"].eq("nearby non-earthquake comparison dates"),
    "mean_daily_total_unit_sales",
].iloc[0]
comparison_median = earthquake_vs_nearby_comparison.loc[
    earthquake_vs_nearby_comparison["period"].eq("nearby non-earthquake comparison dates"),
    "median_daily_total_unit_sales",
].iloc[0]
earthquake_vs_nearby_comparison["mean_difference_vs_nearby_percentage"] = np.where(
    earthquake_vs_nearby_comparison["period"].eq("earthquake-event dates"),
    100.0 * (
        earthquake_vs_nearby_comparison["mean_daily_total_unit_sales"] - comparison_mean
    ) / comparison_mean,
    0.0,
)
earthquake_vs_nearby_comparison["median_difference_vs_nearby_percentage"] = np.where(
    earthquake_vs_nearby_comparison["period"].eq("earthquake-event dates"),
    100.0 * (
        earthquake_vs_nearby_comparison["median_daily_total_unit_sales"] - comparison_median
    ) / comparison_median,
    0.0,
)
nearby_subwindow_summary = pd.DataFrame([
    summarize_daily_period("preceding equal-length window", previous_comparison_daily),
    summarize_daily_period("following equal-length window", following_comparison_daily),
])

display(pd.DataFrame([{
    "earthquake_calendar_span_days": earthquake_calendar_span_days,
    "earthquake_start_date": earthquake_start_date,
    "earthquake_end_date": earthquake_end_date,
    "preceding_window_start": nearby_start_date,
    "following_window_end": nearby_end_date,
}]))
display(earthquake_vs_nearby_comparison)
display(Markdown("### Preceding and following window sensitivity"))
display(nearby_subwindow_summary)

,earthquake_calendar_span_days,earthquake_start_date,earthquake_end_date,preceding_window_start,following_window_end
0,31,2016-04-16,2016-05-16,2016-03-16,2016-06-16


,period,first_represented_date,last_represented_date,represented_date_count,mean_daily_total_unit_sales,median_daily_total_unit_sales,minimum_daily_total_unit_sales,maximum_daily_total_unit_sales,mean_difference_vs_nearby_percentage,median_difference_vs_nearby_percentage
0,earthquake-event dates,2016-04-16,2016-05-16,31,881307.088613,820824.6380,543339.849,1345920.605,16.720146,17.95557
1,nearby non-earthquake comparison dates,2016-03-16,2016-06-16,62,755059.962968,695876.1155,546785.209,1266908.292,0.000000,0.00000


### Preceding and following window sensitivity

,period,first_represented_date,last_represented_date,represented_date_count,mean_daily_total_unit_sales,median_daily_total_unit_sales,minimum_daily_total_unit_sales,maximum_daily_total_unit_sales
0,preceding equal-length window,2016-03-16,2016-04-15,31,751659.125129,689727.196,546785.209,1266908.292
1,following equal-length window,2016-05-17,2016-06-16,31,758460.800806,706024.058,547139.613,1123734.436


## 68.5 Interpretation and provisional SCRUM-9 conclusion

### Interpretation

- The filter identified **33 exact observed description strings** containing `Terremoto`, covering **31 consecutive dates from 2016-04-16 through 2016-05-16**. The inventory includes `Terremoto Manabi` through `Terremoto Manabi+30`, plus combined descriptions on dates that also carry other holiday context. It does not classify every `holiday_type="Event"` row as earthquake-related.
- These are **2,995,997 genuine observed sales rows**, spanning **53 stores** and **3,548 items**. Their total `unit_sales` is approximately **27.32 million**; mean is **9.119008**, median **4.0**, minimum **-4,673**, and maximum **44,142**. There are **271 negative rows** and **205,086 fractional rows (6.845334%)**.
- Mean daily total sales on earthquake-event dates are **881,307.09**, compared with **755,059.96** across the combined equal-length neighboring windows: a descriptive difference of **+16.720146%**. Median daily totals differ by **+17.955570%**. The preceding and following window means are similar at **751,659.13** and **758,460.80**, respectively.
- This local comparison does **not** prove that the earthquake caused the sales difference. It does not control for promotions, seasonality, store or item coverage, other holidays, concurrent events, or wider operational disruption.
- The material descriptive difference is evidence that this period may represent an extraordinary-demand regime worthy of explicit temporal validation. It is not evidence that the historical sales rows are invalid.
- These rows must not be deleted during cleaning. Any later exclusion, down-weighting, robustness treatment, or special-event feature is a modelling and validation decision.
- The investigation shows why `is_holiday` may be semantically too broad for modelling: earthquake-related national `Event` records are represented within the same applicability flag as conventional holidays. That semantic concern supports retaining the detailed event metadata rather than collapsing interpretation to the Boolean flag alone.

### Provisional SCRUM-9 conclusion — evidence pending final review

**Do not change the existing `is_holiday` cleaning decision automatically.** Preserve the observed earthquake-period sales rows and all holiday/event metadata unchanged. Record as pending evidence that `is_holiday` combines conventional holidays with extraordinary events such as `Terremoto Manabi`; final review should decide whether later feature engineering needs a distinct earthquake or extraordinary-event indicator and whether temporal validation requires special-regime sensitivity analysis. No cleaning transformation is applied here.


# 69. Final SCRUM-9 cleaning strategy

## Cleaning philosophy

Source-data cleaning acts only on records that actually exist. It does not create missing `(date, store_nbr, item_nbr)` rows or infer `unit_sales = 0` from row absence. Absence may represent zero demand, store inactivity or closure, item inactivity or assortment status, or source recording rules; the available source does not distinguish these reliably. The cleaned base therefore remains sparse. When a true replacement value cannot be recovered, source semantics and nullability are preserved. Imputation, densification, lag creation, target transformation, encoding, and modelling-specific treatment remain later-stage work unless separately justified as source cleaning.

## Consolidated cleaning rules

| Field / issue | Evidence | Final cleaning action | Explicitly do not do | Later feature-engineering / modelling note |
|---|---|---|---|---|
| Grain / absent-row semantics | The source-derived merged data contains only observed `(date, store_nbr, item_nbr)` records; absence is semantically ambiguous. | Validate and clean existing rows only; keep the cleaned base sparse and preserve the established grain. | Do not create missing panel rows or infer zero sales from absence. | Any panel construction or densification requires separate approval and explicit business semantics. |
| Negative `unit_sales` | **7,795 rows (0.006211%)**; present across all 54 stores, 2,568 items, and 2013-2017. Nearby positives make returns, reversals, or corrections plausible but not proven. | Preserve every signed source value exactly. | Do not drop, zero, take absolute values, cap, or overwrite. | A derived forecasting target and separate extreme-value monitoring may be evaluated later without altering lineage. |
| Positive extreme `unit_sales` | Very large positive observations exist; the executed full-data status aggregation records a maximum of **89,440**. Magnitude alone does not prove invalidity. | Preserve values and monitor positive and negative extremes symmetrically. | Do not cap or delete without authoritative source evidence. | Define robust target treatment or exception thresholds only through modelling and validation. |
| `unit_sales` measurement heterogeneity | Fractional sales are observed, including **205,086** earthquake-period rows; these are consistent with weight-based sales in some product families, although the source rows do not label measurement units explicitly. | Preserve numeric precision exactly and document that `unit_sales` is not always an integer item count. | Do not round to integers. | Family-aware interpretation or target transformations require later validation. |
| Missing `onpromotion` | **21,657,651 rows (17.257499%)**, entirely from 2013-01-01 through 2014-03-31; complete from 2014-04-01 onward. | Preserve as nullable Boolean unknown. | Do not fill `False`, impute, or convert missingness into an ordinary promotion category during cleaning. | Evaluate promo-era handling and temporal confounding later; any indicator/category is a derived feature. |
| Missing `transactions` | **214,625 item rows**, representing **118 store-dates**; strongly concentrated around the outage-like 2016-01-02 to 2016-01-04 period, while valid sales rows remain. | Preserve `NULL` transactions. | Do not zero-fill, forward-fill, backward-fill, interpolate, median-fill, or delete sales rows. | Same-day transactions are not assumed future-known; only leakage-safe lagged or aggregated history may be tested later. |
| Missing `dcoilwtico` | **521 of 1,684 dates** are missing; **481** are weekends, with 40 weekday gaps. Missingness includes market-closed/absent-date patterns and source-null cases whose exact values are unrecoverable. | Preserve source missingness. | Do not interpolate, backfill, or mean/median-fill during cleaning. | Causal forward-fill and oil ablation may be tested later. A derived `oil_missing_reason` could distinguish market-closed/absent dates from explicit source nulls, but is not created here. |
| Holiday detail: `holiday_type`, `holiday_locale`, `holiday_description` | Structural nulls dominate non-applicable store-dates; applicable event records have metadata, and scope consistency checks passed. | Preserve observed values and structural nulls. | Do not insert `No Holiday` or destructively normalize descriptions. | Derive model-friendly categories separately while retaining original fields. |
| `holiday_transferred` | Existing complete Boolean representation includes **321 True** store-dates. `False` was already populated during merge construction where no applicable transferred event existed. | Preserve existing True/False values and document merge lineage. | Do not reinterpret or overwrite the construction-time default during cleaning. | A separate transferred-event feature may be derived if validated. |
| `holiday_event_count` | Values are only **0, 1, or 2**; **229** store-dates contain genuine multi-event combinations. Zero was already created/fill-treated during merge construction where no applicable event existed. | Preserve existing counts and their construction semantics. | Do not split, delete, or collapse multi-event records. | A multi-event indicator or component encoding may be derived without multiplying grain rows. |
| `is_holiday` semantic caveat | The flag covers broader applicable calendar-event context, not only conventional public holidays. Earthquake evidence includes **33 descriptions**, **31 dates (2016-04-16 to 2016-05-16)**, **2,995,997 rows**, 53 stores, 3,548 items, and about **27.32M** sales. Daily totals were descriptively higher than nearby windows, without causal proof. | Preserve current values for lineage and explicitly retain the broader meaning. | Do not treat it as a strict public-holiday-only flag or collapse extraordinary events into ordinary-holiday meaning. | Derive separate `is_event`, `is_actual_holiday`, `is_earthquake_event`, `is_work_day`, or `is_bridge` indicators if useful. |
| Earthquake-period observations | Genuine observed sales and event metadata describe an extraordinary period; descriptive differences do not establish earthquake causality. | Preserve all observed rows and metadata. | Do not delete, relabel as invalid, or down-weight during cleaning solely because the period is rare or non-recurring. | Exclusion, down-weighting, regime handling, and event-specific features are modelling/validation decisions. |
| Core identifiers, metadata, and grain quality | **125,497,040 rows**, 21 expected columns, 54 stores, 4,036 items, and 2013-01-01 to 2017-08-15 coverage; **0 exact duplicates** and **0 grain duplicates**. Previously verified core identifier/product/store metadata are complete. | Preserve IDs, core metadata, row count, order-independent grain uniqueness, and source values unchanged. | Do not invent transformations or remove valid records. | Reassert counts, cardinalities, non-null core fields, and grain uniqueness in SCRUM-10. |
| Schema / dtype contract | Stable nullability and semantics are required to prevent implementation-time coercion. | SCRUM-10 must enforce an explicit schema: nullable Boolean `onpromotion`, an appropriate pure-date representation for `date`, and stable categorical/string semantics. | Do not use accidental `.astype(bool)` coercion on nullable promotions or perform unnecessary schema rewrites here. | Make schema, nullability, range, cardinality, and grain checks executable assertions in SCRUM-10. |


## Final SCRUM-9 conclusion

**SCRUM-9 is technically complete and ready for SCRUM-10 under this governing policy:**

> Only validate and clean rows that actually exist in the source-derived merged dataset. Do not create missing date-store-item rows and do not infer zero sales where no source record exists.

The merged Favorita base is already relatively clean, so the cleaned dataset is expected to preserve most values unchanged. SCRUM-9 defines evidence-based cleaning behavior; it does not force transformations where no valid correction can be recovered. SCRUM-10 should implement these rules, enforce the schema and quality contract, preserve lineage, and create the cleaned artifact. Panel construction, zero materialization, lag features, causal oil filling, modelling target transformation, cold-start handling, and event-specific model treatment remain outside SCRUM-9.

### Ready for SCRUM-10

- [x] Cleaning rules consolidated.
- [x] Sparse absent-row policy documented.
- [x] Earthquake and broader event semantics documented.
- [x] No source rows approved for synthetic creation.
- [x] No unjustified imputation approved.
- [x] Schema and executable quality expectations identified.
- [x] SCRUM-10 may now implement the cleaned dataset while preserving lineage.
